<div align="center">

# Hafta 7: Vision Transformer (ViT): Görüntü işlemede yeni nesil

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/lab.html) · [PDF ders notu](Hafta07_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-07/NB07_vision_transformer.ipynb)

</div>

**Bu Jupyter Notebook iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

**Ders sayfasıyla birlikte.** Bu Jupyter Notebook'a ders sayfasının uygulama duraklarından gelinir. Her durağın sonundaki bağlantı sizi ders sayfasına, okumanın devam ettiği yere geri götürür.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı Jupyter Notebook, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. 5-7. bölümlerdeki her küçük ağın eğitimi sıradan bir dizüstü bilgisayarda bir ile üç dakika, 600 ekokardiyografi karesinin öznitelikleri yaklaşık iki dakika, 4. bölümdeki mini projenin mini ViT ağının CIFAR-10 ile eğitimi ise yaklaşık beş dakika sürer; ekran kartı gerekmez. Colab'da bir ekran kartı (Çalışma zamanı, Çalışma zamanı türünü değiştir) bu sürelerin hepsini kısaltır. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir Jupyter Notebook'a ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Jupyter Notebook tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. İkinci hücre eksik olanları indirir: Human Heart Project'in (CREATIS, Lyon) açık veri tabanından CAMUS'un 300 hastasının apikal ekokardiyogramlarını uzman sınırlarıyla birlikte (yaklaşık 227 MB; 4. hafta bu bilgisayarda çalıştırıldıysa zaten vardır), ViT-B/16'nın (yaklaşık 330 MB) ve EfficientNet-B0'ın (yaklaşık 21 MB) ImageNet ağırlıklarını ve Zenodo'dan ARCADE'in darlık kümesini (451,6 MB'lık bir arşiv; yalnızca darlık kümesi saklanır). İlk çalıştırma birkaç dakika sürer; sonraki çalıştırmalarda zaten var olan dosyalar atlanır. CAMUS, ticari olmayan bilimsel araştırmalar için CC BY-NC-SA 4.0 lisansıyla ücretsiz olarak sunulmaktadır; CAMUS'un her kullanımında Leclerc ve arkadaşlarının IEEE Transactions on Medical Imaging dergisindeki 2019 tarihli makalesine atıf yapılmalıdır. ARCADE CC0 ile yayımlanmıştır; makalesi Popov ve arkadaşlarının Scientific Data dergisindeki 2024 tarihli çalışmasıdır. Sanal kliniğin sentetik anjiyogramları Jupyter Notebook'un içinde çizilir. İki indirme, hücreleri çalıştığında daha sonra yapılır: 3. bölümde Hugging Face Hub'daki hazır Vision Transformer (yaklaşık 350 MB) ve 4. bölümde mini proje için Toronto Üniversitesinden CIFAR-10 (yaklaşık 170 MB).

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy

from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from vclinic import angiogram, arcade, camus, coronary

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torchvision.models import vit_b_16, ViT_B_16_Weights, efficientnet_b0, EfficientNet_B0_Weights
except ImportError as error:
    raise ImportError("PyTorch ya da torchvision kurulu değil. SETUP.tr.md dosyasında anlatıldığı gibi kurun, ardından çekirdeği yeniden başlatın.") from error
try:
    import skimage
except ImportError as error:
    raise ImportError("scikit-image kurulu değil. pip install scikit-image komutuyla kurun, ardından çekirdeği yeniden başlatın.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 8, 0):
    raise ImportError("vclinic ders paketi bu Jupyter Notebook'tan daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
camus.download()
arcade.download()
ViT_B_16_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
EfficientNet_B0_Weights.IMAGENET1K_V1.get_state_dict(progress=False)
print("ViT-B/16 ve EfficientNet-B0'ın ImageNet ağırlıkları hazır.")
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["ipywidgets"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; Jupyter Notebook yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 7.1": (196, 0),
    "Alıştırma 7.2": (0.284, 0.001),
    "Alıştırma 7.3": (0.685, 0.001),
    "Alıştırma A": (274, 0),
    "Alıştırma B": (0.665, 0.001),
    "Alıştırma C": (7087872, 0),
    "Alıştırma D": (0.885, 0.01),
    "Alıştırma E": (0.176, 0.001),
    "Alıştırma F": (1.625, 0.001),
    "Alıştırma G": (133888, 0),
    "Alıştırma H": (0.139, 0.02),
    "Alıştırma I": (0.68, 0.05),
    "Alıştırma J": (905, 25),
    "Adım 1.1": (0.704, 0.002),
    "Adım 1.2": (0.862, 0.01),
    "Adım 1.3": (0.782, 0.01),
    "Adım 1.4": (0.925, 0.05),
    "Adım 2.1": (0.965, 0.05),
    "Adım 2.2": (0.538, 0.02),
    "Adım 2.3": (0.81, 0.05),
    "Adım 3.1": (0.93, 0.05),
    "Adım 3.2": (0.934, 0.05),
    "Adım 3.3": (0.153, 0.01),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; haftanın verilerinden kurulur."""
    g = globals()
    k = anahtar
    sets = {"training": "eğitim", "validation": "doğrulama", "test": "test", "testing": "test", "train": "eğitim", "val": "doğrulama"}
    need = lambda *names: [g[n] for n in names] if all(n in g for n in names) else (_ for _ in ()).throw(
        RuntimeError("Önce bu alıştırmadan önceki bölümleri çalıştırın."))
    if k in ("7.1", "7.3"):
        left, truth = angiogram.patient_frames("VC-2427")["left"]
        if k == "7.1":
            return pd.DataFrame({"anjiyogram": [np.asarray(Image.fromarray(left).resize((224, 224), Image.BILINEAR)) / 255]})
        s = truth["stenoses"][1]
        return pd.DataFrame({"anjiyogram": [left / 255], "x": [int(s["x"])], "y": [int(s["y"])]})
    if k == "A":
        pats, les = angiogram.registry()
        t = les.merge(pats[["patient_id", "split"]], on="patient_id")
        t = t[t["split"] == "test"]
        return pd.DataFrame({"hasta": t["patient_id"].to_numpy(), "damar": t["vessel"].to_numpy(), "darlık": t["stenosis"].to_numpy(),
                             "görüntü": ["sol" if angiogram.VIEW_OF[v] == "left" else "sağ" for v in t["vessel"]],
                             "anlamlı": [int(coronary.significant(a, b)) for a, b in zip(t["territory"], t["stenosis"])]})
    if k == "D":
        echo, echo_split, echo_patient, echo_view, echo_quality = need("echo", "echo_split", "echo_patient", "echo_view", "echo_quality")
        q = {"Good": "iyi", "Medium": "orta", "Poor": "kötü"}
        return pd.DataFrame({"küme": [sets[s] for s in echo_split], "hasta": echo_patient, "görüntü": echo_view,
                             "kalite": [q[x] for x in echo_quality], "kare": list(echo)})
    if k == "E":
        (syn,) = need("syn")
        return pd.concat([pd.DataFrame({"küme": sets[s], "hasta": syn[s][2], "darlık": syn[s][1]}) for s in syn], ignore_index=True)
    if k == "F":
        return pd.DataFrame([{"küme": sets[s], "anjiyogram": a["id"], "darlıklar": len(a["stenoses"])}
                             for s in arcade.SPLITS for a in arcade.annotations(s)])
    if k == "H":
        maps_te, yt, ARC = need("maps_te", "yt", "ARC")
        return pd.DataFrame({"anjiyogram": ARC["test"][2], "darlık": yt, "dikkat": [m / m.sum() for m in maps_te]})
    if k == "I":
        cnn_arcade, thresholds, test_ann, heat_map = need("cnn_arcade", "thresholds", "test_ann", "heat_map")
        return pd.DataFrame({"anjiyogram": [a["id"] for a in test_ann],
                             "olasılıklar": [heat_map(cnn_arcade, arcade.load("test", a["file"]))[0] for a in test_ann],
                             "eşik": thresholds["CNN"]})
    if k == "J":
        (ARC,) = need("ARC")
        return pd.DataFrame({"anjiyogram": ARC["train"][2]})
    raise KeyError(k)
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter Notebook'ta çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-07/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: Ekokardiyografiye aktarım: Kalite kapısı ve mini proje

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-4. bölümler*

Fotoğraflarla önceden eğitilmiş bir Vision Transformer olan ViT-B/16, CAMUS'un her ekokardiyografi karesini 768 sayıya dönüştürür; bir lojistik regresyon da bu sayılardan uzmanların hangi kareleri kötü bulduğunu öğrenir. Aynı kapı dört basit görüntü istatistiğiyle ve 4. haftanın evrişimli ağı EfficientNet-B0 ile de kurulur; 1.4. adım, iki ağ arasındaki farktan 50 test hastasıyla ne kadar emin olunabileceğini sorar. 1.5. adım oturumun mini projesidir: Küçük bir Vision Transformer önce CIFAR-10 fotoğraflarını, ardından EchoNet-Dynamic ya da CAMUS ekokardiyografi kare çiftlerinden düşük ejeksiyon fraksiyonunu öğrenir.

**Durağın sonunda:** Dört görüntü istatistiğinden (0,704), ViT-B/16'nın sınıf token'ından (0,862) ve EfficientNet-B0'ın özniteliklerinden (0,782) kurulan kapıların test AUC değerlerini, test hastalarından çekilen bootstrap örneklerinde ViT-B/16'nın daha iyi olduğu örneklerin oranını (0,925) ve mini projede küçük Vision Transformer'ın CIFAR-10 doğruluğunu ve iki başlangıcının ekokardiyogramlardaki test AUC değerlerini bulacaksınız.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
if "echo" not in globals():
    camus_table = camus.table()
    echo, echo_patient, echo_view, echo_quality, echo_split = [], [], [], [], []
    for _, row in camus_table.iterrows():
        for view in ("2CH", "4CH"):
            img, _, _ = camus.load(row["patient"], view, "ED")
            echo.append(np.asarray(Image.fromarray(img).resize((224, 224), Image.BILINEAR)) / 255)
            echo_patient.append(row["patient"])
            echo_view.append(view)
            echo_quality.append(row["quality_" + view.lower()])
            echo_split.append(row["split"])
    echo = np.clip(np.stack(echo), 0, 1).astype(np.float32)
    echo_patient, echo_view, echo_quality, echo_split = map(np.array, (echo_patient, echo_view, echo_quality, echo_split))
    etr, eva, ete = echo_split == "training", echo_split == "validation", echo_split == "testing"
    poor = (echo_quality == "Poor").astype(int)
_SETS = {"training": "eğitim", "validation": "doğrulama", "testing": "test"}
_QUALITY = {"Good": "iyi", "Medium": "orta", "Poor": "kötü"}


def _frames():
    """CAMUS'un 600 diyastol sonu karesi; kümeleri, hastaları, görüntüleri ve görüntü kaliteleriyle birlikte, promptların tarif ettiği biçimde."""
    return pd.DataFrame({"küme": [_SETS[s] for s in echo_split], "hasta": echo_patient, "görüntü": echo_view,
                         "kalite": [_QUALITY[q] for q in echo_quality], "kare": list(echo)})


def _course_gate():
    """Dersin kapılarının, ViT-B/16 ve EfficientNet-B0 özniteliklerinden, 100 test karesine verdiği kötü kalite olasılıkları (bir kez hesaplanır, yaklaşık iki dakika sürer)."""
    if "_gate_cache" in globals():
        return _gate_cache
    vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).eval()
    vit.heads = nn.Identity()      # son katmanı olmadan ağ sınıf token'ını döndürür: Görüntü başına 768 sayı
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


    def to_imagenet(images, size=224):
        """0 ile 1 arasında değer alan gri görüntüler → ImageNet ortalaması ve standart sapmasıyla 3 x 224 x 224 boyutlu tensörler."""
        x = torch.as_tensor(np.asarray(images, dtype=np.float32))[:, None]
        x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
        return (x - MEAN) / STD


    @torch.no_grad()
    def vit_features(images, batch=16):
        """ViT-B/16'nın her görüntü için verdiği sınıf token'ı: n x 768 sayıdan oluşan bir dizi."""
        return torch.cat([vit(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()


    @torch.no_grad()
    def vit_attention(image):
        """ViT-B/16'nın 12 katmanının bir görüntü için dikkati, 12 başın ortalaması olarak: 12 x 197 x 197 boyutlu bir dizi."""
        x = vit._process_input(to_imagenet([image]))
        x = torch.cat([vit.class_token, x], dim=1) + vit.encoder.pos_embedding
        maps = []
        for block in vit.encoder.layers:
            y = block.ln_1(x)
            out, w = block.self_attention(y, y, y, need_weights=True, average_attn_weights=True)
            maps.append(w[0].numpy())
            x = x + out
            x = x + block.mlp(block.ln_2(x))
        return np.stack(maps)


    def rollout(maps):
        """Dikkat aktarımı (rollout): Artık bağlantısı eklenmiş her katmanın dikkati, ilk katmandan sonuncuya doğru çarpılır."""
        r = np.eye(maps.shape[-1])
        for a in maps:
            a = 0.5 * a + 0.5 * np.eye(len(a))
            r = (a / a.sum(axis=1, keepdims=True)) @ r
        return r

    effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
    effnet.classifier = nn.Identity()      # son katmanı olmadan ağ 1280 kanalının ortalamasını döndürür


    @torch.no_grad()
    def effnet_features(images, batch=16):
        """EfficientNet-B0'ın her görüntü için verdiği 1280 öznitelik (4. hafta)."""
        return torch.cat([effnet(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()


    def image_statistics(images):
        """Görüntü başına dört sayı: Ortalama gri değer, standart sapma, komşu pikseller arasındaki ortalama fark ve en büyük değerin yarısından parlak piksellerin oranı."""
        return np.array([[x.mean(), x.std(), np.abs(np.diff(x, axis=0)).mean(), (x > 0.5 * x.max()).mean()] for x in images])


    def fit_gate(features, target):
        """Standartlaştırılmış özniteliklerle bir lojistik regresyon; ters düzenlileştirme gücü doğrulama karelerinde seçilir."""
        best = None
        for C in (0.001, 0.01, 0.1, 1.0):
            model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(features[etr], target[etr])
            a = roc_auc_score(target[eva], model.predict_proba(features[eva])[:, 1])
            if best is None or a > best[0]:
                best = (a, C, model)
        return best[2], best[1], best[0]


    def bootstrap_auc(target, prob, groups, n=2000, seed=2026):
        """Grupları (hastaları ya da görüntüleri) yerine koyarak çeken 2000 bootstrap örneğinde AUC."""
        rng = np.random.default_rng(seed)
        names = np.unique(groups)
        rows = {g: np.where(groups == g)[0] for g in names}
        out = []
        for _ in range(n):
            idx = np.concatenate([rows[g] for g in rng.choice(names, len(names))])
            if 0 < target[idx].sum() < len(idx):
                out.append(roc_auc_score(target[idx], prob[idx]))
        return np.array(out)
    out = {}
    for name, features in (("vit", vit_features(echo)), ("effnet", effnet_features(echo))):
        model, _, _ = fit_gate(features, poor)
        out[name] = model.predict_proba(features[ete])[:, 1]
    globals()["_gate_cache"] = out
    return out
print(len(echo), "kare,", len(set(echo_patient)), "hasta")
print("1. durak hazır.")

### Adım 1.1: Dört görüntü istatistiğinden bir kapı

**Amaç.** Bu adımda bir karenin parlaklığını, karşıtlığını ve keskinliğini anlatan dört sayıdan ilk kalite kapısı kurulur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Otomatik bir ejeksiyon fraksiyonu ölçümünden önce kalitesi kötü ekokardiyografi karelerini işaretlemek istiyorum ve her karenin dört basit sayısıyla başlıyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan CAMUS'tan 600 apikal ekokardiyografi karesini, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntülerinin diyastol sonu karelerini tutar. Her satır bir karedir: "küme" sütunu hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "hasta" hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "kalite" ise CAMUS uzmanlarının verdiği görüntü kalitesini ("iyi", "orta" ya da "kötü") verir. "kare" sütununun her hücresi kareyi gri değerleri 0 ile 1 arasında olan 224 × 224 piksellik bir görüntü olarak tutar.  
> Görev: Her kare için dört sayı hesapla: Gri değerlerinin ortalaması, standart sapmaları, dikey komşu pikseller arasındaki mutlak farkın ortalaması ve karenin en parlak pikselinin yarısından daha parlak piksellerin oranı. Hedef, kalitesi kötü olan kare için 1, öteki kareler için 0'dır. Öznitelikleri eğitim kareleriyle standartlaştır. Eğitim karelerinde 0,001, 0,01, 0,1 ve 1 ters düzenlileştirme güçlerinin her biriyle bir lojistik regresyon eğit; doğrulama karelerinde ROC eğrisinin altındaki alanı (AUC) en büyük olanı seç ve test karelerinde bu modelin olasılıklarının AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Seçilen gücü, doğrulama ve test AUC değerlerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: AUC 0 ile 1 arasında olmalı ve test kareleri son adımdan önce kullanılmamalı.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _frames()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _frames()
kumeler = veri["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
hedef = (veri["kalite"] == "kötü").astype(int).to_numpy()


def kapi(oznitelikler):
    en_iyi = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        model.fit(oznitelikler[eg_], hedef[eg_])
        auc_dog = roc_auc_score(hedef[do_], model.predict_proba(oznitelikler[do_])[:, 1])
        if en_iyi is None or auc_dog > en_iyi[0]:
            en_iyi = (auc_dog, C, model)
    auc_test = roc_auc_score(hedef[te_], en_iyi[2].predict_proba(oznitelikler[te_])[:, 1])
    print("ters düzenlileştirme gücü", en_iyi[1], "| doğrulama AUC", round(en_iyi[0], 3), "| test AUC", round(auc_test, 3))
    return round(float(auc_test), 3)

kareler = np.stack(veri["kare"])
istatistik = np.array([[x.mean(), x.std(), np.abs(np.diff(x, axis=0)).mean(), (x > 0.5 * x.max()).mean()] for x in kareler])
cevap = kapi(istatistik)
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** Dört sayı 0,704 test AUC değeri verir: Kötü kareler çoğu zaman daha karanlık ve daha az keskindir; ancak bu sayılar kalp hakkında hiçbir şey bilmez.

### Adım 1.2: Bir Vision Transformer'ın sınıf token'ı

**Amaç.** Bu adımda her kare, ImageNet ile önceden eğitilmiş ViT-B/16'nın sınıf token'ındaki 768 sayıyla tarif edilir ve kapı bu sayılarla kurulur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Milyonlarca fotoğraftan öğrenmiş ve hiç ekokardiyogram görmemiş bir Vision Transformer'ın, kalitesi kötü kareleri işaretleyecek kadar iyi tarif edip edemediğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan CAMUS'tan 600 apikal ekokardiyografi karesini, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntülerinin diyastol sonu karelerini tutar. Her satır bir karedir: "küme" sütunu hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "hasta" hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "kalite" ise CAMUS uzmanlarının verdiği görüntü kalitesini ("iyi", "orta" ya da "kötü") verir. "kare" sütununun her hücresi kareyi gri değerleri 0 ile 1 arasında olan 224 × 224 piksellik bir görüntü olarak tutar.  
> Görev: ImageNet ağırlıklarıyla Vision Transformer ViT-B/16'yı son katmanı olmadan kullan; böylece ağ her kare için son katmanındaki sınıf token'ını, yani 768 sayıyı döndürür. Her kareyi ağa, üç renk kanalında aynı gri değerleri taşıyan 224 × 224 piksellik bir görüntü olarak ve ImageNet'in ortalaması ve standart sapmasıyla ölçeklenmiş biçimde ver: Üç kanaldan 0,485, 0,456 ve 0,406 değerlerini çıkar, kanalları 0,229, 0,224 ve 0,225 değerlerine böl. Kareleri ağdan 16'şarlık gruplar hâlinde, değerlendirme modunda ve gradyan kaydetmeden geçir. Ardından 768 sayıyı kapının öznitelikleri olarak kullan. Hedef, kalitesi kötü olan kare için 1, öteki kareler için 0'dır. Öznitelikleri eğitim kareleriyle standartlaştır. Eğitim karelerinde 0,001, 0,01, 0,1 ve 1 ters düzenlileştirme güçlerinin her biriyle bir lojistik regresyon eğit; doğrulama karelerinde ROC eğrisinin altındaki alanı (AUC) en büyük olanı seç ve test karelerinde bu modelin olasılıklarının AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Seçilen gücü, doğrulama ve test AUC değerlerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her kare 768 sayı vermeli ve AUC 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _frames()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _frames()
kumeler = veri["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
hedef = (veri["kalite"] == "kötü").astype(int).to_numpy()


def kapi(oznitelikler):
    en_iyi = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        model.fit(oznitelikler[eg_], hedef[eg_])
        auc_dog = roc_auc_score(hedef[do_], model.predict_proba(oznitelikler[do_])[:, 1])
        if en_iyi is None or auc_dog > en_iyi[0]:
            en_iyi = (auc_dog, C, model)
    auc_test = roc_auc_score(hedef[te_], en_iyi[2].predict_proba(oznitelikler[te_])[:, 1])
    print("ters düzenlileştirme gücü", en_iyi[1], "| doğrulama AUC", round(en_iyi[0], 3), "| test AUC", round(auc_test, 3))
    return round(float(auc_test), 3)

kareler = np.stack(veri["kare"]).astype(np.float32)


@torch.no_grad()
def oznitelik_cikar(ag):
    ag.eval()
    ortalama = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    sapma = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    cikti = []
    for i in range(0, len(kareler), 16):
        x = torch.as_tensor(kareler[i:i + 16])[:, None].repeat(1, 3, 1, 1)
        cikti.append(ag((x - ortalama) / sapma))
    return torch.cat(cikti).numpy()

vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
vit.heads = nn.Identity()
oznitelikler = oznitelik_cikar(vit)
print(oznitelikler.shape)
cevap = kapi(oznitelikler)
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** ViT-B/16'nın sınıf token'ı 0,862 test AUC değeri verir. Ağ hiç ekokardiyogram görmemiştir; ama 768 sayısı kenarları, dokuları ve biçimleri bir kapı kuracak kadar iyi tarif eder.

### Adım 1.3: Evrişimli bir ağın öznitelikleri

**Amaç.** Bu adımda aynı kapı, 4. haftanın ağı EfficientNet-B0'ın 1.280 özniteliğiyle kurulur.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Ekokardiyografi kareleri için bir kalite kapısında öznitelik kaynağı olarak bir Vision Transformer'ı 4. haftada kullanılan evrişimli ağla karşılaştırmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan CAMUS'tan 600 apikal ekokardiyografi karesini, yani 300 hastanın apikal iki boşluk ve dört boşluk görüntülerinin diyastol sonu karelerini tutar. Her satır bir karedir: "küme" sütunu hastanın kümesini ("eğitim", 200 hasta; "doğrulama", 50 hasta; "test", 50 hasta), "hasta" hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH"), "kalite" ise CAMUS uzmanlarının verdiği görüntü kalitesini ("iyi", "orta" ya da "kötü") verir. "kare" sütununun her hücresi kareyi gri değerleri 0 ile 1 arasında olan 224 × 224 piksellik bir görüntü olarak tutar.  
> Görev: ImageNet ağırlıklarıyla evrişimli ağ EfficientNet-B0'ı son katmanı olmadan kullan; böylece ağ her kare için ortalama havuzlamasının çıktısını, yani 1.280 sayıyı döndürür. Her kareyi ağa, üç renk kanalında aynı gri değerleri taşıyan 224 × 224 piksellik bir görüntü olarak ve ImageNet'in ortalaması ve standart sapmasıyla ölçeklenmiş biçimde ver: Üç kanaldan 0,485, 0,456 ve 0,406 değerlerini çıkar, kanalları 0,229, 0,224 ve 0,225 değerlerine böl. Kareleri ağdan 16'şarlık gruplar hâlinde, değerlendirme modunda ve gradyan kaydetmeden geçir. Ardından 1.280 sayıyı kapının öznitelikleri olarak kullan. Hedef, kalitesi kötü olan kare için 1, öteki kareler için 0'dır. Öznitelikleri eğitim kareleriyle standartlaştır. Eğitim karelerinde 0,001, 0,01, 0,1 ve 1 ters düzenlileştirme güçlerinin her biriyle bir lojistik regresyon eğit; doğrulama karelerinde ROC eğrisinin altındaki alanı (AUC) en büyük olanı seç ve test karelerinde bu modelin olasılıklarının AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Seçilen gücü, doğrulama ve test AUC değerlerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her kare 1.280 sayı vermeli ve AUC 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _frames()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _frames()
kumeler = veri["küme"].to_numpy()
eg_, do_, te_ = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
hedef = (veri["kalite"] == "kötü").astype(int).to_numpy()


def kapi(oznitelikler):
    en_iyi = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        model.fit(oznitelikler[eg_], hedef[eg_])
        auc_dog = roc_auc_score(hedef[do_], model.predict_proba(oznitelikler[do_])[:, 1])
        if en_iyi is None or auc_dog > en_iyi[0]:
            en_iyi = (auc_dog, C, model)
    auc_test = roc_auc_score(hedef[te_], en_iyi[2].predict_proba(oznitelikler[te_])[:, 1])
    print("ters düzenlileştirme gücü", en_iyi[1], "| doğrulama AUC", round(en_iyi[0], 3), "| test AUC", round(auc_test, 3))
    return round(float(auc_test), 3)

kareler = np.stack(veri["kare"]).astype(np.float32)


@torch.no_grad()
def oznitelik_cikar(ag):
    ag.eval()
    ortalama = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    sapma = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    cikti = []
    for i in range(0, len(kareler), 16):
        x = torch.as_tensor(kareler[i:i + 16])[:, None].repeat(1, 3, 1, 1)
        cikti.append(ag((x - ortalama) / sapma))
    return torch.cat(cikti).numpy()

effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
effnet.classifier = nn.Identity()
oznitelikler = oznitelik_cikar(effnet)
print(oznitelikler.shape)
cevap = kapi(oznitelikler)
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** EfficientNet-B0 0,782 test AUC değeri verir; ViT-B/16'nın değeri ise 0,862 düzeyindedir. İki ağ aynı fotoğraflardan öğrenmiştir; aradaki fark, dayandığı 15 kötü test karesinin yanında küçüktür.

### Adım 1.4: Farktan ne kadar emin olabiliriz?

**Amaç.** Bu adımda 50 test hastası yerine koyarak tekrar tekrar çekilir ve Vision Transformer'ın ne sıklıkla daha iyi olduğu görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: 50 test hastasının karelerinde ölçülen iki ekokardiyografi kalite kapısı arasındaki farkın başka hastalarda da geçerli olup olmayacağını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir apikal ekokardiyografi veri seti olan CAMUS'un 100 test karesini, yani 50 hastanın apikal iki boşluk ve dört boşluk görüntülerinin diyastol sonu karelerini tutar. Her satır bir karedir: "hasta" sütunu hastayı, "görüntü" görüntüyü ("2CH" ya da "4CH") verir; "kötü" sütunu CAMUS uzmanları görüntü kalitesini kötü olarak değerlendirdiyse 1, değilse 0 değerini alır. "olasılık, ViT-B/16" ve "olasılık, EfficientNet-B0" adlı iki sütun, dersin iki kalite kapısının verdiği kötü kalite olasılığını tutar; bu kapılar, ImageNet ile önceden eğitilmiş iki ağın öznitelikleri üzerinde kurulmuş lojistik regresyonlardır.  
> Görev: İki olasılık sütununun "kötü" sütununa göre AUC değerini hesapla. Ardından 2.000 bootstrap örneği çek: Her örnek 50 test hastası arasından yerine koyarak 50 hasta çeker ve çekilen her hastanın iki karesini de tutar. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. İki tür kareyi de içeren her örnekte iki kapının AUC değerini ve farkı (ViT-B/16 eksi EfficientNet-B0) hesapla. Ortalama farkı, 2,5. ve 97,5. yüzdelikler arasındaki yüzde 95 aralığını ve farkın 0'ın üstünde olduğu örneklerin oranını bildir.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki AUC değerini, ortalama farkı ve aralığını yazdır; oranı üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Oran 0 ile 1 arasında olmalı ve bir hastanın iki karesi her zaman birlikte çekilmeli.  

In [ ]:
#@title Adım 1.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_g = _course_gate()   # ilk seferde yaklaşık iki dakika sürer: 600 karenin öznitelikleri bir kez hesaplanır
_t = _frames().loc[ete].reset_index(drop=True)
veri = pd.DataFrame({"hasta": _t["hasta"], "görüntü": _t["görüntü"], "kötü": (_t["kalite"] == "kötü").astype(int),
                     "olasılık, ViT-B/16": _g["vit"], "olasılık, EfficientNet-B0": _g["effnet"]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.4 kontrolü
teach.check_step("Adım 1.4", "cevap", globals())

In [ ]:
#@title Adım 1.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_g = _course_gate()   # ilk seferde yaklaşık iki dakika sürer: 600 karenin öznitelikleri bir kez hesaplanır
_t = _frames().loc[ete].reset_index(drop=True)
veri = pd.DataFrame({"hasta": _t["hasta"], "görüntü": _t["görüntü"], "kötü": (_t["kalite"] == "kötü").astype(int),
                     "olasılık, ViT-B/16": _g["vit"], "olasılık, EfficientNet-B0": _g["effnet"]})
y = veri["kötü"].to_numpy()
pv = veri["olasılık, ViT-B/16"].to_numpy()
pe = veri["olasılık, EfficientNet-B0"].to_numpy()
print("AUC ViT-B/16", round(roc_auc_score(y, pv), 3), "| AUC EfficientNet-B0", round(roc_auc_score(y, pe), 3))
hastalar = veri["hasta"].to_numpy()
adlar = np.unique(hastalar)
satirlar = {p: np.where(hastalar == p)[0] for p in adlar}
rng = np.random.default_rng(2026)
farklar = []
for _ in range(2000):
    idx = np.concatenate([satirlar[p] for p in rng.choice(adlar, len(adlar))])
    if 0 < y[idx].sum() < len(idx):
        farklar.append(roc_auc_score(y[idx], pv[idx]) - roc_auc_score(y[idx], pe[idx]))
farklar = np.array(farklar)
print("ortalama fark", round(farklar.mean(), 3), "| yüzde 95 aralığı", np.round(np.percentile(farklar, [2.5, 97.5]), 3))
cevap = round(float((farklar > 0).mean()), 3)
teach.check_step("Adım 1.4", "cevap", globals())

**Sonuç ne gösteriyor?** ViT-B/16 örneklerin 0,925 kadarında daha iyidir. Farkın aralığı 0'ı içerir: 50 test hastası ve 15 kötü kareyle veriler iki ağ arasında karar vermeye yetmez.

### Adım 1.5: Mini proje: CIFAR-10'dan ekokardiyografiye küçük bir Vision Transformer

**Amaç.** Bu adımda küçük bir Vision Transformer önce CIFAR-10 fotoğraflarıyla eğitilir, ardından düşük ejeksiyon fraksiyonunu tanımak için ekokardiyografi kare çiftleriyle ince ayar yapılır ve yalnızca ekokardiyogramlarla eğitilen aynı ağla karşılaştırılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Önce gündelik fotoğraflardan öğrenmiş küçük bir Vision Transformer'ın, ekokardiyogramlarda düşük ejeksiyon fraksiyonunu, yalnızca ekokardiyogramlardan öğrenen aynı ağdan daha iyi tanıyıp tanımadığını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta iki tür satırdan oluşan "veri" adlı bir tablo vardır. "kaynak" değeri "CIFAR-10" olan satırlar, uçak, kedi ve gemi gibi on gündelik sınıftan oluşan açık bir veri seti olan CIFAR-10'un 60.000 renkli fotoğrafıdır: "küme" sütunu "eğitim" (50.000 fotoğraf) ya da "test" (10.000) değerini alır, "görüntü" sütununun her hücresi bir fotoğrafı 0 ile 255 arasındaki sayılardan oluşan 32 × 32 × 3 boyutlu bir dizi olarak tutar, "sınıf" sütunu da fotoğrafın sınıfını 0 ile 9 arasında bir sayı olarak verir. "kaynak" değeri "eko" olan satırlar, iki açık veri seti olan EchoNet-Dynamic ya da CAMUS'un apikal ekokardiyografi kare çiftleridir: "küme" sütunu "eğitim", "doğrulama" ya da "test" değerini alır, "hasta" sütunu hastayı ya da videoyu verir, "görüntü" sütununun her hücresi 32 × 32 × 3 boyutlu bir dizi tutar; bu dizinin üç kanalı, değerleri 0 ile 1 arasında olan diyastol sonu ve sistol sonu kareler ile değerleri −1 ile 1 arasında olan farklarıdır. "düşük EF" sütunu ejeksiyon fraksiyonu yüzde 50'nin altındaysa 1, değilse 0 olur.  
> Görev: 32 × 32 piksellik renkli görüntüler için küçük bir Vision Transformer kur. Görüntüyü 4 × 4 piksellik 64 kareye böl ve her kareyi aynı öğrenilen doğrusal dönüşümle 64 sayıya dönüştür. 64 token'ın önüne 64 sayılık, öğrenilen bir sınıf token'ı koy ve öğrenilen konumları ekle: 65 yerin her biri için, standart sapması 0,02 olan küçük rastgele sayılarla başlatılan 64 sayılık bir vektör. Token'ları dört Transformer bloğundan geçir. Her blokta katman normalleştirmesini dört başlı öz dikkat izler ve öz dikkatin çıktısı bloğun girdisine eklenir; ikinci bir katman normalleştirmesini 128 birimlik bir gizli katman ve GELU etkinleştirmesi içeren ileri beslemeli bir ağ izler ve bu ağın çıktısı da yeniden eklenir. Sonda sınıf token'ını al, katman normalleştirmesi uygula ve on çıktılı, yani on sınıfın puanlarını veren doğrusal bir katmandan geçir. Ağı eğitim kümesindeki CIFAR-10 satırlarıyla eğit: Fotoğrafları 255'e böl ve her renk kanalını eğitim fotoğraflarının ortalaması ve standart sapmasıyla standartlaştır. 128 fotoğraflık karıştırılmış gruplarla, çapraz entropi kaybıyla ve öğrenme hızı 0,001, ağırlık azaltması 0,05 olan AdamW optimizer'ı ile 5 epoch eğit. Her grubu her yönde 4 piksele kadar rastgele kaydır, kenarları yansıtmayla doldur ve yarı olasılıkla soldan sağa aynala. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. 10.000 test fotoğrafında doğruluğu hesapla. Ardından ekokardiyografi satırları için, her biri tek çıktılı, yani düşük ejeksiyon fraksiyonunun logitini veren doğrusal bir katmanla biten iki ağ hazırla: Biri aynı yapıda ve yeni rastgele ağırlıklarla, diğeri de CIFAR-10 ile eğitilen ağın, son katmanı tek çıktılı yeni bir katmanla değiştirilmiş bir kopyası olsun. Ekokardiyografi görüntülerini oldukları gibi kullan. Her ağı eğitim satırlarıyla, 32'lik karıştırılmış gruplarla, logit üzerinde ikili çapraz entropi kaybıyla ve öğrenme hızı 0,0005 olan Adam optimizer'ı ile 30 epoch eğit. Her epoch'tan sonra doğrulama satırlarında AUC değerini hesapla ve doğrulama AUC değeri en büyük olan epoch'un ağırlıklarını sakla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. İki ağın test AUC değerini "düşük EF" sütununa göre hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: CIFAR-10 doğruluğunu ve iki ağın test AUC değerini yazdır; CIFAR-10 ile başlayan ağın test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: CIFAR-10 doğruluğu, on sınıftan birinin şans eseri verdiği 0,1 değerinin belirgin biçimde üstünde olmalı ve test satırları sona kadar kullanılmamalı.  

In [ ]:
#@title Adım 1.5 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
class Attention(nn.Module):
    """Çok başlı öz dikkat; son çağrının ağırlıkları haritalar için saklanır."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """Bir Transformer bloğu: Dikkat ve iki katmanlı küçük bir ağ; her biri katman normalleştirmesinden sonra gelir ve bir artık bağlantısı taşır."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class SmallViT(nn.Module):
    """Küçük bir Vision Transformer: Token olarak 8 x 8 piksellik kareler, bir sınıf token'ı, öğrenilen konumlar ve dört blok."""
    def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
        super().__init__()
        self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # her kare için aynı doğrusal dönüşüm
        n = (size // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, outputs)

    def encode(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        for block in self.blocks:
            t = block(t)
        out = self.head(self.norm(t)[:, 0])
        return out.squeeze(1) if out.shape[1] == 1 else out

    def forward(self, x):
        return self.encode(self.embed(x).flatten(2).transpose(1, 2))


def n_parameters(model):
    """Bir ağın eğitilebilir ağırlıklarının sayısı."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

from torchvision.datasets import CIFAR10
from vclinic import datasets, echonet

CIFAR_CLASSES = ["uçak", "otomobil", "kuş", "kedi", "geyik",
                 "köpek", "kurbağa", "at", "gemi", "kamyon"]


def cifar10():
    """Toronto Üniversitesindeki kaynağından CIFAR-10; yaklaşık 170 MB, data/cifar10 klasörüne bir kez indirilir: Üç renk kanallı, 32 x 32 piksellik 50.000 eğitim ve 10.000 test görüntüsü ile sınıfları."""
    folder = str(datasets.data_path("cifar10"))
    out = {}
    for part in ("training", "test"):
        d = CIFAR10(folder, train=(part == "training"), download=True)
        out[part] = (d.data, np.array(d.targets))
    return out


def to_tensor(images, mean, std):
    """n x 32 x 32 x 3 boyutlu görüntüler (0-255 değerleri) → n x 3 x 32 x 32 boyutlu tensörler; her kanal eğitim görüntülerinin ortalaması ve standart sapmasıyla standartlaştırılır."""
    x = torch.as_tensor(np.asarray(images), dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.as_tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)) / torch.as_tensor(std, dtype=torch.float32).view(1, 3, 1, 1)


def shift_and_mirror(x):
    """Veri artırma: Grup her yönde 4 piksele kadar kaydırılır ve yarı yarıya soldan sağa aynalanır."""
    i, j = (int(v) for v in torch.randint(0, 9, (2,)))
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")[:, :, i:i + 32, j:j + 32]
    return torch.flip(x, dims=(3,)) if torch.rand(1) < 0.5 else x


def train_cifar(model, x, y, epochs=5, batch=128, seed=2026):
    """Karıştırılmış 128 görüntülük gruplar üzerinde 0,001 öğrenme hızı ve 0,05 ağırlık azaltmasıyla AdamW; kayıp on sınıfın çapraz entropisidir."""
    torch.manual_seed(seed)
    optimiser = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        order, total = torch.randperm(len(x)), 0.0
        for i in range(0, len(x), batch):
            b = order[i:i + batch]
            loss = loss_fn(model(shift_and_mirror(x[b])), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total += loss.item() * len(b)
        print(f"epoch {epoch + 1}: eğitim kaybı {total / len(x):.3f}")


@torch.no_grad()
def accuracy(model, x, y, batch=500):
    """Sınıfı en yüksek puanı alan görüntülerin oranı."""
    model.eval()
    return float(torch.cat([model(x[i:i + batch]).argmax(1) for i in range(0, len(x), batch)]).eq(y).float().mean())


def camus_pairs():
    """CAMUS'un 300 hastasının iki görüntüsünün diyastol sonu ve sistol sonu kareleri, 32 x 32 piksellik çiftler olarak ve hastanın EF değeriyle."""
    rows = []
    for _, r in camus.table().iterrows():
        for view in ("2CH", "4CH"):
            pair = [np.asarray(Image.fromarray(camus.load(r["patient"], view, phase)[0]).resize((32, 32), Image.BILINEAR)) / 255 for phase in ("ED", "ES")]
            rows.append({"küme": {"training": "TRAIN", "validation": "VAL", "testing": "TEST"}[r["split"]], "hasta": r["patient"],
                         "çift": np.stack(pair).astype(np.float32), "EF": r["ef"]})
    return pd.DataFrame(rows)


def echonet_pairs(source, n=(2000, 400, 400)):
    """EchoNet-Dynamic videolarının çizilmiş iki karesi, 32 x 32 piksellik çiftler olarak ve EF değeriyle: 2.000 eğitim, 400 doğrulama ve 400 test videosu."""
    parts = []
    for split, k in zip(("TRAIN", "VAL", "TEST"), n):
        d = echonet.frame_pairs(source, split=split, n=k, size=32)
        parts.append(pd.DataFrame({"küme": split, "hasta": d["file"], "çift": list(d["images"]), "EF": d["ef"]}))
    return pd.concat(parts, ignore_index=True)


def three_channels(pairs):
    """Bir kare çifti → üç kanal: Diyastol sonu kare, sistol sonu kare ve farkları; bir fotoğrafın kırmızı, yeşil ve mavi kanallarının yerinde."""
    p = np.stack(pairs)
    return np.stack([p[:, 0], p[:, 1], p[:, 1] - p[:, 0]], axis=1).astype(np.float32)


def fine_tune(model, x, y, xv, yv, epochs=30, lr=0.0005, seed=2026):
    """Karıştırılmış 32 çiftlik gruplar üzerinde Adam; kayıp, düşük EF için ikili çapraz entropidir. Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        for i in range(0, len(x), 32):
            b = order[i:i + 32]
            loss = loss_fn(model(x[b]), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            a = roc_auc_score(yv.numpy(), torch.sigmoid(model(xv)).numpy())
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
    model.load_state_dict(state)
    return best

try:
    _cifar = cifar10()
except Exception as _error:
    _cifar = None
    print("CIFAR-10 indirilemedi:", type(_error).__name__)
try:
    _source = echonet.find()
except FileNotFoundError:
    _source = None
_pairs = echonet_pairs(_source) if _source is not None else camus_pairs()
_SET = {"TRAIN": "eğitim", "VAL": "doğrulama", "TEST": "test"}
_rows = []
if _cifar is not None:
    for _part, (_images, _labels) in _cifar.items():
        _rows.append(pd.DataFrame({"kaynak": "CIFAR-10", "küme": "eğitim" if _part == "training" else "test",
                                   "hasta": None, "görüntü": list(_images), "sınıf": _labels, "düşük EF": np.nan}))
_rows.append(pd.DataFrame({"kaynak": "eko", "küme": _pairs["küme"].map(_SET).to_numpy(),
                           "hasta": _pairs["hasta"].to_numpy(),
                           "görüntü": list(three_channels(_pairs["çift"]).transpose(0, 2, 3, 1)),
                           "sınıf": np.nan, "düşük EF": (_pairs["EF"] < 50).astype(int).to_numpy()}))
veri = pd.concat(_rows, ignore_index=True)
print("ekokardiyogramlar:", "EchoNet-Dynamic" if _source is not None else "CAMUS")
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.5 kontrolü
if globals().get("_cifar") is None:
    print("CIFAR-10 burada indirilemedi; bu yüzden bu adım bu bilgisayarda çalışamaz. Adımı Google Colab'da ya da internete erişimi olan bir bilgisayarda çalıştırın.")
else:
    teach.check_range("Adım 1.5", "cevap", 0.5, 1.0, globals())

In [ ]:
#@title Adım 1.5 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
class Attention(nn.Module):
    """Çok başlı öz dikkat; son çağrının ağırlıkları haritalar için saklanır."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """Bir Transformer bloğu: Dikkat ve iki katmanlı küçük bir ağ; her biri katman normalleştirmesinden sonra gelir ve bir artık bağlantısı taşır."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class SmallViT(nn.Module):
    """Küçük bir Vision Transformer: Token olarak 8 x 8 piksellik kareler, bir sınıf token'ı, öğrenilen konumlar ve dört blok."""
    def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
        super().__init__()
        self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # her kare için aynı doğrusal dönüşüm
        n = (size // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, outputs)

    def encode(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        for block in self.blocks:
            t = block(t)
        out = self.head(self.norm(t)[:, 0])
        return out.squeeze(1) if out.shape[1] == 1 else out

    def forward(self, x):
        return self.encode(self.embed(x).flatten(2).transpose(1, 2))


def n_parameters(model):
    """Bir ağın eğitilebilir ağırlıklarının sayısı."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

from torchvision.datasets import CIFAR10
from vclinic import datasets, echonet

CIFAR_CLASSES = ["uçak", "otomobil", "kuş", "kedi", "geyik",
                 "köpek", "kurbağa", "at", "gemi", "kamyon"]


def cifar10():
    """Toronto Üniversitesindeki kaynağından CIFAR-10; yaklaşık 170 MB, data/cifar10 klasörüne bir kez indirilir: Üç renk kanallı, 32 x 32 piksellik 50.000 eğitim ve 10.000 test görüntüsü ile sınıfları."""
    folder = str(datasets.data_path("cifar10"))
    out = {}
    for part in ("training", "test"):
        d = CIFAR10(folder, train=(part == "training"), download=True)
        out[part] = (d.data, np.array(d.targets))
    return out


def to_tensor(images, mean, std):
    """n x 32 x 32 x 3 boyutlu görüntüler (0-255 değerleri) → n x 3 x 32 x 32 boyutlu tensörler; her kanal eğitim görüntülerinin ortalaması ve standart sapmasıyla standartlaştırılır."""
    x = torch.as_tensor(np.asarray(images), dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.as_tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)) / torch.as_tensor(std, dtype=torch.float32).view(1, 3, 1, 1)


def shift_and_mirror(x):
    """Veri artırma: Grup her yönde 4 piksele kadar kaydırılır ve yarı yarıya soldan sağa aynalanır."""
    i, j = (int(v) for v in torch.randint(0, 9, (2,)))
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")[:, :, i:i + 32, j:j + 32]
    return torch.flip(x, dims=(3,)) if torch.rand(1) < 0.5 else x


def train_cifar(model, x, y, epochs=5, batch=128, seed=2026):
    """Karıştırılmış 128 görüntülük gruplar üzerinde 0,001 öğrenme hızı ve 0,05 ağırlık azaltmasıyla AdamW; kayıp on sınıfın çapraz entropisidir."""
    torch.manual_seed(seed)
    optimiser = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        order, total = torch.randperm(len(x)), 0.0
        for i in range(0, len(x), batch):
            b = order[i:i + batch]
            loss = loss_fn(model(shift_and_mirror(x[b])), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total += loss.item() * len(b)
        print(f"epoch {epoch + 1}: eğitim kaybı {total / len(x):.3f}")


@torch.no_grad()
def accuracy(model, x, y, batch=500):
    """Sınıfı en yüksek puanı alan görüntülerin oranı."""
    model.eval()
    return float(torch.cat([model(x[i:i + batch]).argmax(1) for i in range(0, len(x), batch)]).eq(y).float().mean())


def camus_pairs():
    """CAMUS'un 300 hastasının iki görüntüsünün diyastol sonu ve sistol sonu kareleri, 32 x 32 piksellik çiftler olarak ve hastanın EF değeriyle."""
    rows = []
    for _, r in camus.table().iterrows():
        for view in ("2CH", "4CH"):
            pair = [np.asarray(Image.fromarray(camus.load(r["patient"], view, phase)[0]).resize((32, 32), Image.BILINEAR)) / 255 for phase in ("ED", "ES")]
            rows.append({"küme": {"training": "TRAIN", "validation": "VAL", "testing": "TEST"}[r["split"]], "hasta": r["patient"],
                         "çift": np.stack(pair).astype(np.float32), "EF": r["ef"]})
    return pd.DataFrame(rows)


def echonet_pairs(source, n=(2000, 400, 400)):
    """EchoNet-Dynamic videolarının çizilmiş iki karesi, 32 x 32 piksellik çiftler olarak ve EF değeriyle: 2.000 eğitim, 400 doğrulama ve 400 test videosu."""
    parts = []
    for split, k in zip(("TRAIN", "VAL", "TEST"), n):
        d = echonet.frame_pairs(source, split=split, n=k, size=32)
        parts.append(pd.DataFrame({"küme": split, "hasta": d["file"], "çift": list(d["images"]), "EF": d["ef"]}))
    return pd.concat(parts, ignore_index=True)


def three_channels(pairs):
    """Bir kare çifti → üç kanal: Diyastol sonu kare, sistol sonu kare ve farkları; bir fotoğrafın kırmızı, yeşil ve mavi kanallarının yerinde."""
    p = np.stack(pairs)
    return np.stack([p[:, 0], p[:, 1], p[:, 1] - p[:, 0]], axis=1).astype(np.float32)


def fine_tune(model, x, y, xv, yv, epochs=30, lr=0.0005, seed=2026):
    """Karıştırılmış 32 çiftlik gruplar üzerinde Adam; kayıp, düşük EF için ikili çapraz entropidir. Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        for i in range(0, len(x), 32):
            b = order[i:i + 32]
            loss = loss_fn(model(x[b]), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            a = roc_auc_score(yv.numpy(), torch.sigmoid(model(xv)).numpy())
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
    model.load_state_dict(state)
    return best

try:
    _cifar = cifar10()
except Exception as _error:
    _cifar = None
    print("CIFAR-10 indirilemedi:", type(_error).__name__)
try:
    _source = echonet.find()
except FileNotFoundError:
    _source = None
_pairs = echonet_pairs(_source) if _source is not None else camus_pairs()
_SET = {"TRAIN": "eğitim", "VAL": "doğrulama", "TEST": "test"}
_rows = []
if _cifar is not None:
    for _part, (_images, _labels) in _cifar.items():
        _rows.append(pd.DataFrame({"kaynak": "CIFAR-10", "küme": "eğitim" if _part == "training" else "test",
                                   "hasta": None, "görüntü": list(_images), "sınıf": _labels, "düşük EF": np.nan}))
_rows.append(pd.DataFrame({"kaynak": "eko", "küme": _pairs["küme"].map(_SET).to_numpy(),
                           "hasta": _pairs["hasta"].to_numpy(),
                           "görüntü": list(three_channels(_pairs["çift"]).transpose(0, 2, 3, 1)),
                           "sınıf": np.nan, "düşük EF": (_pairs["EF"] < 50).astype(int).to_numpy()}))
veri = pd.concat(_rows, ignore_index=True)
print("ekokardiyogramlar:", "EchoNet-Dynamic" if _source is not None else "CAMUS")
if globals().get("_cifar") is None:
    print("CIFAR-10 burada indirilemedi; bu yüzden bu adım bu bilgisayarda çalışamaz. Adımı Google Colab'da ya da internete erişimi olan bir bilgisayarda çalıştırın.")
else:
    _photos = veri[veri["kaynak"] == "CIFAR-10"]
    _train, _test = _photos[_photos["küme"] == "eğitim"], _photos[_photos["küme"] == "test"]
    _images = np.stack(_train["görüntü"])
    _mean, _std = _images.mean(axis=(0, 1, 2)) / 255, _images.std(axis=(0, 1, 2)) / 255
    torch.manual_seed(2026)
    foto_ag = SmallViT(size=32, patch=4, channels=3, outputs=10)
    train_cifar(foto_ag, to_tensor(_images, _mean, _std), torch.as_tensor(_train["sınıf"].to_numpy(int)), epochs=5)
    _accuracy = accuracy(foto_ag, to_tensor(np.stack(_test["görüntü"]), _mean, _std), torch.as_tensor(_test["sınıf"].to_numpy(int)))
    print("CIFAR-10 doğruluğu:", round(_accuracy, 3))
    _echo = veri[veri["kaynak"] == "eko"]
    _X = {s: torch.as_tensor(np.stack(_echo[_echo["küme"] == s]["görüntü"]).transpose(0, 3, 1, 2).copy()) for s in ("eğitim", "doğrulama", "test")}
    _Y = {s: torch.as_tensor(_echo[_echo["küme"] == s]["düşük EF"].to_numpy(np.float32)) for s in _X}
    _aucs = {}
    for _start in ("rastgele ağırlıklar", "CIFAR-10"):
        torch.manual_seed(2026)
        if _start == "CIFAR-10":
            _net = copy.deepcopy(foto_ag)
            _net.head = nn.Linear(64, 1)
        else:
            _net = SmallViT(size=32, patch=4, channels=3)
        fine_tune(_net, _X["eğitim"], _Y["eğitim"], _X["doğrulama"], _Y["doğrulama"])
        with torch.no_grad():
            _aucs[_start] = round(float(roc_auc_score(_Y["test"].numpy(), torch.sigmoid(_net(_X["test"])).numpy())), 3)
    print("test AUC:", _aucs)
    cevap = _aucs["CIFAR-10"]
    teach.check_range("Adım 1.5", "cevap", 0.5, 1.0, globals())

**Sonuç ne gösteriyor?** CIFAR-10 doğruluğu, küçük ağın beş epoch'ta fotoğraflardan ne öğrendiğini gösterir; şans eseri her on sınıftan biri doğru olurdu. İki test AUC değeri bu başlangıcın ekokardiyogramlarda yardımcı olup olmadığını gösterir. İki değer de az sayıda test hastasına, yani CAMUS'un 50 hastasına ya da EchoNet-Dynamic'in 400 videosuna dayanır ve bilgisayardan bilgisayara değişir; bu yüzden kontrol, CIFAR-10 başlangıcı için 0,5 ile 1,0 arasındaki bir AUC değerini kabul eder.

**1. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#last-two-hours-hybrid-cnn-vit-and-medical-imaging) ve okumaya «Son 2 saat: Hibrit CNN-ViT ve medikal görüntüleme» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 2: Sentetik ve gerçek anjiyogramlarda darlıklar

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 5-6. bölümler*

Küçük bir evrişimli ağ, merkezi anlamlı bir darlıkta olan 64 × 64 piksellik yamaları tanımayı önce sanal kliniğin sentetik anjiyogramlarında öğrenir. Aynı ağ daha sonra hiçbir değişiklik yapılmadan ARCADE'in gerçek anjiyogramlarında sınanır ve son olarak ARCADE'in eğitim anjiyogramlarıyla yeniden eğitilir.

**Durağın sonunda:** Ağın sentetik yamalardaki test AUC değerini (0,965), yalnızca sentetik yamalardan öğrenen aynı tür ağın ARCADE'in gerçek yamalarındaki AUC değerini (0,538) ve ağ ARCADE'den öğrendiğinde ulaştığı test AUC değerini (0,81) bulacaksınız.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "train": "eğitim", "val": "doğrulama"}


def _synthetic_table():
    """Sanal kliniğin 300 eğitim, 100 doğrulama ve 100 test hastasının sentetik anjiyogramlarından yamalar."""
    if "_synthetic_cache" not in globals():
        parts = []
        for s, n in (("training", 300), ("validation", 100), ("test", 100)):
            X, y, ids = angiogram.patches(s, n_patients=n)
            parts.append(pd.DataFrame({"küme": _SETS[s], "hasta": ids, "yama": list(X), "darlık": y}))
        globals()["_synthetic_cache"] = pd.concat(parts, ignore_index=True)
    return _synthetic_cache


def _arcade_table():
    """ARCADE'in (darlık kümesi) 1500 anjiyogramından yamalar; kümeleri ve anjiyogramlarıyla birlikte."""
    if "_arcade_cache" not in globals():
        arcade.download()
        parts = []
        for s in arcade.SPLITS:
            X, y, ids = arcade.patches(s)
            parts.append(pd.DataFrame({"küme": _SETS[s], "anjiyogram": ids, "yama": list(X), "darlık": y}))
        globals()["_arcade_cache"] = pd.concat(parts, ignore_index=True)
    return _arcade_cache


def _course_model(kind):
    """Dersin bir ağı; eğitim yamalarıyla bir kez eğitilir ve sonraki adımlar için saklanır (bir ile dört dakika sürer)."""
    key = "_course_" + kind
    if key in globals():
        return globals()[key]
    def standardise(X):
        """Her yama kendi içinde: Ortalama gri değer 0, standart sapma 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """3 x 3 evrişim, batch normalizasyonu, ReLU ve 2 x 2 en büyük havuzlama: Boyut yarıya iner, b kanal elde edilir."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Dört evrişim bloğu (16, 32, 64 ve 64 kanal), her kanalın ortalaması ve tek bir çıktı."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Çok başlı öz dikkat; son çağrının ağırlıkları haritalar için saklanır."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """Bir Transformer bloğu: Dikkat ve iki katmanlı küçük bir ağ; her biri katman normalleştirmesinden sonra gelir ve bir artık bağlantısı taşır."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """Küçük bir Vision Transformer: Token olarak 8 x 8 piksellik kareler, bir sınıf token'ı, öğrenilen konumlar ve dört blok."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # her kare için aynı doğrusal dönüşüm
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """Bir ağın eğitilebilir ağırlıklarının sayısı."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """Hibrit model: Üç evrişim bloğu yamayı 64 sayılık 8 x 8 konuma dönüştürür; bunlar iki Transformer bloğunun token'larıdır."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """Her yama için darlık olasılığı."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam, 0,001 öğrenme hızı, rastgele bir çeyrek tur döndürülen ve yarı yarıya aynalanan 64 yamalık gruplar; doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best
    t = _synthetic_table() if kind == "synthetic_cnn" else _arcade_table()
    sets = t["küme"].to_numpy()
    X, y = standardise(np.stack(t["yama"])), t["darlık"].to_numpy()
    tr_, va_ = sets == "eğitim", sets == "doğrulama"
    torch.manual_seed(2026)
    model = SmallCNN() if kind == "synthetic_cnn" else SmallViT()
    train(model, X[tr_], y[tr_], X[va_], y[va_], epochs=8, verbose=False)
    globals()[key] = (model, standardise, predict)
    return globals()[key]

print("2. durağın tabloları adımları çalıştığında kurulur: Sentetik yamalar yaklaşık iki dakikada, ARCADE (451,6 MB, bir kez indirilir) ve yamaları birkaç dakikada hazırlanır.")
print("2. durak hazır.")

### Adım 2.1: Sentetik anjiyogramlarda küçük bir evrişimli ağ

**Amaç.** Bu adımda sentetik anjiyogramların yamalarıyla küçük bir evrişimli ağ eğitilir ve ağ test hastalarının yamalarında ölçülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Küçük bir ağın koroner anjiyogramlardan, bir görüntünün küçük bir karesinin merkezinin anlamlı bir darlıkta olup olmadığını öğrenip öğrenemeyeceğini görmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; sanal kliniğin sentetik koroner anjiyogramlarından, 500 hastanın her biri için iki görüntüden (eğitim 300 hasta, doğrulama 100, test 100) kesilmiş 64 × 64 piksellik yamaları tutar. Her satır bir yamadır: "küme" sütunu yamanın kümesini ("eğitim", "doğrulama" ya da "test"), "hasta" hastayı verir; "yama" sütununun her hücresi yamayı gri değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü olarak tutar. "darlık" sütunu, yamanın merkezi anlamlı bir darlığa (sol ana koroner arterde yüzde 50 ya da daha fazla, öteki arterlerde yüzde 70 ya da daha fazla) 6 pikselden yakınsa 1, yama her darlıktan en az 48 piksel uzakta bir damarın üzerindeyse 0 değerini alır.  
> Görev: Her yamayı kendi içinde standartlaştır: Gri değerlerinin ortalamasını çıkar ve standart sapmalarının 0,02 fazlasına böl. Tek kanallı, 64 × 64 piksellik bir yama için küçük bir evrişimli ağ kur: Her biri 3 × 3 piksellik filtreler ve bir piksellik dolgu içeren bir evrişim (16, 32, 64 ve 64 filtre), batch normalizasyonu, ReLU ve 2 × 2 en büyük havuzlamadan oluşan dört blok; ardından her kanalın konumları üzerinden ortalaması ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman. Ağı eğitim yamalarıyla 8 epoch boyunca, karıştırılmış 64 yamalık gruplar hâlinde, logit üzerinde ikili çapraz entropi kaybıyla ve öğrenme hızı 0,001 olan Adam optimizasyon algoritmasıyla eğit. Veri çoğaltma için her grubu rastgele bir çeyrek tur döndür ve yarı olasılıkla soldan sağa aynala. Her epoch'un sonunda doğrulama yamalarının olasılıklarının AUC değerini hesapla; eğitimin sonunda doğrulama AUC değeri en büyük olan epoch'un ağırlıklarını sakla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Ardından test yamalarının olasılıklarını ve bunların "darlık" sütununa göre AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitilebilir ağırlık sayısını ve her epoch'un doğrulama AUC değerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Ağın 60.641 eğitilebilir ağırlığı olmalı ve test yamaları son adımdan önce kullanılmamalı.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _synthetic_table()   # yaklaşık iki dakika sürer: Sanal klinik 1000 anjiyogram çizer
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _synthetic_table()   # yaklaşık iki dakika sürer: Sanal klinik 1000 anjiyogram çizer


def standartlastir(yamalar):
    yamalar = np.asarray(yamalar, dtype=np.float32)
    o = yamalar.mean(axis=(1, 2), keepdims=True)
    s = yamalar.std(axis=(1, 2), keepdims=True)
    return (yamalar - o) / (s + 0.02)


def blok(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class Ag(nn.Module):
    def __init__(self):
        super().__init__()
        self.oznitelik = nn.Sequential(blok(1, 16), blok(16, 32), blok(32, 64), blok(64, 64))
        self.cikis = nn.Linear(64, 1)

    def forward(self, x):
        return self.cikis(self.oznitelik(x).mean(dim=(2, 3))).squeeze(1)

torch.manual_seed(2026)
ag = Ag()
print("eğitilebilir ağırlık:", sum(p.numel() for p in ag.parameters() if p.requires_grad))
kumeler = veri["küme"].to_numpy()
X = torch.as_tensor(standartlastir(np.stack(veri["yama"])))[:, None]
y = torch.as_tensor(veri["darlık"].to_numpy(), dtype=torch.float32)
eg_, do_, te_ = [torch.as_tensor(kumeler == s) for s in ("eğitim", "doğrulama", "test")]


@torch.no_grad()
def olasiliklar(X_):
    ag.eval()
    return torch.cat([torch.sigmoid(ag(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xeg, yeg = X[eg_], y[eg_]
torch.manual_seed(2026)
optimizer = torch.optim.Adam(ag.parameters(), lr=0.001)
kayip = nn.BCEWithLogitsLoss()
en_iyi, durum = -1.0, None
for epoch in range(8):
    ag.train()
    sira = torch.randperm(len(Xeg))
    for i in range(0, len(Xeg), 64):
        g = sira[i:i + 64]
        xg = torch.rot90(Xeg[g], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xg = torch.flip(xg, dims=(3,))
        k = kayip(ag(xg), yeg[g])
        optimizer.zero_grad()
        k.backward()
        optimizer.step()
    auc = roc_auc_score(y[do_].numpy(), olasiliklar(X[do_]))
    print(f"epoch {epoch + 1}: doğrulama AUC {auc:.3f}")
    if auc > en_iyi:
        en_iyi, durum = auc, copy.deepcopy(ag.state_dict())
ag.load_state_dict(durum)
cevap = round(float(roc_auc_score(y[te_].numpy(), olasiliklar(X[te_]))), 3)
print("test AUC:", cevap)
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** Ağ sentetik yamalarda 0,965 test AUC değerine ulaşır. Görev çözülmüş gibi görünür; ama yalnızca sanal kliniğin çizdiği görüntüler için.

### Adım 2.2: Aynı ağ gerçek anjiyogramlarda

**Amaç.** Bu adımda yalnızca sentetik anjiyogramlardan öğrenen bir ağ, ARCADE'in gerçek anjiyogramlarında sınanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Sentetik anjiyogramlarda darlık bulmayı öğrenen bir ağın hiç görmediği gerçek X ışınlı anjiyogramlarda nasıl çalıştığını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; iki kaynaktan gelen test yamalarını tutar: Sanal kliniğin sentetik koroner anjiyogramlarından 1.700, açık bir veri seti olan ARCADE'in gerçek X ışınlı anjiyogramlarından 3.088 yama; yamaların her biri 64 × 64 pikseldir. Her satır bir yamadır: "kaynak" sütunu yamanın kaynağını ("sentetik" ya da "ARCADE") verir; "darlık" sütunu, merkezi bir darlıkta olan yama için 1, her darlıktan uzakta bir damarın üzerindeki yama için 0 değerini alır. "olasılık" sütunu, dersin yalnızca sentetik eğitim yamalarından öğrenen küçük evrişimli ağının verdiği darlık olasılığını tutar.  
> Görev: Her kaynak için ayrı ayrı "olasılık" sütununun "darlık" sütununa göre ROC eğrisinin altındaki alanını (AUC) ve farkı (sentetik eksi ARCADE) hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki AUC değerini ve farklarını yazdır; "ARCADE" kaynağının AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her AUC yalnızca kendi kaynağının yamalarından hesaplanmalı.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_m, _std, _pred = _course_model("synthetic_cnn")   # ilk seferde birkaç dakika sürer: Dersin ağı sentetik yamaları bir kez öğrenir
_s = _synthetic_table()
_s = _s[_s["küme"] == "test"]
_a = _arcade_table()
_a = _a[_a["küme"] == "test"]
veri = pd.DataFrame({"kaynak": ["sentetik"] * len(_s) + ["ARCADE"] * len(_a),
                     "darlık": np.concatenate([_s["darlık"], _a["darlık"]]),
                     "olasılık": np.concatenate([_pred(_m, _std(np.stack(_s["yama"]))), _pred(_m, _std(np.stack(_a["yama"])))])})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_m, _std, _pred = _course_model("synthetic_cnn")   # ilk seferde birkaç dakika sürer: Dersin ağı sentetik yamaları bir kez öğrenir
_s = _synthetic_table()
_s = _s[_s["küme"] == "test"]
_a = _arcade_table()
_a = _a[_a["küme"] == "test"]
veri = pd.DataFrame({"kaynak": ["sentetik"] * len(_s) + ["ARCADE"] * len(_a),
                     "darlık": np.concatenate([_s["darlık"], _a["darlık"]]),
                     "olasılık": np.concatenate([_pred(_m, _std(np.stack(_s["yama"]))), _pred(_m, _std(np.stack(_a["yama"])))])})
auclar = {}
for kaynak, parca in veri.groupby("kaynak"):
    auclar[kaynak] = roc_auc_score(parca["darlık"], parca["olasılık"])
print({k: round(v, 3) for k, v in auclar.items()}, "| fark", round(auclar["sentetik"] - auclar["ARCADE"], 3))
cevap = round(float(auclar["ARCADE"]), 3)
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** ARCADE'de AUC 0,538 düzeyine düşer. Gerçek anjiyogramların dokuları, gürültüsü ve üst üste binen yapıları farklıdır; ayrıca her dereceden darlık içerirler. Ağ darlığı değil, sanal kliniğin dünyasını öğrenmiştir.

### Adım 2.3: Gerçek anjiyogramlarla eğitim

**Amaç.** Bu adımda aynı küçük ağ, ARCADE'in eğitim anjiyogramlarının yamalarıyla eğitilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Küçük bir evrişimli ağın gerçek anjiyogramlardan öğrendiğinde gerçek koroner anjiyogramlarda darlıkları ne kadar iyi bulduğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan ARCADE'in darlık kümesindeki 1.500 X ışınlı koroner anjiyogramdan (eğitim 1.000 anjiyogram, doğrulama 200, test 300) kesilmiş 64 × 64 piksellik yamaları tutar. Her satır bir yamadır: "küme" sütunu yamanın kümesini ("eğitim", "doğrulama" ya da "test"), "anjiyogram" anjiyogramının numarasını verir; "yama" sütununun her hücresi yamayı gri değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü olarak tutar. "darlık" sütunu, yamanın merkezi uzmanların sınırını çizdiği bir darlığın merkezine 6 pikselden yakınsa 1, yama sınırı çizilmiş her darlıktan ve görüntünün kenarından en az 48 piksel uzakta bir damarın üzerindeyse 0 değerini alır.  
> Görev: Her yamayı kendi içinde standartlaştır: Gri değerlerinin ortalamasını çıkar ve standart sapmalarının 0,02 fazlasına böl. Tek kanallı, 64 × 64 piksellik bir yama için küçük bir evrişimli ağ kur: Her biri 3 × 3 piksellik filtreler ve bir piksellik dolgu içeren bir evrişim (16, 32, 64 ve 64 filtre), batch normalizasyonu, ReLU ve 2 × 2 en büyük havuzlamadan oluşan dört blok; ardından her kanalın konumları üzerinden ortalaması ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman. Ağı eğitim yamalarıyla 8 epoch boyunca, karıştırılmış 64 yamalık gruplar hâlinde, logit üzerinde ikili çapraz entropi kaybıyla ve öğrenme hızı 0,001 olan Adam optimizasyon algoritmasıyla eğit. Veri çoğaltma için her grubu rastgele bir çeyrek tur döndür ve yarı olasılıkla soldan sağa aynala. Her epoch'un sonunda doğrulama yamalarının olasılıklarının AUC değerini hesapla; eğitimin sonunda doğrulama AUC değeri en büyük olan epoch'un ağırlıklarını sakla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Ardından test yamalarının olasılıklarını ve bunların "darlık" sütununa göre AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitilebilir ağırlık sayısını ve her epoch'un doğrulama AUC değerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Ağın 60.641 eğitilebilir ağırlığı olmalı ve hiçbir test anjiyogramının yaması eğitimde kullanılmamalı.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _arcade_table()   # ilk seferde birkaç dakika sürer: ARCADE bir kez indirilir ve yamaları kesilir
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _arcade_table()   # ilk seferde birkaç dakika sürer: ARCADE bir kez indirilir ve yamaları kesilir


def standartlastir(yamalar):
    yamalar = np.asarray(yamalar, dtype=np.float32)
    o = yamalar.mean(axis=(1, 2), keepdims=True)
    s = yamalar.std(axis=(1, 2), keepdims=True)
    return (yamalar - o) / (s + 0.02)


def blok(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class Ag(nn.Module):
    def __init__(self):
        super().__init__()
        self.oznitelik = nn.Sequential(blok(1, 16), blok(16, 32), blok(32, 64), blok(64, 64))
        self.cikis = nn.Linear(64, 1)

    def forward(self, x):
        return self.cikis(self.oznitelik(x).mean(dim=(2, 3))).squeeze(1)

torch.manual_seed(2026)
ag = Ag()
print("eğitilebilir ağırlık:", sum(p.numel() for p in ag.parameters() if p.requires_grad))
kumeler = veri["küme"].to_numpy()
X = torch.as_tensor(standartlastir(np.stack(veri["yama"])))[:, None]
y = torch.as_tensor(veri["darlık"].to_numpy(), dtype=torch.float32)
eg_, do_, te_ = [torch.as_tensor(kumeler == s) for s in ("eğitim", "doğrulama", "test")]


@torch.no_grad()
def olasiliklar(X_):
    ag.eval()
    return torch.cat([torch.sigmoid(ag(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xeg, yeg = X[eg_], y[eg_]
torch.manual_seed(2026)
optimizer = torch.optim.Adam(ag.parameters(), lr=0.001)
kayip = nn.BCEWithLogitsLoss()
en_iyi, durum = -1.0, None
for epoch in range(8):
    ag.train()
    sira = torch.randperm(len(Xeg))
    for i in range(0, len(Xeg), 64):
        g = sira[i:i + 64]
        xg = torch.rot90(Xeg[g], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xg = torch.flip(xg, dims=(3,))
        k = kayip(ag(xg), yeg[g])
        optimizer.zero_grad()
        k.backward()
        optimizer.step()
    auc = roc_auc_score(y[do_].numpy(), olasiliklar(X[do_]))
    print(f"epoch {epoch + 1}: doğrulama AUC {auc:.3f}")
    if auc > en_iyi:
        en_iyi, durum = auc, copy.deepcopy(ag.state_dict())
ag.load_state_dict(durum)
cevap = round(float(roc_auc_score(y[te_].numpy(), olasiliklar(X[te_]))), 3)
print("test AUC:", cevap)
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** ARCADE ile eğitilen ağ 0,81 test AUC değerine ulaşır; sentetik anjiyogramlardan öğrendiğinde bu değer 0,538 düzeyindeydi. Bir model, öğrendiği görüntüler kadar iyidir.

**2. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#a-convolutional-network-a-vision-transformer-and-a-hybrid-compared) ve okumaya «Evrişimli ağ, Vision Transformer ve hibrit model karşılaştırması» bölümünden devam edin. Bir sonraki uygulama durağında ders sayfası sizi yine bu Jupyter Notebook'a getirir.

## Durak 3: Evrişimli ağ, Vision Transformer ve hibrit model

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 7-9. bölümler*

Küçük bir Vision Transformer ve evrişimle dikkati birleştiren hibrit bir model, ARCADE'in yamalarını 2. duraktaki evrişimli ağla aynı biçimde öğrenir; böylece üç model aynı test anjiyogramlarında karşılaştırılabilir. Son adım Vision Transformer'ın nereye baktığını sorar: Sınıf token'ının dikkatinin ne kadarı, darlığın bulunduğu yama merkezine düşmektedir?

**Durağın sonunda:** Küçük Vision Transformer'ın (0,93) ve hibrit modelin (0,934) test AUC değerlerini ve sınıf token'ının dikkatinin, darlık içeren yamalarda ortadaki 16 × 16 piksele düşen ortalama payını (0,153) bulacaksınız.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "train": "eğitim", "val": "doğrulama"}


def _synthetic_table():
    """Sanal kliniğin 300 eğitim, 100 doğrulama ve 100 test hastasının sentetik anjiyogramlarından yamalar."""
    if "_synthetic_cache" not in globals():
        parts = []
        for s, n in (("training", 300), ("validation", 100), ("test", 100)):
            X, y, ids = angiogram.patches(s, n_patients=n)
            parts.append(pd.DataFrame({"küme": _SETS[s], "hasta": ids, "yama": list(X), "darlık": y}))
        globals()["_synthetic_cache"] = pd.concat(parts, ignore_index=True)
    return _synthetic_cache


def _arcade_table():
    """ARCADE'in (darlık kümesi) 1500 anjiyogramından yamalar; kümeleri ve anjiyogramlarıyla birlikte."""
    if "_arcade_cache" not in globals():
        arcade.download()
        parts = []
        for s in arcade.SPLITS:
            X, y, ids = arcade.patches(s)
            parts.append(pd.DataFrame({"küme": _SETS[s], "anjiyogram": ids, "yama": list(X), "darlık": y}))
        globals()["_arcade_cache"] = pd.concat(parts, ignore_index=True)
    return _arcade_cache


def _course_model(kind):
    """Dersin bir ağı; eğitim yamalarıyla bir kez eğitilir ve sonraki adımlar için saklanır (bir ile dört dakika sürer)."""
    key = "_course_" + kind
    if key in globals():
        return globals()[key]
    def standardise(X):
        """Her yama kendi içinde: Ortalama gri değer 0, standart sapma 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """3 x 3 evrişim, batch normalizasyonu, ReLU ve 2 x 2 en büyük havuzlama: Boyut yarıya iner, b kanal elde edilir."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Dört evrişim bloğu (16, 32, 64 ve 64 kanal), her kanalın ortalaması ve tek bir çıktı."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Çok başlı öz dikkat; son çağrının ağırlıkları haritalar için saklanır."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """Bir Transformer bloğu: Dikkat ve iki katmanlı küçük bir ağ; her biri katman normalleştirmesinden sonra gelir ve bir artık bağlantısı taşır."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """Küçük bir Vision Transformer: Token olarak 8 x 8 piksellik kareler, bir sınıf token'ı, öğrenilen konumlar ve dört blok."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # her kare için aynı doğrusal dönüşüm
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """Bir ağın eğitilebilir ağırlıklarının sayısı."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """Hibrit model: Üç evrişim bloğu yamayı 64 sayılık 8 x 8 konuma dönüştürür; bunlar iki Transformer bloğunun token'larıdır."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """Her yama için darlık olasılığı."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam, 0,001 öğrenme hızı, rastgele bir çeyrek tur döndürülen ve yarı yarıya aynalanan 64 yamalık gruplar; doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best
    t = _synthetic_table() if kind == "synthetic_cnn" else _arcade_table()
    sets = t["küme"].to_numpy()
    X, y = standardise(np.stack(t["yama"])), t["darlık"].to_numpy()
    tr_, va_ = sets == "eğitim", sets == "doğrulama"
    torch.manual_seed(2026)
    model = SmallCNN() if kind == "synthetic_cnn" else SmallViT()
    train(model, X[tr_], y[tr_], X[va_], y[va_], epochs=8, verbose=False)
    globals()[key] = (model, standardise, predict)
    return globals()[key]

_arcade_table()
print(len(_arcade_cache), "ARCADE yaması")
print("3. durak hazır.")

### Adım 3.1: Küçük bir Vision Transformer

**Amaç.** Bu adımda her yamayı 64 kareye bölen ve bu karelerin birbirine dikkat etmesini sağlayan küçük bir Vision Transformer ARCADE'in yamalarıyla eğitilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Gerçek koroner anjiyogramlarda darlık bulmak için aynı biçimde eğitilmiş küçük bir Vision Transformer'ı küçük bir evrişimli ağla karşılaştırmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan ARCADE'in darlık kümesindeki 1.500 X ışınlı koroner anjiyogramdan (eğitim 1.000 anjiyogram, doğrulama 200, test 300) kesilmiş 64 × 64 piksellik yamaları tutar. Her satır bir yamadır: "küme" sütunu yamanın kümesini ("eğitim", "doğrulama" ya da "test"), "anjiyogram" anjiyogramının numarasını verir; "yama" sütununun her hücresi yamayı gri değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü olarak tutar. "darlık" sütunu, yamanın merkezi uzmanların sınırını çizdiği bir darlığın merkezine 6 pikselden yakınsa 1, yama sınırı çizilmiş her darlıktan ve görüntünün kenarından en az 48 piksel uzakta bir damarın üzerindeyse 0 değerini alır.  
> Görev: Her yamayı kendi içinde standartlaştır: Gri değerlerinin ortalamasını çıkar ve standart sapmalarının 0,02 fazlasına böl. Tek kanallı, 64 × 64 piksellik bir yama için küçük bir Vision Transformer kur. Yamayı 8 × 8 piksellik 64 kareye böl ve her kareyi öğrenilen aynı doğrusal dönüşümle 64 sayıya dönüştür. 64 token'ın önüne 64 sayıdan oluşan öğrenilen bir sınıf token'ı koy ve öğrenilen konumları ekle: 65 yerin her biri için, standart sapması 0,02 olan küçük rastgele sayılarla başlayan 64 sayılık bir vektör. Token'ları dört Transformer bloğundan geçir. Her blokta katman normalleştirmesini dört başlı öz dikkat izler ve öz dikkatin çıktısı bloğun girdisine eklenir; ikinci bir katman normalleştirmesini 128 birimlik bir gizli katman ve GELU etkinleştirmesi içeren ileri beslemeli bir ağ izler ve bu ağın çıktısı da yeniden eklenir. Sonunda sınıf token'ını al, katman normalleştirmesi ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman uygula. Ağı eğitim yamalarıyla 8 epoch boyunca, karıştırılmış 64 yamalık gruplar hâlinde, logit üzerinde ikili çapraz entropi kaybıyla ve öğrenme hızı 0,001 olan Adam optimizasyon algoritmasıyla eğit. Veri çoğaltma için her grubu rastgele bir çeyrek tur döndür ve yarı olasılıkla soldan sağa aynala. Her epoch'un sonunda doğrulama yamalarının olasılıklarının AUC değerini hesapla; eğitimin sonunda doğrulama AUC değeri en büyük olan epoch'un ağırlıklarını sakla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Ardından test yamalarının olasılıklarını ve bunların "darlık" sütununa göre AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitilebilir ağırlık sayısını ve her epoch'un doğrulama AUC değerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Ağın 142.465 eğitilebilir ağırlığı olmalı ve hiçbir test anjiyogramının yaması eğitimde kullanılmamalı.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _arcade_table()   # ilk seferde birkaç dakika sürer: ARCADE bir kez indirilir ve yamaları kesilir
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
teach.check_step("Adım 3.1", "cevap", globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _arcade_table()   # ilk seferde birkaç dakika sürer: ARCADE bir kez indirilir ve yamaları kesilir


def standartlastir(yamalar):
    yamalar = np.asarray(yamalar, dtype=np.float32)
    o = yamalar.mean(axis=(1, 2), keepdims=True)
    s = yamalar.std(axis=(1, 2), keepdims=True)
    return (yamalar - o) / (s + 0.02)


class Dikkat(nn.Module):
    def __init__(self, boyut, bas):
        super().__init__()
        self.bas = bas
        self.qkv = nn.Linear(boyut, 3 * boyut)
        self.cikis = nn.Linear(boyut, boyut)

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.bas, d // self.bas).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.bas) ** 0.5, dim=-1)
        return self.cikis((a @ v).transpose(1, 2).reshape(n, t, d))


class Blok(nn.Module):
    def __init__(self, boyut=64, bas=4, gizli=128):
        super().__init__()
        self.norm1, self.dikkat = nn.LayerNorm(boyut), Dikkat(boyut, bas)
        self.norm2 = nn.LayerNorm(boyut)
        self.mlp = nn.Sequential(nn.Linear(boyut, gizli), nn.GELU(), nn.Linear(gizli, boyut))

    def forward(self, x):
        x = x + self.dikkat(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class Transformer(nn.Module):
    def __init__(self, token_sayisi, derinlik, boyut=64):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, boyut))
        self.konum = nn.Parameter(torch.randn(1, token_sayisi + 1, boyut) * 0.02)
        self.bloklar = nn.Sequential(*[Blok(boyut) for _ in range(derinlik)])
        self.norm = nn.LayerNorm(boyut)
        self.cikis = nn.Linear(boyut, 1)

    def forward(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.konum
        return self.cikis(self.norm(self.bloklar(t))[:, 0]).squeeze(1)



class Ag(nn.Module):
    def __init__(self):
        super().__init__()
        self.gomme = nn.Conv2d(1, 64, 8, stride=8)
        self.transformer = Transformer(64, 4)

    def forward(self, x):
        return self.transformer(self.gomme(x).flatten(2).transpose(1, 2))

torch.manual_seed(2026)
ag = Ag()
print("eğitilebilir ağırlık:", sum(p.numel() for p in ag.parameters() if p.requires_grad))
kumeler = veri["küme"].to_numpy()
X = torch.as_tensor(standartlastir(np.stack(veri["yama"])))[:, None]
y = torch.as_tensor(veri["darlık"].to_numpy(), dtype=torch.float32)
eg_, do_, te_ = [torch.as_tensor(kumeler == s) for s in ("eğitim", "doğrulama", "test")]


@torch.no_grad()
def olasiliklar(X_):
    ag.eval()
    return torch.cat([torch.sigmoid(ag(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xeg, yeg = X[eg_], y[eg_]
torch.manual_seed(2026)
optimizer = torch.optim.Adam(ag.parameters(), lr=0.001)
kayip = nn.BCEWithLogitsLoss()
en_iyi, durum = -1.0, None
for epoch in range(8):
    ag.train()
    sira = torch.randperm(len(Xeg))
    for i in range(0, len(Xeg), 64):
        g = sira[i:i + 64]
        xg = torch.rot90(Xeg[g], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xg = torch.flip(xg, dims=(3,))
        k = kayip(ag(xg), yeg[g])
        optimizer.zero_grad()
        k.backward()
        optimizer.step()
    auc = roc_auc_score(y[do_].numpy(), olasiliklar(X[do_]))
    print(f"epoch {epoch + 1}: doğrulama AUC {auc:.3f}")
    if auc > en_iyi:
        en_iyi, durum = auc, copy.deepcopy(ag.state_dict())
ag.load_state_dict(durum)
cevap = round(float(roc_auc_score(y[te_].numpy(), olasiliklar(X[te_]))), 3)
print("test AUC:", cevap)
teach.check_step("Adım 3.1", "cevap", globals())

**Sonuç ne gösteriyor?** Küçük Vision Transformer 0,93 test AUC değerine ulaşır; evrişimli ağın değeri ise 0,81 düzeyindedir. Yaklaşık 13.000 eğitim yamasıyla dikkat, evrişimlere tasarımlarının verdiği bilgiyi öğrenebilmektedir.

### Adım 3.2: Evrişim ve dikkatten oluşan hibrit bir model

**Amaç.** Bu adımda hibrit bir model eğitilir: Önce üç evrişim bloğu, ardından bunların çıktısındaki konumlar üzerinde iki Transformer bloğu.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Önce evrişimlerle küçük komşuluklara bakan, ardından yamanın parçalarını dikkatle birbirine bağlayan bir ağın gerçek anjiyogramlarda darlıkları iki türün her birinden daha iyi bulup bulmadığını öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan ARCADE'in darlık kümesindeki 1.500 X ışınlı koroner anjiyogramdan (eğitim 1.000 anjiyogram, doğrulama 200, test 300) kesilmiş 64 × 64 piksellik yamaları tutar. Her satır bir yamadır: "küme" sütunu yamanın kümesini ("eğitim", "doğrulama" ya da "test"), "anjiyogram" anjiyogramının numarasını verir; "yama" sütununun her hücresi yamayı gri değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü olarak tutar. "darlık" sütunu, yamanın merkezi uzmanların sınırını çizdiği bir darlığın merkezine 6 pikselden yakınsa 1, yama sınırı çizilmiş her darlıktan ve görüntünün kenarından en az 48 piksel uzakta bir damarın üzerindeyse 0 değerini alır.  
> Görev: Her yamayı kendi içinde standartlaştır: Gri değerlerinin ortalamasını çıkar ve standart sapmalarının 0,02 fazlasına böl. Tek kanallı, 64 × 64 piksellik bir yama için hibrit bir ağ kur. Her biri 3 × 3 piksellik filtreler ve bir piksellik dolgu içeren bir evrişim (16, 32 ve 64 filtre), batch normalizasyonu, ReLU ve 2 × 2 en büyük havuzlamadan oluşan üç evrişim bloğu, yamayı 64 sayılık 8 × 8 konuma dönüştürür. Bu 64 konum küçük bir Transformer modelinin token'larıdır: Önlerine 64 sayıdan oluşan öğrenilen bir sınıf token'ı koy, 65 yer için standart sapması 0,02 olan küçük rastgele sayılarla başlayan öğrenilen konumları ekle ve token'ları iki Transformer bloğundan geçir. Her blokta katman normalleştirmesini dört başlı öz dikkat izler ve öz dikkatin çıktısı bloğun girdisine eklenir; ikinci bir katman normalleştirmesini 128 birimlik bir gizli katman ve GELU etkinleştirmesi içeren ileri beslemeli bir ağ izler ve bu ağın çıktısı da yeniden eklenir. Sonunda sınıf token'ını al, katman normalleştirmesi ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman uygula. Ağı eğitim yamalarıyla 8 epoch boyunca, karıştırılmış 64 yamalık gruplar hâlinde, logit üzerinde ikili çapraz entropi kaybıyla ve öğrenme hızı 0,001 olan Adam optimizasyon algoritmasıyla eğit. Veri çoğaltma için her grubu rastgele bir çeyrek tur döndür ve yarı olasılıkla soldan sağa aynala. Her epoch'un sonunda doğrulama yamalarının olasılıklarının AUC değerini hesapla; eğitimin sonunda doğrulama AUC değeri en büyük olan epoch'un ağırlıklarını sakla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Ardından test yamalarının olasılıklarını ve bunların "darlık" sütununa göre AUC değerini hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitilebilir ağırlık sayısını ve her epoch'un doğrulama AUC değerini yazdır; test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Ağın 94.881 eğitilebilir ağırlığı olmalı ve hiçbir test anjiyogramının yaması eğitimde kullanılmamalı.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _arcade_table()   # ilk seferde birkaç dakika sürer: ARCADE bir kez indirilir ve yamaları kesilir
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
teach.check_step("Adım 3.2", "cevap", globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = _arcade_table()   # ilk seferde birkaç dakika sürer: ARCADE bir kez indirilir ve yamaları kesilir


def standartlastir(yamalar):
    yamalar = np.asarray(yamalar, dtype=np.float32)
    o = yamalar.mean(axis=(1, 2), keepdims=True)
    s = yamalar.std(axis=(1, 2), keepdims=True)
    return (yamalar - o) / (s + 0.02)


class Dikkat(nn.Module):
    def __init__(self, boyut, bas):
        super().__init__()
        self.bas = bas
        self.qkv = nn.Linear(boyut, 3 * boyut)
        self.cikis = nn.Linear(boyut, boyut)

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.bas, d // self.bas).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.bas) ** 0.5, dim=-1)
        return self.cikis((a @ v).transpose(1, 2).reshape(n, t, d))


class Blok(nn.Module):
    def __init__(self, boyut=64, bas=4, gizli=128):
        super().__init__()
        self.norm1, self.dikkat = nn.LayerNorm(boyut), Dikkat(boyut, bas)
        self.norm2 = nn.LayerNorm(boyut)
        self.mlp = nn.Sequential(nn.Linear(boyut, gizli), nn.GELU(), nn.Linear(gizli, boyut))

    def forward(self, x):
        x = x + self.dikkat(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class Transformer(nn.Module):
    def __init__(self, token_sayisi, derinlik, boyut=64):
        super().__init__()
        self.cls = nn.Parameter(torch.zeros(1, 1, boyut))
        self.konum = nn.Parameter(torch.randn(1, token_sayisi + 1, boyut) * 0.02)
        self.bloklar = nn.Sequential(*[Blok(boyut) for _ in range(derinlik)])
        self.norm = nn.LayerNorm(boyut)
        self.cikis = nn.Linear(boyut, 1)

    def forward(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.konum
        return self.cikis(self.norm(self.bloklar(t))[:, 0]).squeeze(1)



def blok(a, b):
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class Ag(nn.Module):
    def __init__(self):
        super().__init__()
        self.govde = nn.Sequential(blok(1, 16), blok(16, 32), blok(32, 64))
        self.transformer = Transformer(64, 2)

    def forward(self, x):
        return self.transformer(self.govde(x).flatten(2).transpose(1, 2))

torch.manual_seed(2026)
ag = Ag()
print("eğitilebilir ağırlık:", sum(p.numel() for p in ag.parameters() if p.requires_grad))
kumeler = veri["küme"].to_numpy()
X = torch.as_tensor(standartlastir(np.stack(veri["yama"])))[:, None]
y = torch.as_tensor(veri["darlık"].to_numpy(), dtype=torch.float32)
eg_, do_, te_ = [torch.as_tensor(kumeler == s) for s in ("eğitim", "doğrulama", "test")]


@torch.no_grad()
def olasiliklar(X_):
    ag.eval()
    return torch.cat([torch.sigmoid(ag(X_[i:i + 256])) for i in range(0, len(X_), 256)]).numpy()


Xeg, yeg = X[eg_], y[eg_]
torch.manual_seed(2026)
optimizer = torch.optim.Adam(ag.parameters(), lr=0.001)
kayip = nn.BCEWithLogitsLoss()
en_iyi, durum = -1.0, None
for epoch in range(8):
    ag.train()
    sira = torch.randperm(len(Xeg))
    for i in range(0, len(Xeg), 64):
        g = sira[i:i + 64]
        xg = torch.rot90(Xeg[g], int(torch.randint(0, 4, (1,))), dims=(2, 3))
        if torch.rand(1) < 0.5:
            xg = torch.flip(xg, dims=(3,))
        k = kayip(ag(xg), yeg[g])
        optimizer.zero_grad()
        k.backward()
        optimizer.step()
    auc = roc_auc_score(y[do_].numpy(), olasiliklar(X[do_]))
    print(f"epoch {epoch + 1}: doğrulama AUC {auc:.3f}")
    if auc > en_iyi:
        en_iyi, durum = auc, copy.deepcopy(ag.state_dict())
ag.load_state_dict(durum)
cevap = round(float(roc_auc_score(y[te_].numpy(), olasiliklar(X[te_]))), 3)
print("test AUC:", cevap)
teach.check_step("Adım 3.2", "cevap", globals())

**Sonuç ne gösteriyor?** Hibrit model, Vision Transformer'dan daha az ağırlıkla 0,934 test AUC değerine ulaşır ve iki ağ da evrişimli ağın önüne geçer. Hibrit model ile Vision Transformer arasındaki küçük farkın gerçek olup olmadığı, Python yolunun 7. bölümündeki gibi test anjiyogramlarından çekilen bootstrap örnekleriyle sınanır.

### Adım 3.3: Vision Transformer nereye bakıyor?

**Amaç.** Bu adımda Vision Transformer'ın sınıf token'ının dikkatinin ne kadarının, darlığın bulunduğu yama merkezine düştüğü ölçülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Anjiyogramlarda darlık bulan bir Vision Transformer'ın dikkat haritalarını bir açıklama olarak okumadan önce, dikkatinin darlığın bulunduğu yere düşüp düşmediğini öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir X ışınlı koroner anjiyogram veri seti olan ARCADE'in 64 × 64 piksellik 3.088 test yamasını tutar. Her satır bir yamadır: "anjiyogram" sütunu yamanın anjiyogramının numarasını verir; "darlık" sütunu, merkezi uzmanların sınırını çizdiği bir darlıkta olan yama için 1, her darlıktan uzakta bir damarın üzerindeki yama için 0 değerini alır. "dikkat" sütununun her hücresi toplamı 1 olan 8 × 8 sayılık bir tablo tutar: Dersin küçük Vision Transformer'ın sınıf token'ının, yamanın 8 × 8 piksellik 64 karesinin her birine gösterdiği ve dört katmanı boyunca izlenen dikkat (dikkat aktarımı). Ortadaki dört kare, yamanın ortadaki 16 × 16 pikselini kaplar; 1'den başlayarak sayıldığında bunlar 4. ve 5. satır ve sütunlardır.  
> Görev: Her yama için ortadaki dört karenin dikkatini topla. Bu payın darlık içeren yamalar ve öteki yamalar üzerindeki ortalamasını hesapla ve iki ortalamayı, dikkat eşit dağılsaydı dört karenin alacağı pay olan 4/64 = 0,0625 ile karşılaştır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki ortalamayı yazdır; darlık içeren yamaların ortalamasını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Her pay 0 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_m, _std, _pred = _course_model("vit")   # ilk seferde yaklaşık dört dakika sürer: Dersin küçük Vision Transformer bir kez eğitilir
_a = _arcade_table()
_a = _a[_a["küme"] == "test"].reset_index(drop=True)
_X = _std(np.stack(_a["yama"]))
_m.eval()
_maps = []
with torch.no_grad():
    for _i in range(0, len(_X), 256):
        _m(torch.as_tensor(_X[_i:_i + 256])[:, None])
        for _w in np.stack([_b.attn.weights.mean(1).numpy() for _b in _m.blocks], axis=1):
            _r = np.eye(65)
            for _l in _w:
                _l = 0.5 * _l + 0.5 * np.eye(65)
                _r = (_l / _l.sum(axis=1, keepdims=True)) @ _r
            _maps.append((_r[0, 1:] / _r[0, 1:].sum()).reshape(8, 8))
veri = pd.DataFrame({"anjiyogram": _a["anjiyogram"], "darlık": _a["darlık"], "dikkat": _maps})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
teach.check_step("Adım 3.3", "cevap", globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_m, _std, _pred = _course_model("vit")   # ilk seferde yaklaşık dört dakika sürer: Dersin küçük Vision Transformer bir kez eğitilir
_a = _arcade_table()
_a = _a[_a["küme"] == "test"].reset_index(drop=True)
_X = _std(np.stack(_a["yama"]))
_m.eval()
_maps = []
with torch.no_grad():
    for _i in range(0, len(_X), 256):
        _m(torch.as_tensor(_X[_i:_i + 256])[:, None])
        for _w in np.stack([_b.attn.weights.mean(1).numpy() for _b in _m.blocks], axis=1):
            _r = np.eye(65)
            for _l in _w:
                _l = 0.5 * _l + 0.5 * np.eye(65)
                _r = (_l / _l.sum(axis=1, keepdims=True)) @ _r
            _maps.append((_r[0, 1:] / _r[0, 1:].sum()).reshape(8, 8))
veri = pd.DataFrame({"anjiyogram": _a["anjiyogram"], "darlık": _a["darlık"], "dikkat": _maps})
paylar = np.array([h[3:5, 3:5].sum() for h in veri["dikkat"]])
y = veri["darlık"].to_numpy()
print("darlık içeren", round(paylar[y == 1].mean(), 3), "| öteki", round(paylar[y == 0].mean(), 3), "| eşit", 4 / 64)
cevap = round(float(paylar[y == 1].mean()), 3)
teach.check_step("Adım 3.3", "cevap", globals())

**Sonuç ne gösteriyor?** Darlık içeren yamalarda sınıf token'ı dikkatinin 0,153 kadarını, yamanın on altıda birini kaplayan ortadaki dört kareye gönderir. Harita merkezi eşit bir dağılımdan daha çok gösterir; ancak bilginin nereden aktığını gösterir, ağın neden o kararı verdiğini göstermez.

**3. durak tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#ai-pair-programming-part-7-reviewing-an-image-pipeline) ve okumaya «YZ asistanıyla çalışmak, 7. bölüm: Bir görüntü işlem hattını denetlemek» bölümünden devam edin. Haftalık proje prompt yoluyla da yapılabilir: Promptları aşağıdaki «Prompt yolunda haftalık proje» bölümündedir.

## Prompt yolunda haftalık proje

*Haftanın projesi: [7. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-07/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
_SETS = {"training": "eğitim", "validation": "doğrulama", "test": "test", "train": "eğitim", "val": "doğrulama"}


def _synthetic_table():
    """Sanal kliniğin 300 eğitim, 100 doğrulama ve 100 test hastasının sentetik anjiyogramlarından yamalar."""
    if "_synthetic_cache" not in globals():
        parts = []
        for s, n in (("training", 300), ("validation", 100), ("test", 100)):
            X, y, ids = angiogram.patches(s, n_patients=n)
            parts.append(pd.DataFrame({"küme": _SETS[s], "hasta": ids, "yama": list(X), "darlık": y}))
        globals()["_synthetic_cache"] = pd.concat(parts, ignore_index=True)
    return _synthetic_cache


def _arcade_table():
    """ARCADE'in (darlık kümesi) 1500 anjiyogramından yamalar; kümeleri ve anjiyogramlarıyla birlikte."""
    if "_arcade_cache" not in globals():
        arcade.download()
        parts = []
        for s in arcade.SPLITS:
            X, y, ids = arcade.patches(s)
            parts.append(pd.DataFrame({"küme": _SETS[s], "anjiyogram": ids, "yama": list(X), "darlık": y}))
        globals()["_arcade_cache"] = pd.concat(parts, ignore_index=True)
    return _arcade_cache


def _course_model(kind):
    """Dersin bir ağı; eğitim yamalarıyla bir kez eğitilir ve sonraki adımlar için saklanır (bir ile dört dakika sürer)."""
    key = "_course_" + kind
    if key in globals():
        return globals()[key]
    def standardise(X):
        """Her yama kendi içinde: Ortalama gri değer 0, standart sapma 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """3 x 3 evrişim, batch normalizasyonu, ReLU ve 2 x 2 en büyük havuzlama: Boyut yarıya iner, b kanal elde edilir."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Dört evrişim bloğu (16, 32, 64 ve 64 kanal), her kanalın ortalaması ve tek bir çıktı."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Çok başlı öz dikkat; son çağrının ağırlıkları haritalar için saklanır."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """Bir Transformer bloğu: Dikkat ve iki katmanlı küçük bir ağ; her biri katman normalleştirmesinden sonra gelir ve bir artık bağlantısı taşır."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """Küçük bir Vision Transformer: Token olarak 8 x 8 piksellik kareler, bir sınıf token'ı, öğrenilen konumlar ve dört blok."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # her kare için aynı doğrusal dönüşüm
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """Bir ağın eğitilebilir ağırlıklarının sayısı."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """Hibrit model: Üç evrişim bloğu yamayı 64 sayılık 8 x 8 konuma dönüştürür; bunlar iki Transformer bloğunun token'larıdır."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """Her yama için darlık olasılığı."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam, 0,001 öğrenme hızı, rastgele bir çeyrek tur döndürülen ve yarı yarıya aynalanan 64 yamalık gruplar; doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best
    t = _synthetic_table() if kind == "synthetic_cnn" else _arcade_table()
    sets = t["küme"].to_numpy()
    X, y = standardise(np.stack(t["yama"])), t["darlık"].to_numpy()
    tr_, va_ = sets == "eğitim", sets == "doğrulama"
    torch.manual_seed(2026)
    model = {"synthetic_cnn": SmallCNN, "cnn": SmallCNN, "vit": SmallViT, "hybrid": Hybrid}[kind]()
    train(model, X[tr_], y[tr_], X[va_], y[va_], epochs=8, verbose=False)
    globals()[key] = (model, standardise, predict)
    return globals()[key]



def _probability_table():
    """ARCADE'in doğrulama ve test yamaları, dersin üç ağının olasılıklarıyla birlikte (ilk seferde yaklaşık yedi dakika sürer)."""
    t = _arcade_table()
    t = t[t["küme"] != "eğitim"].reset_index(drop=True)
    out = t[["küme", "anjiyogram", "darlık"]].copy()
    for kind, name in (("cnn", "evrişimli ağ"), ("vit", "Vision Transformer"), ("hybrid", "hibrit model")):
        m, std, pred = _course_model(kind)
        out[name] = pred(m, std(np.stack(t["yama"])))
    return out


def _angiogram_table():
    """ARCADE'in 1.500 anjiyogramı, sınırı çizilmiş darlıklarıyla birlikte, 0 ile 255 arasında gri değerler olarak."""
    if "_angio_cache" not in globals():
        arcade.download()
        rows = []
        for s in arcade.SPLITS:
            for a in arcade.annotations(s):
                rows.append({"küme": _SETS[s], "anjiyogram": a["id"], "görüntü": (arcade.load(s, a["file"]) * 255).round().astype(np.uint8),
                             "darlıklar": [(round(x["x"]), round(x["y"])) for x in a["stenoses"]]})
        globals()["_angio_cache"] = pd.DataFrame(rows)
    return _angio_cache


def _detector_table():
    """Test anjiyogramları; dersin tespit modelinin her 16 pikseldeki olasılıkları ve eşiğiyle birlikte (ilk seferde yaklaşık on beş dakika sürer)."""
    if "_detector_cache" in globals():
        return _detector_cache
    def standardise(X):
        """Her yama kendi içinde: Ortalama gri değer 0, standart sapma 1."""
        X = np.asarray(X, dtype=np.float32)
        m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
        return (X - m) / (s + 0.02)


    def conv_block(a, b):
        """3 x 3 evrişim, batch normalizasyonu, ReLU ve 2 x 2 en büyük havuzlama: Boyut yarıya iner, b kanal elde edilir."""
        return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


    class SmallCNN(nn.Module):
        """Dört evrişim bloğu (16, 32, 64 ve 64 kanal), her kanalın ortalaması ve tek bir çıktı."""
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
            self.head = nn.Linear(64, 1)

        def forward(self, x):
            return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



    class Attention(nn.Module):
        """Çok başlı öz dikkat; son çağrının ağırlıkları haritalar için saklanır."""
        def __init__(self, dim, heads):
            super().__init__()
            self.heads = heads
            self.qkv = nn.Linear(dim, 3 * dim)
            self.out = nn.Linear(dim, dim)
            self.weights = None

        def forward(self, x):
            n, t, d = x.shape
            q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
            a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
            self.weights = a.detach()
            return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


    class Block(nn.Module):
        """Bir Transformer bloğu: Dikkat ve iki katmanlı küçük bir ağ; her biri katman normalleştirmesinden sonra gelir ve bir artık bağlantısı taşır."""
        def __init__(self, dim, heads, hidden):
            super().__init__()
            self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
            self.norm2 = nn.LayerNorm(dim)
            self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

        def forward(self, x):
            x = x + self.attn(self.norm1(x))
            return x + self.mlp(self.norm2(x))


    class SmallViT(nn.Module):
        """Küçük bir Vision Transformer: Token olarak 8 x 8 piksellik kareler, bir sınıf token'ı, öğrenilen konumlar ve dört blok."""
        def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
            super().__init__()
            self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # her kare için aynı doğrusal dönüşüm
            n = (size // patch) ** 2
            self.cls = nn.Parameter(torch.zeros(1, 1, dim))
            self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
            self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Linear(dim, outputs)

        def encode(self, t):
            t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
            for block in self.blocks:
                t = block(t)
            out = self.head(self.norm(t)[:, 0])
            return out.squeeze(1) if out.shape[1] == 1 else out

        def forward(self, x):
            return self.encode(self.embed(x).flatten(2).transpose(1, 2))


    def n_parameters(model):
        """Bir ağın eğitilebilir ağırlıklarının sayısı."""
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    class Hybrid(SmallViT):
        """Hibrit model: Üç evrişim bloğu yamayı 64 sayılık 8 x 8 konuma dönüştürür; bunlar iki Transformer bloğunun token'larıdır."""
        def __init__(self, dim=64, depth=2, heads=4):
            super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
            self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



    @torch.no_grad()
    def predict(model, X, batch=256):
        """Her yama için darlık olasılığı."""
        model.eval()
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


    def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
        """Adam, 0,001 öğrenme hızı, rastgele bir çeyrek tur döndürülen ve yarı yarıya aynalanan 64 yamalık gruplar; doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
        torch.manual_seed(seed)
        X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
        optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
        loss_fn = nn.BCEWithLogitsLoss()
        best, state = -1.0, None
        for epoch in range(epochs):
            model.train()
            order = torch.randperm(len(X))
            for i in range(0, len(X), 64):
                b = order[i:i + 64]
                xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=(3,))
                loss = loss_fn(model(xb), y[b])
                optimiser.zero_grad()
                loss.backward()
                optimiser.step()
            a = roc_auc_score(yv, predict(model, Xv))
            if a > best:
                best, state = a, copy.deepcopy(model.state_dict())
            if verbose:
                print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
        model.load_state_dict(state)
        return best


    def heat_map(model, image, size=64, stride=32):
        """Her 32 pikselde bir alınan 64 x 64 piksellik pencereler için darlık olasılığı: 512 x 512 boyutlu bir anjiyogram için 16 x 16 boyutlu bir ızgara."""
        h = size // 2
        pad = np.pad(image, h, mode="reflect")
        centres = np.arange(stride // 2, image.shape[0], stride)
        windows = [pad[r:r + size, c:c + size] for r in centres for c in centres]
        return predict(model, standardise(windows)).reshape(len(centres), len(centres)), centres


    def lesions_found(grid, centres, stenoses, threshold, near=32):
        """Merkezi bir darlığa 32 pikselden yakın olan bir pencere eşiğe ulaştığında darlık bulunmuş sayılır; her darlıktan 32 pikselden uzak olup eşiği aşan pencere yanlış alarmdır."""
        cy, cx = np.meshgrid(centres, centres, indexing="ij")
        hot = grid >= threshold
        found, alarm = 0, hot.copy()
        for s in stenoses:
            close = np.hypot(cx - s["x"], cy - s["y"]) <= near
            found += int((hot & close).any())
            alarm &= ~close
        return found, int(alarm.sum())
    from scipy.ndimage import maximum_filter
    data_ = {s: arcade.patches(s, per_stenosis=4, neg_per_pos=1.5, background=1.5, jitter=16) for s in arcade.SPLITS}
    torch.manual_seed(2026)
    net = Hybrid()
    train(net, standardise(data_["train"][0]), data_["train"][1], standardise(data_["val"][0]), data_["val"][1], epochs=8, verbose=False)

    def scan(split):
        return [(a, heat_map(net, arcade.load(split, a["file"]), stride=16)) for a in arcade.annotations(split)]

    def alarms(grid, centres, stenoses, t):
        cy, cx = np.meshgrid(centres, centres, indexing="ij")
        peaks = (grid >= t) & (grid == maximum_filter(grid, size=5))
        far = np.ones(peaks.sum(), dtype=bool)
        for s in stenoses:
            far &= np.hypot(cx[peaks] - s["x"], cy[peaks] - s["y"]) > 32
        return int(far.sum())

    val = scan("val")
    thr = min(t for t in np.arange(0.5, 1.0, 0.01) if np.mean([alarms(g, c, a["stenoses"], t) for a, (g, c) in val]) <= 1.0)
    test = scan("test")
    globals()["_detector_cache"] = pd.DataFrame({"anjiyogram": [a["id"] for a, _ in test], "görüntü": [(arcade.load("test", a["file"]) * 255).round().astype(np.uint8) for a, _ in test],
                                                 "darlıklar": [[(round(x["x"]), round(x["y"])) for x in a["stenoses"]] for a, _ in test],
                                                 "olasılıklar": [g for _, (g, c) in test], "eşik": round(float(thr), 2)})
    return _detector_cache
print("Proje hazır.")

### Proje adımı 1: Üç ağ, tek bir test

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Gerçek koroner anjiyogramlarda darlık bulmak için aynı biçimde eğitilmiş ve aynı anjiyogramlarda sınanmış evrişimli bir ağı, bir Vision Transformer'ı ve hibrit bir modeli karşılaştırmak istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan ARCADE'in darlık kümesindeki 1.500 X ışınlı koroner anjiyogramdan (eğitim 1.000 anjiyogram, doğrulama 200, test 300) kesilmiş 64 × 64 piksellik yamaları tutar. Her satır bir yamadır: "küme" sütunu yamanın kümesini ("eğitim", "doğrulama" ya da "test"), "anjiyogram" anjiyogramının numarasını verir; "yama" sütununun her hücresi yamayı gri değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü olarak tutar. "darlık" sütunu, yamanın merkezi uzmanların sınırını çizdiği bir darlığın merkezine 6 pikselden yakınsa 1, yama sınırı çizilmiş her darlıktan ve görüntünün kenarından en az 48 piksel uzakta bir damarın üzerindeyse 0 değerini alır.  
> Görev: Her yamayı kendi içinde standartlaştır: Gri değerlerinin ortalamasını çıkar ve standart sapmalarının 0,02 fazlasına böl. Üç ağ kur. Birincisi: Tek kanallı, 64 × 64 piksellik bir yama için küçük bir evrişimli ağ kur: Her biri 3 × 3 piksellik filtreler ve bir piksellik dolgu içeren bir evrişim (16, 32, 64 ve 64 filtre), batch normalizasyonu, ReLU ve 2 × 2 en büyük havuzlamadan oluşan dört blok; ardından her kanalın konumları üzerinden ortalaması ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman. İkincisi: Tek kanallı, 64 × 64 piksellik bir yama için küçük bir Vision Transformer kur. Yamayı 8 × 8 piksellik 64 kareye böl ve her kareyi öğrenilen aynı doğrusal dönüşümle 64 sayıya dönüştür. 64 token'ın önüne 64 sayıdan oluşan öğrenilen bir sınıf token'ı koy ve öğrenilen konumları ekle: 65 yerin her biri için, standart sapması 0,02 olan küçük rastgele sayılarla başlayan 64 sayılık bir vektör. Token'ları dört Transformer bloğundan geçir. Her blokta katman normalleştirmesini dört başlı öz dikkat izler ve öz dikkatin çıktısı bloğun girdisine eklenir; ikinci bir katman normalleştirmesini 128 birimlik bir gizli katman ve GELU etkinleştirmesi içeren ileri beslemeli bir ağ izler ve bu ağın çıktısı da yeniden eklenir. Sonunda sınıf token'ını al, katman normalleştirmesi ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman uygula. Üçüncüsü: Tek kanallı, 64 × 64 piksellik bir yama için hibrit bir ağ kur. Her biri 3 × 3 piksellik filtreler ve bir piksellik dolgu içeren bir evrişim (16, 32 ve 64 filtre), batch normalizasyonu, ReLU ve 2 × 2 en büyük havuzlamadan oluşan üç evrişim bloğu, yamayı 64 sayılık 8 × 8 konuma dönüştürür. Bu 64 konum küçük bir Transformer modelinin token'larıdır: Önlerine 64 sayıdan oluşan öğrenilen bir sınıf token'ı koy, 65 yer için standart sapması 0,02 olan küçük rastgele sayılarla başlayan öğrenilen konumları ekle ve token'ları iki Transformer bloğundan geçir. Her blokta katman normalleştirmesini dört başlı öz dikkat izler ve öz dikkatin çıktısı bloğun girdisine eklenir; ikinci bir katman normalleştirmesini 128 birimlik bir gizli katman ve GELU etkinleştirmesi içeren ileri beslemeli bir ağ izler ve bu ağın çıktısı da yeniden eklenir. Sonunda sınıf token'ını al, katman normalleştirmesi ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman uygula. Her ağı eğitim yamalarıyla 8 epoch boyunca, karıştırılmış 64 yamalık gruplar hâlinde, logit üzerinde ikili çapraz entropi kaybıyla ve öğrenme hızı 0,001 olan Adam optimizasyon algoritmasıyla eğit. Veri çoğaltma için her grubu rastgele bir çeyrek tur döndür ve yarı olasılıkla soldan sağa aynala. Her epoch'un sonunda doğrulama yamalarının olasılıklarının AUC değerini hesapla; eğitimin sonunda doğrulama AUC değeri en büyük olan epoch'un ağırlıklarını sakla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Ardından test yamalarının olasılıklarını ve bunların "darlık" sütununa göre AUC değerini hesapla. 300 test anjiyogramından yerine koyarak 2.000 bootstrap örneği çek, çekilen her anjiyogramın bütün yamalarını tut ve her AUC değerine 2,5. ve 97,5. yüzdelikler arasındaki yüzde 95 aralığını ver. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Aynı örneklerde Vision Transformer'ın ve hibrit modelin AUC değerinin evrişimli ağınkine göre farkını hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her ağ için eğitilebilir ağırlıklarını, eğitim süresini ve aralığıyla birlikte test AUC değerini içeren bir satırdan oluşan bir tablo ile iki farkı aralıklarıyla birlikte göster; en büyük test AUC değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Üç ağın eğitilebilir ağırlık sayıları 60.641, 142.465 ve 94.881 olmalı ve aralıklar yamalardan değil anjiyogramlardan hesaplanmalı.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _arcade_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Laboratuvar için bir eşik

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Anjiyogramlarda darlık arayan ağların sabit bir eşikte uygulamada ne vereceğini, hem özenle seçilmiş yamalar arasında hem de darlığın seyrek olduğu bütün bir anjiyogramın pencereleri arasında öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir X ışınlı koroner anjiyogram veri seti olan ARCADE'in 64 × 64 piksellik 6.336 doğrulama ve test yamasını (doğrulama 200 anjiyogram, test 300) tutar. Her satır bir yamadır: "küme" sütunu yamanın kümesini ("doğrulama" ya da "test"), "anjiyogram" anjiyogramının numarasını verir; "darlık" sütunu merkezi uzmanların sınırını çizdiği bir darlıkta olan yama için 1, her darlıktan uzaktaki bir damarın üzerindeki yama için 0 değerini alır. "evrişimli ağ", "Vision Transformer" ve "hibrit model" adlı üç sütun, dersin eğitim yamalarıyla eğitilmiş üç küçük ağının verdiği darlık olasılığını tutar.  
> Görev: Üç ağın her biri için "doğrulama" kümesinin yamalarında, darlık içeren yamaların yüzde 90'ının ulaştığı eşiği seç. Bu eşiği "test" kümesinin yamalarına uygula; duyarlılığı, özgüllüğü ve pozitif öngörü değerini hesapla. Ardından aynı duyarlılık ve özgüllüğün yüzde 2 prevalansta vereceği pozitif öngörü değerini hesapla: Duyarlılık ile 0,02'nin çarpımı, kendisi ile bir eksi özgüllüğün 0,98 ile çarpımının toplamına bölünür.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her ağ için eşiğini ve dört değeri içeren bir satırdan oluşan bir tablo göster; hibrit modelin yüzde 2 prevalanstaki pozitif öngörü değerini üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Eşikler yalnızca doğrulama yamalarında seçilmeli ve her değer 0 ile 1 arasında olmalı.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _probability_table()   # ilk seferde yaklaşık yedi dakika sürer: Dersin üç ağı bir kez eğitilir
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: Bütün anjiyogramlar için bir tespit modeli

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Bütün koroner anjiyogramlarda darlıkları, anjiyogram başına yaklaşık bir yanlış tespitle işaretleyen bir program istiyorum ve programın darlıkların ne kadarını bulduğunu öğrenmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir veri seti olan ARCADE'in darlık kümesindeki 1.500 X ışınlı koroner anjiyogramı (eğitim 1.000, doğrulama 200, test 300) tutar. Her satır bir anjiyogramdır: "küme" sütunu kümesini ("eğitim", "doğrulama" ya da "test"), "anjiyogram" numarasını verir; "görüntü" sütununun her hücresi anjiyogramı gri değerleri 0 ile 255 arasında olan 512 × 512 piksellik bir görüntü olarak tutar; "darlıklar" sütunu uzmanların sınırını çizdiği darlıkların merkezlerini, 0'dan başlayarak sayılan piksel olarak (sütun, satır) çiftleri hâlinde listeler.  
> Görev: Her görüntüyü 0 ile 1 arasında gri değerlere getir. Her anjiyogramdan 64 × 64 piksellik pencereler kes: Her darlık için, darlığı ortalayan ve her yönde rastgele en fazla 16 piksel kaydırılmış dört pencere (etiket 1); her darlık için, görüntünün kenarından en az 48 piksel uzaktaki ince ve koyu yapıları ortalayan altı pencere (bu yapılar, yarıçapı 6 olan bir diskle uygulanan siyah şapka dönüşümünde değeri en büyük yüzde 3 içinde kalan piksellerdir) ve rastgele yerlerde altı pencere; bunların hepsi her darlıktan en az 48 piksel uzakta olmalı (etiket 0). Her yamayı kendi içinde standartlaştır: Gri değerlerinin ortalamasını çıkar ve standart sapmalarının 0,02 fazlasına böl. Tek kanallı, 64 × 64 piksellik bir yama için hibrit bir ağ kur. Her biri 3 × 3 piksellik filtreler ve bir piksellik dolgu içeren bir evrişim (16, 32 ve 64 filtre), batch normalizasyonu, ReLU ve 2 × 2 en büyük havuzlamadan oluşan üç evrişim bloğu, yamayı 64 sayılık 8 × 8 konuma dönüştürür. Bu 64 konum küçük bir Transformer modelinin token'larıdır: Önlerine 64 sayıdan oluşan öğrenilen bir sınıf token'ı koy, 65 yer için standart sapması 0,02 olan küçük rastgele sayılarla başlayan öğrenilen konumları ekle ve token'ları iki Transformer bloğundan geçir. Her blokta katman normalleştirmesini dört başlı öz dikkat izler ve öz dikkatin çıktısı bloğun girdisine eklenir; ikinci bir katman normalleştirmesini 128 birimlik bir gizli katman ve GELU etkinleştirmesi içeren ileri beslemeli bir ağ izler ve bu ağın çıktısı da yeniden eklenir. Sonunda sınıf token'ını al, katman normalleştirmesi ve tek çıktılı, yani darlığın logitini veren bir doğrusal katman uygula. Ağı eğitim yamalarıyla 8 epoch boyunca, karıştırılmış 64 yamalık gruplar hâlinde, logit üzerinde ikili çapraz entropi kaybıyla ve öğrenme hızı 0,001 olan Adam optimizasyon algoritmasıyla eğit. Veri çoğaltma için her grubu rastgele bir çeyrek tur döndür ve yarı olasılıkla soldan sağa aynala. Her epoch'un sonunda doğrulama yamalarının olasılıklarının AUC değerini hesapla; eğitimin sonunda doğrulama AUC değeri en büyük olan epoch'un ağırlıklarını sakla. Sonucun tekrarlanabilmesi için seed değerini 2026 olarak sabitle. Ardından her doğrulama ve test anjiyogramını, merkezleri 8'den başlayarak her 16 pikselde bir bulunan 64 × 64 piksellik pencerelerle tara; görüntü kenarlarında yansıtılarak genişletilir. Her pencerenin olasılığını hesapla. Tespit, olasılığı eşiğe eşit ya da eşikten büyük olan ve 5 × 5 merkezlik komşuluğundaki pencereler arasında en büyük olasılığa sahip penceredir. Bir tespit bir darlığa 32 pikselden yakınsa darlık bulunmuş sayılır; öteki her tespit yanlıştır. Doğrulama anjiyogramlarında anjiyogram başına en fazla bir yanlış tespit veren, 0,50 ile 0,99 arasında 0,01 adımlarla seçilmiş en küçük eşiği bul ve bu eşiği test anjiyogramlarına uygula.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Seçilen eşiği yazdır. Ardından 0,50, 0,70, 0,90 eşiklerinin ve seçilen eşiğin her biri için bir satırdan oluşan bir tablo göster: Test anjiyogramlarındaki bütün darlıklar içinde bulunan darlık sayısı ve test anjiyogramı başına yanlış tespit sayısı. Seçilen eşikte bulunan darlıkların oranını üç ondalık basamakla "cevap" adıyla sakla.  
> Test: Eşik yalnızca doğrulama anjiyogramlarında seçilmeli ve test anjiyogramlarında 386 darlık bulunmalı.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _angiogram_table()
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Bir kardiyoloğun okuduğu hatalar

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter Notebook'uma yapıştırıp çalıştıracağım.  
> Bağlam: Böyle bir aracın kateter laboratuvarında neler yapabileceğine karar verebilmek için bir tespit modelinin kaçırdığı darlıkları ve yanlış işaretlediği yerleri görmek istiyorum.  
> Veri: Jupyter Notebook'ta "veri" adında bir tablo var; açık bir X ışınlı koroner anjiyogram veri seti olan ARCADE'in 300 test anjiyogramını tutar. Her satır bir anjiyogramdır: "anjiyogram" sütunu numarasını verir; "görüntü" sütununun her hücresi anjiyogramı gri değerleri 0 ile 255 arasında olan 512 × 512 piksellik bir görüntü olarak tutar; "darlıklar" sütunu uzmanların sınırını çizdiği darlıkların merkezlerini 0'dan başlayarak sayılan piksel olarak (sütun, satır) çiftleri hâlinde listeler. "olasılıklar" sütununun her hücresi, dersin tespit modelinin verdiği darlık olasılıklarını 32 × 32 sayılık bir tablo olarak tutar; her sayı, merkezi her yönde 8, 24, 40 ve böyle 504 piksele kadar giden noktalarda bulunan 64 × 64 piksellik bir pencereye aittir. "eşik" sütunu tespit modelinin, doğrulama anjiyogramlarında anjiyogram başına en fazla bir yanlış tespit verecek biçimde seçilmiş eşiğini tutar.  
> Görev: Tespit, olasılığı eşiğe eşit ya da eşikten büyük olan ve 5 × 5 merkezlik komşuluğundaki pencereler arasında en büyük olasılığa sahip penceredir. Bir tespit bir darlığa 32 pikselden yakınsa darlık bulunmuş sayılır; öteki her tespit yanlıştır. Her anjiyogram için kaçırılan darlıkları ve yanlış tespitleri olasılıklarıyla birlikte listele. Ardından beş kaçırılmış darlığı ve olasılığı en büyük beş yanlış tespiti, her birini çevresindeki 128 × 128 piksellik pencerede, olasılıklar görüntünün üzerine çizilmiş, uzmanların darlıkları daire ve tespitler kare olarak gösterilmiş biçimde göster.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: On pencereyi göster, kaçırılan darlıkların ve yanlış tespitlerin sayısını yazdır; kaçırılan darlıkların sayısını "cevap" adıyla sakla.  
> Test: Kaçırılan ve bulunan darlıkların toplamı 386 olmalı.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = _detector_table()   # ilk seferde yaklaşık on beş dakika sürer: Dersin tespit modeli bir kez eğitilir ve anjiyogramları tarar
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 7: Dizi olarak görüntüler ve dikkatin aritmetiği

Bu haftanın verileri görüntülerdir ve ağlar bir görüntünün her parçasını öteki her parçasıyla karşılaştırır. Bu adımda bir görüntünün bir sayı dizisinde nasıl durduğu, bir dizinin içinden bir pencerenin nasıl kesildiği, bir görüntünün `reshape` ve `transpose` ile nasıl yamalara bölündüğü, matris çarpımı `@` ve softmax'ın bütün bir token tablosu için dikkati nasıl tek seferde hesapladığı ve bir dizinin bir ağın okuyabileceği bir tensöre nasıl dönüştüğü gösterilir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
views = angiogram.patient_frames("VC-2427")
left, truth = views["left"]
print(left.shape, left.dtype, left.min(), left.max())
image = left / 255
lad = truth["stenoses"][0]
x, y = int(lad["x"]), int(lad["y"])
window = image[y - 32:y + 32, x - 32:x + 32]
print(lad["vessel"], lad["stenosis"], "%", "| pencere", window.shape, "| ortalama gri değer", round(float(window.mean()), 3))
fig, ax = plt.subplots(1, 2, figsize=(7, 3.4))
ax[0].imshow(image, cmap="gray", vmin=0, vmax=1)
ax[0].add_patch(plt.Rectangle((x - 32, y - 32), 64, 64, fill=False, color="#9B1B1B"))
ax[1].imshow(window, cmap="gray", vmin=0, vmax=1)
for a in ax:
    a.axis("off")
plt.show()

**Python notu.** `angiogram.patient_frames` sanal kliniğin bir hastasının iki anjiyogramını çizer ve `"left"` ile `"right"` anahtarlarını içeren bir sözlük döndürür; her değer bir görüntü ile onun doğrusundan oluşan bir çifttir ve `left, truth = ...` ikisine de bir ad verir. Görüntü, 0 ile 255 arasında değer alan 512 × 512 tam sayıdan (`uint8`) oluşan bir dizidir; 255'e bölmek 0 ile 1 arasında gri değerler verir. Önce satırlar gelir: `image[y - 32:y + 32, x - 32:x + 32]`, `y` satırının çevresindeki 64 satırı ve `x` sütununun çevresindeki 64 sütunu, yani darlığın çevresinde 64 × 64 piksellik bir pencereyi alır. `plt.Rectangle`, köşesi ve boyutu piksel olarak verilen bir kutu çizer.

In [ ]:
def to_patches(image, p):
    """Gri bir görüntüyü satır satır p x p piksellik karelere böler; her kare p * p sayıdan oluşan bir satır olur."""
    h, w = image.shape[0] // p * p, image.shape[1] // p * p
    return image[:h, :w].reshape(h // p, p, w // p, p).transpose(0, 2, 1, 3).reshape(-1, p * p)

# Python alıştırması: yamalara bölmek
a = np.arange(16).reshape(4, 4)
print(a)
print(to_patches(a, 2))
small = np.asarray(Image.fromarray(left).resize((224, 224), Image.BILINEAR)) / 255
rows = to_patches(small, 16)
print(small.shape, "→", rows.shape)

**Python notu.** `np.arange(16).reshape(4, 4)`, 0'dan 15'e kadar sayıları 4 satırlık ve 4 sütunluk bir diziye yazar. `to_patches` bir görüntüyü dört adımda karelere böler: `reshape(h // p, p, w // p, p)` her piksele dört indis verir (yamasının satırı, yama içindeki satırı, yamasının sütunu, yama içindeki sütunu); `transpose(0, 2, 1, 3)` yamanın iki indisini öne alır; son `reshape` her yamayı $p \times p$ sayıdan oluşan bir satıra dönüştürür. `a` dizisinin ilk yaması 0, 1, 4, 5'tir: Sol üstteki kare. `Image.fromarray(...).resize((224, 224), Image.BILINEAR)`, anjiyogramı ViT-B/16'nın beklediği 224 × 224 piksele küçültür; her yönde 224/16 = 14 yama, 256 sayılık 196 satır verir.

In [ ]:
def softmax(scores, axis=-1):
    """Puanları, bir eksen boyunca pozitif olan ve toplamı 1 eden ağırlıklara dönüştürür."""
    e = np.exp(scores - scores.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V):
    """Ölçeklenmiş iç çarpım dikkati: Her sorgu, anahtarlara benzerliğinden gelen ağırlıklarla değerlerin ağırlıklı ortalamasını alır."""
    weights = softmax(Q @ K.T / np.sqrt(K.shape[1]))
    return weights @ V, weights

# Python alıştırması: matris çarpımı ve softmax
Q = np.array([[1.0, 0.0]])
K = np.array([[2.0, 0.0], [0.0, 2.0], [1.0, 1.0]])
V = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
scores = Q @ K.T / np.sqrt(2)
print("puanlar", np.round(scores, 3))
output, weights = attention(Q, K, V)
print("ağırlıklar", np.round(weights, 3), "| toplam", weights.sum())
print("çıktı", np.round(output, 3))

**Python notu.** `@` matris çarpımıdır: `Q @ K.T`, her sorguyu (`Q` dizisinin bir satırı) her anahtarla (`.T` ile sütuna dönüştürülmüş `K` satırı) çarpar ve her sorgu için bir satır, her anahtar için bir sütun içeren bir tablo verir. `softmax`, `np.exp` uygulanmadan önce her satırın en büyük puanını çıkarır; bu işlem sonucu değiştirmez, ama büyük puanların taşmasını önler. `keepdims=True` toplamları bir sütun olarak tutar; böylece her satır kendi toplamına bölünür. `attention` iki şey döndürür, çıktıyı ve ağırlıkları; `output, weights = ...` ikisine de ad verir. Sayılar ders sayfasındaki çözümlü örneğin sayılarıdır.

In [ ]:
# Python alıştırması: diziden tensöre
x = torch.as_tensor(small, dtype=torch.float32)[None, None]
print(x.shape, "→", x.repeat(1, 3, 1, 1).shape)
embed = nn.Linear(256, 768)
print("doğrusal dönüşümün ağırlıkları:", sum(p.numel() for p in embed.parameters()))
with torch.no_grad():
    tokens = embed(torch.as_tensor(rows, dtype=torch.float32))
print("token'lar:", tuple(tokens.shape))

**Python notu.** Bir ağ tensörleri, yani PyTorch dizilerini okur: Bir batch'teki görüntüler birinci eksende, kanallar ikinci eksende durur. `[None, None]` ikisini birden ekler; böylece 224 × 224 boyutlu tek bir gri görüntü 1 × 1 × 224 × 224 boyutlu bir tensör olur. `.repeat(1, 3, 1, 1)` renkli görüntü bekleyen bir ağ için gri kanalı üç kez kopyalar. `nn.Linear(256, 768)`, 256 sayıdan 768 sayıya giden doğrusal bir dönüşümdür: 256 × 768 ağırlık ve 768 sabit terimden oluşan bir tablo, toplam 197.376 sayı. Yamaların 196 satırına uygulandığında 768 sayılık 196 token verir; bu, tek gri kanallı bir Vision Transformer'ın yama gömmesidir. `torch.no_grad()`, eğitimin ihtiyaç duyacağı kayıtları tutmadan hesap yapar.

**Hızlı kontrol.** Bir görüntünün biçimi (512, 512) olsun. `image[100:164, 200:264]` ifadesinin biçimi nedir?

<details><summary>Cevap</summary>

(64, 64): 100-163. satırlar ve 200-263. sütunlar. Bir dilimin sonu dâhil değildir.

</details>

**Alıştırma 7.1.** `to_patches`, vakanın 224 × 224 piksele küçültülmüş sol anjiyogramını (`small`) 16 × 16 piksellik kaç yamaya böler? Yama sayısını saklayın.

In [ ]:
veri = alistirma_tablosu("7.1")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 7.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = to_patches(small, 16).shape[0]
teach.check("Alıştırma 7.1", cevap)

**Alıştırma 7.2.** Çözümlü örneğin anahtarları ve değerleriyle (0, 1) sorgusu için üçüncü token'ın dikkat ağırlığı kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma 7.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
_, w = attention(np.array([[0.0, 1.0]]), K, V)
cevap = round(float(w[0, 2]), 3)
teach.check("Alıştırma 7.2", cevap)

**Alıştırma 7.3.** Vakanın sol anjiyogramında sirkumfleks arterdeki darlığı ortalayan 64 × 64 piksellik pencerenin 0 ile 1 arasındaki ortalama gri değeri kaçtır? Darlık `truth["stenoses"][1]` öğesidir. Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("7.3")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 7.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
s = truth["stenoses"][1]
x1, y1 = int(s["x"]), int(s["y"])
cevap = round(float(image[y1 - 32:y1 + 32, x1 - 32:x1 + 32].mean()), 3)
teach.check("Alıştırma 7.3", cevap)

## 1. Sanal kliniğin anjiyogramları

*Ders: [Kateter laboratuvarının görüntüleri: Sanal kliniğin anjiyogramları](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#images-of-the-catheterisation-laboratory-the-angiograms-of-the-virtual-clinic)*

Kayıt sisteminde anjiyografisi olan her hastanın iki sentetik anjiyogramı vardır: Sol koroner arterin bir görüntüsü ve sağ koroner arterin bir görüntüsü. Bu görüntüler 6. haftanın raporlarının anlattığı lezyonlardan çizilir. Her karenin doğrusu, karedeki darlıkları damarları, segmentleri, dereceleri, merkezleri ve kutularıyla birlikte listeler.

**Formüllerden koda.** Anlamlı darlık, sol ana koroner arterde yüzde 50 ya da üstü, öteki bölgelerde yüzde 70 ya da üstü darlıktır: `coronary.significant(territory, stenosis)`. Bir damarın görüntüsü: `angiogram.VIEW_OF[vessel]`, yani `"left"` ya da `"right"`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
patients, lesions = angiogram.registry()
split = patients["split"].to_numpy()
CASE = "VC-2427"
case_views = angiogram.patient_frames(CASE)

print(patients["split"].value_counts())
print(lesions[lesions["patient_id"] == CASE][["vessel", "segment", "stenosis", "kind"]])
fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
for ax, (view, (img, t)) in zip(axes, case_views.items()):
    ax.imshow(img, cmap="gray")
    for s in t["stenoses"]:
        bx, by, bw, bh = s["box"]
        ax.add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, lw=1.5, color="#9B1B1B" if s["significant"] else "#B97813"))
        ax.text(bx, by - 6, f"{s['vessel']} {s['stenosis']}%", color="#9B1B1B" if s["significant"] else "#B97813", fontsize=9)
    ax.set_title(view)
    ax.axis("off")
plt.show()

**Python notu.** `angiogram.registry()` iki tablo döndürür: Anjiyografisi olan her hasta için kümesini de içeren bir satır bulunan `patients` ve her lezyon için damarı, segmenti, yüzde olarak derecesi ve türüyle birer satır bulunan `lesions`. Kümeler, hasta numarasına göre ayrılan 6. hafta kümeleridir. `case_views.items()` (görüntü, kare) çiftlerini verir; `zip` bunları şeklin iki ekseniyle birlikte dolaşır. Bir karenin doğrusunda `box`, darlığın çevresindeki karenin piksel olarak (x, y, genişlik, yükseklik) değerleridir.

In [ ]:
sig = [int(coronary.significant(t, s)) for t, s in zip(lesions["territory"], lesions["stenosis"])]
table = lesions.assign(significant=sig, view=lesions["vessel"].map(angiogram.VIEW_OF)).merge(patients[["patient_id", "split"]], on="patient_id")
print(pd.crosstab(table["split"], [table["view"], table["significant"]]))

**Gözlemle ve düşün.** Vakanın iki anjiyogramına bakın. Hangi darlığa gözle anlamlı derdiniz ve sentetik görüntü gerçek bir anjiyogramın hangi ayrıntısını dışarıda bırakıyor?

**Alıştırma A.** Test kümesindeki 394 hastanın kaçında sol koroner görüntüsündeki bir damarda en az bir anlamlı darlık vardır? Yukarıdaki hücrenin `table` tablosunu kullanın.

In [ ]:
veri = alistirma_tablosu("A")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(table[(table["split"] == "test") & (table["view"] == "left") & (table["significant"] == 1)]["patient_id"].nunique())
teach.check("Alıştırma A", cevap)

## 2. Elle dikkat

*Ders: [Self-attention: Görüntünün farklı bölgeleri arası ilişki](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#self-attention-the-relation-between-different-regions-of-the-image)*

Dikkat bir sorguyu her anahtarla karşılaştırır, puanları softmax ile ağırlıklara dönüştürür ve değerlerin ağırlıklı ortalamasını aktarır. Öz dikkatte sorgular, anahtarlar ve değerler aynı token'ların doğrusal dönüşümleridir.

**Formüllerden koda.** $\mathrm{softmax}(QK^{\top}/\sqrt{d_k})\,V$: `attention` fonksiyonundaki `softmax(Q @ K.T / np.sqrt(K.shape[1])) @ V`. Öz dikkat: `Q = X @ Wq`, `K = X @ Wk`, `V = X @ Wv`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

**Evrişimli bir ağ ne kadar uzağı görür?** *Ders: [CNN'in sınırları: Global bağlam eksikliği](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#from-convolution-to-attention)*

Evrişimli bir katmandaki bir konum görüntünün yalnızca bir kısmını, yani *alıcı alanını* (receptive field) görür. 3 × 3 piksellik her evrişim alanı alttaki katmanın iki adımı kadar genişletir; 2 × 2 boyutlu her havuzlama ise adımı iki katına çıkarır. Hücre, bu haftanın 5. bölümden itibaren kullanılan küçük evrişimli ağını sekiz katmanı boyunca izler.

**Formüllerden koda.** $r_l = r_{l-1} + (k_l - 1)\,j_{l-1}$ ve $j_l = j_{l-1}\,s_l$: `receptive_field` içindeki `field += (kernel - 1) * jump` ve `jump *= stride`.

In [ ]:
def receptive_field(layers):
    """Bir katman yığınında her katmandan sonraki alıcı alan; katmanlar (ad, çekirdek, adım) biçimindedir. Alan, (çekirdek - 1) ile önceki katmanların adımlarının çarpımı kadar büyür."""
    field, jump, rows = 1, 1, []
    for name, kernel, stride in layers:
        field += (kernel - 1) * jump
        jump *= stride
        rows.append({"katman": name, "alıcı alan (piksel)": field, "konumlar arası adım": jump})
    return pd.DataFrame(rows)


SMALL_CNN_LAYERS = [("evrişim 1", 3, 1), ("havuzlama 1", 2, 2), ("evrişim 2", 3, 1), ("havuzlama 2", 2, 2),
                    ("evrişim 3", 3, 1), ("havuzlama 3", 2, 2), ("evrişim 4", 3, 1), ("havuzlama 4", 2, 2)]

print(receptive_field(SMALL_CNN_LAYERS).to_string(index=False))
layers = int(np.ceil((512 - 1) / 2))
print("512 piksellik bir alan için havuzlamasız 3 x 3 evrişim sayısı:", layers)

**Python notu.** `receptive_field` katmanları sırayla dolaşır ve iki sayı tutar: Alanın girdi pikselleri cinsinden genişliği ve katmandaki iki komşu konum arasındaki adım. `int(np.ceil(...))` sonucu tam sayıda katmana yukarı yuvarlar. Küçük ağın son konumları 64 × 64 piksellik bir yamanın 46 × 46 pikselini görür; bütün yamayı ancak sondaki, tüm konumların ortalaması bir araya getirir.

In [ ]:
def softmax(scores, axis=-1):
    """Puanları, bir eksen boyunca pozitif olan ve toplamı 1 eden ağırlıklara dönüştürür."""
    e = np.exp(scores - scores.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def attention(Q, K, V):
    """Ölçeklenmiş iç çarpım dikkati: Her sorgu, anahtarlara benzerliğinden gelen ağırlıklarla değerlerin ağırlıklı ortalamasını alır."""
    weights = softmax(Q @ K.T / np.sqrt(K.shape[1]))
    return weights @ V, weights

for angle in (0, 45, 90, 180):
    q = np.array([[np.cos(np.radians(angle)), np.sin(np.radians(angle))]])
    _, w = attention(q, K, V)
    print(f"sorgu açısı {angle:3}°: ağırlıklar {np.round(w[0], 3)}")

**Python notu.** `np.radians` dereceyi radyana çevirir; `np.cos` ve `np.sin` o yöne bakan 1 uzunluğunda bir sorgu verir. 0° açıda sorgu ilk token'ın anahtarıyla aynı yöne bakar ve ona en büyük ağırlığı verir; 90° açıda ikinci token'ın anahtarıyla aynı yöne bakar; 180° açıda ise ikisinden de uzağa bakar ve ağırlıklar, anahtarı en az ters yönde olan üçüncü token'a kayar.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
X = to_patches(small, 16)[[40, 41, 54, 120, 190]]          # anjiyogramdan beş yama
Wq, Wk, Wv = (rng.normal(0, 0.05, size=(256, 16)) for _ in range(3))
out, w = attention(X @ Wq, X @ Wk, X @ Wv)
print(np.round(w, 2))
plt.figure(figsize=(3.2, 2.8))
plt.imshow(w, cmap="Oranges", vmin=0, vmax=1)
plt.colorbar()
plt.xlabel("anahtar")
plt.ylabel("sorgu")
plt.show()

**Python notu.** Anjiyogramdan beş yama, her biri 256 gri değerden oluşan bir satır olarak, beş token olur. 256 × 16 boyutlu, rastgele sayılardan oluşan üç matris bunları 16 sayılık sorgulara, anahtarlara ve değerlere dönüştürür; bir ağ bu matrisleri öğrenirdi. Ağırlık tablosunda her sorgu için bir satır vardır ve her satırın toplamı 1'dir. Rastgele matrislerle ağırlıklar yalnızca yamaların birbirine ne kadar benzediğini yansıtır: Parlaklığı benzer yamalar birbirine dikkat eder.

**Alıştırma B.** Anahtar uzunluğunun kareköküne bölme yapılmadan, (1, 0) sorgusu için çözümlü örneğin ilk token'ı hangi ağırlığı alır? Üç ondalık basamakla saklayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(softmax(np.array([[1.0, 0.0]]) @ K.T)[0, 0]), 3)
teach.check("Alıştırma B", cevap)

## 3. Yamalar ve Vision Transformer ViT-B/16

*Ders: [Patch embedding: Görüntüyü 16×16 parçalara ayırma](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#the-vision-transformer-an-image-as-a-sequence-of-patches)*

ViT-B/16, 224 × 224 piksellik bir görüntüyü 16 × 16 piksellik 196 yamaya böler, her yamayı 768 sayılık bir token'a dönüştürür, token'ların önüne bir sınıf token'ı koyar ve öğrenilen konumları ekler; ardından 12 Transformer bloğu gelir. Son katmanı olmadan ağ, son bloğun sınıf token'ını döndürür.

**Formüllerden koda.** $N = (224/16)^2 = 196$: `to_patches(small, 16)` dizisinin 196 satırı. Yama gömme $E$, `vit.conv_proj`; sınıf token'ı $x_{\mathrm{cls}}$, `vit.class_token`; konumlar $E_{\mathrm{pos}}$, `vit.encoder.pos_embedding`; 12 blok ise `vit.encoder.layers` öğesidir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1).eval()
vit.heads = nn.Identity()      # son katmanı olmadan ağ sınıf token'ını döndürür: Görüntü başına 768 sayı
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def to_imagenet(images, size=224):
    """0 ile 1 arasında değer alan gri görüntüler → ImageNet ortalaması ve standart sapmasıyla 3 x 224 x 224 boyutlu tensörler."""
    x = torch.as_tensor(np.asarray(images, dtype=np.float32))[:, None]
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
    return (x - MEAN) / STD


@torch.no_grad()
def vit_features(images, batch=16):
    """ViT-B/16'nın her görüntü için verdiği sınıf token'ı: n x 768 sayıdan oluşan bir dizi."""
    return torch.cat([vit(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()


@torch.no_grad()
def vit_attention(image):
    """ViT-B/16'nın 12 katmanının bir görüntü için dikkati, 12 başın ortalaması olarak: 12 x 197 x 197 boyutlu bir dizi."""
    x = vit._process_input(to_imagenet([image]))
    x = torch.cat([vit.class_token, x], dim=1) + vit.encoder.pos_embedding
    maps = []
    for block in vit.encoder.layers:
        y = block.ln_1(x)
        out, w = block.self_attention(y, y, y, need_weights=True, average_attn_weights=True)
        maps.append(w[0].numpy())
        x = x + out
        x = x + block.mlp(block.ln_2(x))
    return np.stack(maps)


def rollout(maps):
    """Dikkat aktarımı (rollout): Artık bağlantısı eklenmiş her katmanın dikkati, ilk katmandan sonuncuya doğru çarpılır."""
    r = np.eye(maps.shape[-1])
    for a in maps:
        a = 0.5 * a + 0.5 * np.eye(len(a))
        r = (a / a.sum(axis=1, keepdims=True)) @ r
    return r

parts = {"yama gömme": vit.conv_proj, "12 blok": vit.encoder.layers, "son normalleştirme": vit.encoder.ln}
for name, part in parts.items():
    print(f"{name:22} {sum(p.numel() for p in part.parameters()):>12,}")
counts = {"sınıf token'ı": vit.class_token.numel(),
          "konumlar": vit.encoder.pos_embedding.numel(),
          "toplam": sum(p.numel() for p in vit.parameters())}
for name, n_params in counts.items():
    print(f"{name:22} {n_params:>12,}")
with torch.no_grad():
    print("vakanın sınıf token'ı:", vit(to_imagenet([image])).shape)

**Python notu.** `vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)` ağı kurar ve 0. bölümde indirilen, ImageNet üzerinde öğrenilmiş ağırlıkları yükler; `.eval()` yalnızca eğitimin ihtiyaç duyduğu şeyleri kapatır. 1.000 sınıflı son katman `vit.heads`, girdisini değiştirmeden aktaran bir katman olan `nn.Identity()` ile değiştirildiğinde ağ sınıf token'ını döndürür. `to_imagenet`, gri görüntüleri ağın öğrendiği girdiye dönüştürür: 224 × 224 piksel, üç eşit renk kanalı ve ImageNet'in ortalaması ve standart sapması. 85.798.656 olan toplamda, kaldırılan son katmanın 769.000 ağırlığı yoktur.

In [ ]:
maps = vit_attention(image)
roll = rollout(maps)
fig, ax = plt.subplots(1, 4, figsize=(11, 3))
ax[0].imshow(small, cmap="gray")
for a, (title, m) in zip(ax[1:], (("1. katman", maps[0]), ("12. katman", maps[11]), ("dikkat aktarımı", roll))):
    a.imshow(small, cmap="gray")
    a.imshow(np.kron(m[0, 1:].reshape(14, 14), np.ones((16, 16))), cmap="inferno", alpha=0.55)
    a.set_title(title)
for a in ax:
    a.axis("off")
plt.show()

**Python notu.** `vit_attention` ağın 12 bloğunu tek tek çalıştırır ve her bloktan, 12 başının ortalaması olan dikkat ağırlıklarını ister: 12 × 197 × 197 boyutlu bir dizi. 0. satır sınıf token'ıdır; `m[0, 1:]` onun 196 yama üzerindeki dikkatidir ve `reshape(14, 14)` bu dikkati görüntünün ızgarasına geri yerleştirir. `np.kron` bir birler bloğuyla her değeri çizim için 16 × 16 piksele büyütür. `rollout` katmanları ders sayfasındaki gibi çarpar.

**Gözlemle ve düşün.** Ağ hiç anjiyogram görmemiştir. Son katmanda ve dikkat aktarımında dikkati hangi yapılarda toplanıyor ve fotoğraflarla eğitilmiş bir ağ neden oraya dikkat eder?

**Alıştırma C.** ViT-B/16'nın ilk Transformer bloğu `vit.encoder.layers[0]` kaç ağırlık taşır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in vit.encoder.layers[0].parameters())
teach.check("Alıştırma C", cevap)

**PyTorch ile kurulan bir Vision Transformer.** *Ders: [ViT PyTorch ile inşa; Hugging Face üzerinden hazır ViT modeli](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#building-a-vit-with-pytorch-a-ready-vit-model-from-hugging-face)*

Aşağıdaki sınıflar küçük bir Vision Transformer'ı parça parça kurar; plan, daha küçük ölçekte ViT-B/16'nın planıdır: Yama gömme, sınıf token'ı, öğrenilen konumlar, Transformer blokları ve son katman. 4. bölümdeki mini proje bu ağı CIFAR-10 fotoğraflarıyla, 5-8. bölümler ise anjiyogram yamalarıyla eğitir.

**Formüllerden koda.** Her yama için $x_i E$: Filtresi adımı kadar büyük olan ve böylece her kareye bir kez uygulanan bir evrişim, `nn.Conv2d(channels, dim, patch, stride=patch)`. $x_{\mathrm{cls}}$: `self.cls`. $E_{\mathrm{pos}}$: `self.pos`. Ders sayfasındaki blok, $X' = X + \mathrm{MHA}(\mathrm{LN}(X))$ ve $X'' = X' + \mathrm{MLP}(\mathrm{LN}(X'))$: `Block.forward` içindeki iki satır.

In [ ]:
class Attention(nn.Module):
    """Çok başlı öz dikkat; son çağrının ağırlıkları haritalar için saklanır."""
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)
        self.weights = None

    def forward(self, x):
        n, t, d = x.shape
        q, k, v = self.qkv(x).reshape(n, t, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        a = torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)
        self.weights = a.detach()
        return self.out((a @ v).transpose(1, 2).reshape(n, t, d))


class Block(nn.Module):
    """Bir Transformer bloğu: Dikkat ve iki katmanlı küçük bir ağ; her biri katman normalleştirmesinden sonra gelir ve bir artık bağlantısı taşır."""
    def __init__(self, dim, heads, hidden):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Linear(hidden, dim))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class SmallViT(nn.Module):
    """Küçük bir Vision Transformer: Token olarak 8 x 8 piksellik kareler, bir sınıf token'ı, öğrenilen konumlar ve dört blok."""
    def __init__(self, size=64, patch=8, dim=64, depth=4, heads=4, channels=1, outputs=1):
        super().__init__()
        self.embed = nn.Conv2d(channels, dim, patch, stride=patch)    # her kare için aynı doğrusal dönüşüm
        n = (size // patch) ** 2
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, n + 1, dim) * 0.02)
        self.blocks = nn.ModuleList([Block(dim, heads, 2 * dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, outputs)

    def encode(self, t):
        t = torch.cat([self.cls.expand(len(t), -1, -1), t], dim=1) + self.pos
        for block in self.blocks:
            t = block(t)
        out = self.head(self.norm(t)[:, 0])
        return out.squeeze(1) if out.shape[1] == 1 else out

    def forward(self, x):
        return self.encode(self.embed(x).flatten(2).transpose(1, 2))


def n_parameters(model):
    """Bir ağın eğitilebilir ağırlıklarının sayısı."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

torch.manual_seed(RANDOM_STATE)
mini_vit = SmallViT(size=32, patch=4, dim=64, depth=4, heads=4, channels=3, outputs=10)
parts = {"yama gömme": mini_vit.embed, "sınıf token'ı": [mini_vit.cls], "konumlar": [mini_vit.pos],
         "4 blok": mini_vit.blocks, "son normalleştirme": mini_vit.norm, "son katman": mini_vit.head}
for name, part in parts.items():
    weights = part if isinstance(part, list) else list(part.parameters())
    print(f"{name:24} {sum(w.numel() for w in weights):>8,}")
print(f"{'toplam':24} {n_parameters(mini_vit):>8,}")
x = torch.randn(2, 3, 32, 32)
tokens = mini_vit.embed(x).flatten(2).transpose(1, 2)
print("2 görüntü", tuple(x.shape), "→ token'lar", tuple(tokens.shape), "→ 10 sınıfın puanları", tuple(mini_vit(x).shape))

**Python notu.** `nn.Parameter` bir tensörü eğitimin değiştirdiği bir ağırlık olarak işaretler; burada sınıf token'ı ve konumlar böyledir. `nn.ModuleList` blokları, ağ ağırlıklarını bilsin diye bir arada tutar. `forward` içinde `self.embed(x)`, 8 × 8 karenin her biri için 64 sayı verir; `.flatten(2)` kareleri bir sıraya dizer, `.transpose(1, 2)` de her kareyi 64 sayılık bir token yapar. `encode`, `torch.cat` ile sınıf token'ını token'ların önüne koyar, konumları ekler, hepsini bloklardan geçirir ve 0. sıradaki sınıf token'ını okur.

**Hugging Face Hub'dan hazır bir Vision Transformer.** *Ders: [ViT PyTorch ile inşa; Hugging Face üzerinden hazır ViT modeli](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#building-a-vit-with-pytorch-a-ready-vit-model-from-hugging-face)*

Hugging Face Hub, ViT-B/16'yı Google'ın eğittiği biçimiyle "google/vit-base-patch16-224" adı altında saklar: ImageNet-21k ile önceden eğitilmiş ve ImageNet'in 1.000 sınıfıyla ince ayar yapılmıştır. Hücre ağın planını, görüntüleri hazırlama biçimini ve bir kez indirilen, yaklaşık 350 MB'lık ağırlıklarını yükler. Hub'a ulaşılamadığında hücre aynı planı rastgele ağırlıklarla kurar ve bunu bildirir: Bu durumda yapı ve boyutlar aynıdır, sayılar değildir.

**Formüllerden koda.** Plan: `ViTConfig.from_pretrained(HF_NAME)`. Ağırlıklarıyla birlikte ağ: `ViTForImageClassification.from_pretrained(HF_NAME)`. Her katmanın dikkati: `output_attentions=True`.

In [ ]:
from transformers import ViTConfig, ViTForImageClassification, ViTImageProcessor

HF_NAME = "google/vit-base-patch16-224"
hf_config = ViTConfig.from_pretrained(HF_NAME, attn_implementation="eager")   # küçük bir metin dosyası: Ağın planı
hf_processor = ViTImageProcessor.from_pretrained(HF_NAME)  # bir görüntünün ağ için nasıl hazırlandığı
try:
    hf_vit = ViTForImageClassification.from_pretrained(HF_NAME, attn_implementation="eager").eval()   # ağırlıklar: Yaklaşık 350 MB, bir kez indirilir
    hf_pretrained = True
except Exception:
    hf_vit = ViTForImageClassification(hf_config).eval()                 # aynı plan, rastgele ağırlıklarla
    hf_pretrained = False

print("Hub'dan gelen ağırlıklar:", hf_pretrained)
print("görüntü", hf_config.image_size, "· yama", hf_config.patch_size, "· katman", hf_config.num_hidden_layers,
      "· baş", hf_config.num_attention_heads, "· token başına sayı", hf_config.hidden_size,
      "· sınıf", len(hf_config.id2label))
print("ağırlık:", f"{sum(p.numel() for p in hf_vit.parameters()):,}", "· hazırlamanın ortalaması ve standart sapması:",
      hf_processor.image_mean, hf_processor.image_std)
inputs = hf_processor(images=np.repeat((small[..., None] * 255).astype(np.uint8), 3, axis=2), return_tensors="pt")
with torch.no_grad():
    out = hf_vit(**inputs, output_attentions=True)
print("girdi", tuple(inputs["pixel_values"].shape), "· puanlar", tuple(out.logits.shape),
      "· 12 katmanın dikkati", len(out.attentions), "×", tuple(out.attentions[0].shape))
if hf_pretrained:
    print("anjiyogram için en yüksek puanlı ImageNet sınıfı:", hf_config.id2label[int(out.logits.argmax())])

**Python notu.** `from_pretrained` bir modelin dosyalarını Hub'da adıyla arar, bir kez indirir ve bilgisayardaki bir klasörde saklar. `try` ve `except` bağlantı olmamasından doğan hatayı yakalar; hücre de `ViTForImageClassification(hf_config)` ile, yani rastgele ağırlıklı aynı ağla devam eder. Bu modelin görüntü hazırlayıcısı üç kanalı 0,5 ortalamaya ve 0,5 standart sapmaya ölçekler; torchvision ağı için `to_imagenet` fonksiyonunun kullandığı ImageNet değerlerini kullanmaz. `np.repeat`, gri anjiyograma üç eşit renk kanalı verir. `attn_implementation="eager"`, dikkatin düz hesaplanmasını ister; `output_attentions=True` ile istenen dikkat haritalarını bu hesaplama döndürebilir.

**Gözlemle ve düşün.** Yukarıdaki torchvision ağı ile Hub'daki ağ aynı plana ve aynı sayıda ağırlığa sahiptir, ama ağırlıkları farklıdır. İkisi nereden gelmektedir ve bir görüntünün hazırlanışı neden onu okuyacak ağa uymalıdır?

## 4. Ekokardiyografi için bir kalite kapısı

*Ders: [Ekokardiyografiye aktarım: Otomatik EF'den önce bir kalite kapısı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#transfer-to-echocardiography-a-quality-gate)*

CAMUS'un 300 hastasının 600 diyastol sonu karesi üç yolla özniteliğe dönüşür: Dört görüntü istatistiği, EfficientNet-B0'ın 1.280 özniteliği ve ViT-B/16'nın sınıf token'ındaki 768 sayı. Her biri üzerinde kurulan bir lojistik regresyon, uzmanların hangi kareleri kötü bulduğunu öğrenir. Düzenlileştirme doğrulama karelerinde seçilir; test AUC değeri, test hastalarından çekilen 2.000 bootstrap örneğiyle hesaplanan yüzde 95 aralığıyla birlikte verilir.

**Formüllerden koda.** Hedef, kötü bir kare için 1'dir: `poor`. Kapı: `fit_gate(features, poor)`. Aralık: Gruplar hastalar olacak biçimde `np.percentile(bootstrap_auc(...), [2.5, 97.5])`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
effnet = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1).eval()
effnet.classifier = nn.Identity()      # son katmanı olmadan ağ 1280 kanalının ortalamasını döndürür


@torch.no_grad()
def effnet_features(images, batch=16):
    """EfficientNet-B0'ın her görüntü için verdiği 1280 öznitelik (4. hafta)."""
    return torch.cat([effnet(to_imagenet(images[i:i + batch])) for i in range(0, len(images), batch)]).numpy()

camus_table = camus.table()
echo, echo_patient, echo_view, echo_quality, echo_split = [], [], [], [], []
for _, row in camus_table.iterrows():
    for view in ("2CH", "4CH"):
        img, _, _ = camus.load(row["patient"], view, "ED")
        echo.append(np.asarray(Image.fromarray(img).resize((224, 224), Image.BILINEAR)) / 255)
        echo_patient.append(row["patient"])
        echo_view.append(view)
        echo_quality.append(row["quality_" + view.lower()])
        echo_split.append(row["split"])
echo = np.clip(np.stack(echo), 0, 1).astype(np.float32)
echo_patient, echo_view, echo_quality, echo_split = map(np.array, (echo_patient, echo_view, echo_quality, echo_split))
etr, eva, ete = echo_split == "training", echo_split == "validation", echo_split == "testing"
poor = (echo_quality == "Poor").astype(int)


def image_statistics(images):
    """Görüntü başına dört sayı: Ortalama gri değer, standart sapma, komşu pikseller arasındaki ortalama fark ve en büyük değerin yarısından parlak piksellerin oranı."""
    return np.array([[x.mean(), x.std(), np.abs(np.diff(x, axis=0)).mean(), (x > 0.5 * x.max()).mean()] for x in images])


def fit_gate(features, target):
    """Standartlaştırılmış özniteliklerle bir lojistik regresyon; ters düzenlileştirme gücü doğrulama karelerinde seçilir."""
    best = None
    for C in (0.001, 0.01, 0.1, 1.0):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(features[etr], target[etr])
        a = roc_auc_score(target[eva], model.predict_proba(features[eva])[:, 1])
        if best is None or a > best[0]:
            best = (a, C, model)
    return best[2], best[1], best[0]


def bootstrap_auc(target, prob, groups, n=2000, seed=2026):
    """Grupları (hastaları ya da görüntüleri) yerine koyarak çeken 2000 bootstrap örneğinde AUC."""
    rng = np.random.default_rng(seed)
    names = np.unique(groups)
    rows = {g: np.where(groups == g)[0] for g in names}
    out = []
    for _ in range(n):
        idx = np.concatenate([rows[g] for g in rng.choice(names, len(names))])
        if 0 < target[idx].sum() < len(idx):
            out.append(roc_auc_score(target[idx], prob[idx]))
    return np.array(out)

print(len(echo), "kare")
print(pd.crosstab(echo_split, echo_quality))
features = {"istatistikler": image_statistics(echo), "EfficientNet-B0": effnet_features(echo), "ViT-B/16": vit_features(echo)}
for name, X in features.items():
    print(name, X.shape)

**Python notu.** `camus.table()` 300 hastanın Info dosyalarını okur; döngü iki görüntünün diyastol sonu karesini `camus.load` ile yükler, 224 × 224 piksele küçültür ve 255'e böler. `np.stack` 600 kareyi 600 × 224 × 224 boyutlu tek bir diziye koyar; `map(np.array, ...)` dört listeyi tek seferde diziye dönüştürür. `pd.crosstab` her kümede her kalitedeki kareleri sayar. En uzun süren ViT-B/16 öznitelikleridir; sıradan bir dizüstü bilgisayarda bir ile iki dakika sürer.

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4))
probs = {}
for name, X in features.items():
    model, C_, va = fit_gate(X, poor)
    probs[name] = model.predict_proba(X[ete])[:, 1]
    boot = bootstrap_auc(poor[ete], probs[name], echo_patient[ete])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    print(f"{name:16} C={C_:<6} doğrulama AUC {va:.3f}  test AUC {roc_auc_score(poor[ete], probs[name]):.3f} ({lo:.3f}-{hi:.3f})")
    fpr, tpr, _ = roc_curve(poor[ete], probs[name])
    ax.plot(fpr, tpr, label=name)
ax.plot([0, 1], [0, 1], ":", color="grey")
ax.set_xlabel("1 − özgüllük")
ax.set_ylabel("duyarlılık")
ax.legend()
plt.show()

**Python notu.** `fit_gate` dört ters düzenlileştirme gücünü dener, her biriyle eğitim karelerinin standartlaştırılmış öznitelikleri üzerinde bir lojistik regresyon kurar ve doğrulama karelerinde AUC değeri en büyük olanı tutar. `make_pipeline(StandardScaler(), LogisticRegression(...))` standartlaştırmayı ve modeli bir arada tutar; böylece test kareleri eğitim karelerinin ortalamalarıyla ölçeklenir. `bootstrap_auc` 50 test hastasını yerine koyarak 2.000 kez çeker ve çekilen her hastanın iki karesini de tutar; `np.percentile` 2.000 AUC değerinin 2,5. ve 97,5. yüzdeliklerini okur.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
names = np.unique(echo_patient[ete])
rows_of = {p: np.where(echo_patient[ete] == p)[0] for p in names}
y_te, pv, pe = poor[ete], probs["ViT-B/16"], probs["EfficientNet-B0"]
diff = []
for _ in range(2000):
    idx = np.concatenate([rows_of[p] for p in rng.choice(names, len(names))])
    if 0 < y_te[idx].sum() < len(idx):
        diff.append(roc_auc_score(y_te[idx], pv[idx]) - roc_auc_score(y_te[idx], pe[idx]))
diff = np.array(diff)
print("ViT-B/16 eksi EfficientNet-B0:", round(diff.mean(), 3), np.round(np.percentile(diff, [2.5, 97.5]), 3),
      "| 0'ın üstündeki örneklerin oranı:", round(float((diff > 0).mean()), 3))

**Gözlemle ve düşün.** Farkın aralığı 0'ı içeriyor. İki ağdan birine daha iyi kapı diyebilmek için kaç test hastasına ya da kaç kötü kareye ihtiyacınız olurdu?

**Alıştırma D.** ViT-B/16'nın sınıf token'ından iyi kareleri ötekilerden ayıran kapıyı `good = (echo_quality == "Good").astype(int)` ile kurun. Test AUC değeri kaçtır? Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("D")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
good = (echo_quality == "Good").astype(int)
model, _, _ = fit_gate(features["ViT-B/16"], good)
cevap = round(float(roc_auc_score(good[ete], model.predict_proba(features["ViT-B/16"][ete])[:, 1])), 3)
teach.check("Alıştırma D", cevap)

**Mini proje: CIFAR-10 üzerinde mini ViT, ardından EchoNet'e transfer.** *Ders: [Mini proje: CIFAR-10 üzerinde mini ViT eğitimi; ardından EchoNet'e transfer uygulaması](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#mini-project-training-a-mini-vit-on-cifar-10-then-transfer-to-echonet)*

3. bölümdeki küçük Vision Transformer önce gündelik fotoğraflardan oluşan on sınıfı, yani CIFAR-10'u öğrenir; ardından bir ekokardiyogramın iki karesinden düşük ejeksiyon fraksiyonunu tanımayı öğrenir. EchoNet-Dynamic için kişisel kaydı olan katılımcılar, 3. haftanın haftalık projesinde olduğu gibi onun videolarını kullanır; diğer katılımcılar CAMUS ekokardiyogramlarını kullanır. Karşılaştırma, yalnızca ekokardiyogramlarla ve rastgele ağırlıklarla eğitilen aynı ağdır.

İlk hücre CIFAR-10'u Toronto Üniversitesinden bir kez indirir (yaklaşık 170 MB). Eğitim sıradan bir bilgisayarda epoch başına yaklaşık bir dakika sürer; Colab'da ekran kartı gerekmez.

**Formüllerden koda.** On sınıf: 3. haftanın çapraz entropisi, `nn.CrossEntropyLoss()`. Yeni görev: `copy.deepcopy(mini_vit)` ve yeni bir son katman, `nn.Linear(64, 1)`. Görüntü olarak bir kare çifti: `three_channels`; diyastol sonu kare, sistol sonu kare ve farkları.

In [ ]:
from torchvision.datasets import CIFAR10
from vclinic import datasets, echonet

CIFAR_CLASSES = ["uçak", "otomobil", "kuş", "kedi", "geyik",
                 "köpek", "kurbağa", "at", "gemi", "kamyon"]


def cifar10():
    """Toronto Üniversitesindeki kaynağından CIFAR-10; yaklaşık 170 MB, data/cifar10 klasörüne bir kez indirilir: Üç renk kanallı, 32 x 32 piksellik 50.000 eğitim ve 10.000 test görüntüsü ile sınıfları."""
    folder = str(datasets.data_path("cifar10"))
    out = {}
    for part in ("training", "test"):
        d = CIFAR10(folder, train=(part == "training"), download=True)
        out[part] = (d.data, np.array(d.targets))
    return out


def to_tensor(images, mean, std):
    """n x 32 x 32 x 3 boyutlu görüntüler (0-255 değerleri) → n x 3 x 32 x 32 boyutlu tensörler; her kanal eğitim görüntülerinin ortalaması ve standart sapmasıyla standartlaştırılır."""
    x = torch.as_tensor(np.asarray(images), dtype=torch.float32).permute(0, 3, 1, 2) / 255
    return (x - torch.as_tensor(mean, dtype=torch.float32).view(1, 3, 1, 1)) / torch.as_tensor(std, dtype=torch.float32).view(1, 3, 1, 1)


def shift_and_mirror(x):
    """Veri artırma: Grup her yönde 4 piksele kadar kaydırılır ve yarı yarıya soldan sağa aynalanır."""
    i, j = (int(v) for v in torch.randint(0, 9, (2,)))
    x = F.pad(x, (4, 4, 4, 4), mode="reflect")[:, :, i:i + 32, j:j + 32]
    return torch.flip(x, dims=(3,)) if torch.rand(1) < 0.5 else x


def train_cifar(model, x, y, epochs=5, batch=128, seed=2026):
    """Karıştırılmış 128 görüntülük gruplar üzerinde 0,001 öğrenme hızı ve 0,05 ağırlık azaltmasıyla AdamW; kayıp on sınıfın çapraz entropisidir."""
    torch.manual_seed(seed)
    optimiser = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.05)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        model.train()
        order, total = torch.randperm(len(x)), 0.0
        for i in range(0, len(x), batch):
            b = order[i:i + batch]
            loss = loss_fn(model(shift_and_mirror(x[b])), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total += loss.item() * len(b)
        print(f"epoch {epoch + 1}: eğitim kaybı {total / len(x):.3f}")


@torch.no_grad()
def accuracy(model, x, y, batch=500):
    """Sınıfı en yüksek puanı alan görüntülerin oranı."""
    model.eval()
    return float(torch.cat([model(x[i:i + batch]).argmax(1) for i in range(0, len(x), batch)]).eq(y).float().mean())


def camus_pairs():
    """CAMUS'un 300 hastasının iki görüntüsünün diyastol sonu ve sistol sonu kareleri, 32 x 32 piksellik çiftler olarak ve hastanın EF değeriyle."""
    rows = []
    for _, r in camus.table().iterrows():
        for view in ("2CH", "4CH"):
            pair = [np.asarray(Image.fromarray(camus.load(r["patient"], view, phase)[0]).resize((32, 32), Image.BILINEAR)) / 255 for phase in ("ED", "ES")]
            rows.append({"küme": {"training": "TRAIN", "validation": "VAL", "testing": "TEST"}[r["split"]], "hasta": r["patient"],
                         "çift": np.stack(pair).astype(np.float32), "EF": r["ef"]})
    return pd.DataFrame(rows)


def echonet_pairs(source, n=(2000, 400, 400)):
    """EchoNet-Dynamic videolarının çizilmiş iki karesi, 32 x 32 piksellik çiftler olarak ve EF değeriyle: 2.000 eğitim, 400 doğrulama ve 400 test videosu."""
    parts = []
    for split, k in zip(("TRAIN", "VAL", "TEST"), n):
        d = echonet.frame_pairs(source, split=split, n=k, size=32)
        parts.append(pd.DataFrame({"küme": split, "hasta": d["file"], "çift": list(d["images"]), "EF": d["ef"]}))
    return pd.concat(parts, ignore_index=True)


def three_channels(pairs):
    """Bir kare çifti → üç kanal: Diyastol sonu kare, sistol sonu kare ve farkları; bir fotoğrafın kırmızı, yeşil ve mavi kanallarının yerinde."""
    p = np.stack(pairs)
    return np.stack([p[:, 0], p[:, 1], p[:, 1] - p[:, 0]], axis=1).astype(np.float32)


def fine_tune(model, x, y, xv, yv, epochs=30, lr=0.0005, seed=2026):
    """Karıştırılmış 32 çiftlik gruplar üzerinde Adam; kayıp, düşük EF için ikili çapraz entropidir. Doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
    torch.manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x))
        for i in range(0, len(x), 32):
            b = order[i:i + 32]
            loss = loss_fn(model(x[b]), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            a = roc_auc_score(yv.numpy(), torch.sigmoid(model(xv)).numpy())
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
    model.load_state_dict(state)
    return best

try:
    cifar = cifar10()
except Exception as error:
    cifar = None
    print("CIFAR-10 burada indirilemedi:", type(error).__name__,
          "Ona ihtiyaç duyan mini proje hücreleri bunu bildirir ve geri kalanla devam eder.")
if cifar is not None:
    images, labels = cifar["training"]
    print("eğitim", images.shape, "· test", cifar["test"][0].shape, "· sınıf başına görüntü", np.bincount(labels))
    fig, ax = plt.subplots(1, 10, figsize=(11, 1.6))
    for c, a in enumerate(ax):
        a.imshow(images[np.where(labels == c)[0][0]])
        a.set_title(CIFAR_CLASSES[c], fontsize=8)
        a.axis("off")
    plt.show()

In [ ]:
if cifar is None:
    print("CIFAR-10 olmadığı için mini ViT burada fotoğraflarla eğitilmez.")
else:
    mean = cifar["training"][0].mean(axis=(0, 1, 2)) / 255
    std = cifar["training"][0].std(axis=(0, 1, 2)) / 255
    x_train, y_train = to_tensor(cifar["training"][0], mean, std), torch.as_tensor(cifar["training"][1])
    x_test, y_test = to_tensor(cifar["test"][0], mean, std), torch.as_tensor(cifar["test"][1])
    torch.manual_seed(RANDOM_STATE)
    mini_vit = SmallViT(size=32, patch=4, dim=64, depth=4, heads=4, channels=3, outputs=10)
    train_cifar(mini_vit, x_train, y_train, epochs=5)
    print("10.000 test görüntüsünde doğruluk:", round(accuracy(mini_vit, x_test, y_test), 3))

**Python notu.** `to_tensor` fotoğrafları PyTorch'un düzenine, yani kanal önde olacak biçime getirir ve her renk kanalını eğitim fotoğraflarıyla standartlaştırır. `shift_and_mirror` bir fotoğraf grubunu dört piksele kadar kaydırır ve yarı yarıya aynalar; böylece ağ her epoch'ta biraz farklı bir resim görür. `accuracy`, en yüksek puanı alan sınıfın, yani `argmax(1)` değerinin ne sıklıkla doğru olduğunu sayar; şans eseri her on sınıftan biri doğru olurdu.

In [ ]:
try:
    echo_source = echonet.find()
except FileNotFoundError:
    echo_source = None
pairs = echonet_pairs(echo_source) if echo_source is not None else camus_pairs()
print("EchoNet-Dynamic" if echo_source is not None else "CAMUS", "·", len(pairs), "çift")
X = {s: torch.as_tensor(three_channels(pairs[pairs["küme"] == s]["çift"])) for s in ("TRAIN", "VAL", "TEST")}
Y = {s: torch.as_tensor((pairs[pairs["küme"] == s]["EF"] < 50).to_numpy(np.float32)) for s in ("TRAIN", "VAL", "TEST")}
for s in X:
    print(s, tuple(X[s].shape), "düşük EF:", int(Y[s].sum()))
starts = {"rastgele ağırlıklar": SmallViT(size=32, patch=4, dim=64, depth=4, heads=4, channels=3)}
if "mini_vit" in globals() and cifar is not None:
    pretrained = copy.deepcopy(mini_vit)
    pretrained.head = nn.Linear(64, 1)
    starts["CIFAR-10"] = pretrained
else:
    print("CIFAR-10 başlangıcı yukarıdaki hücrede eğitilen mini ViT'e ihtiyaç duyar.")
for name, net in starts.items():
    torch.manual_seed(RANDOM_STATE)
    best = fine_tune(net, X["TRAIN"], Y["TRAIN"], X["VAL"], Y["VAL"])
    with torch.no_grad():
        p_test = torch.sigmoid(net(X["TEST"])).numpy()
    print(f"{name:20} doğrulama AUC {best:.3f} · test AUC {roc_auc_score(Y['TEST'].numpy(), p_test):.3f}")

**Python notu.** `echonet.find()`, katılımcının kendi EchoNet-Dynamic kopyasını data/echonet klasöründe arar; kopya yoksa `FileNotFoundError` hatası verir ve hücre bu durumda CAMUS'u kullanır. `pairs`, her kare çifti için bir satırı olan bir tablodur. `X` ve `Y` sözlükleri üç kümeyi EchoNet-Dynamic'in adlarıyla, "TRAIN", "VAL" ve "TEST" olarak tutar. `copy.deepcopy`, eğitilmiş ağın bağımsız bir kopyasını oluşturur; böylece son katmanını değiştirmek `mini_vit` ağını değiştirmez. `fine_tune` doğrulama AUC değeri en iyi olan epoch'un ağırlıklarını saklar.

**Gözlemle ve düşün.** İki test AUC değerini karşılaştırın. On sınıflık fotoğraflar ağın ekokardiyogramları okumasına yardım etti mi? CAMUS'un 400 karesi yerine EchoNet-Dynamic'in 2.000 videosuyla cevabın değişmesini bekler miydiniz?

**1. uygulama durağının Python yolu tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#last-two-hours-hybrid-cnn-vit-and-medical-imaging) ve okumaya «Son 2 saat: Hibrit CNN-ViT ve medikal görüntüleme» bölümünden devam edin.

## 5. Sentetik anjiyogramlarda darlık yamaları

*Ders: [Sentetik ve gerçek anjiyogramlar: Başka bir dünyada bir model](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#synthetic-and-real-angiograms-a-model-in-another-world)*

300 eğitim, 100 doğrulama ve 100 test hastasının anjiyogramlarından 64 × 64 piksellik yamalar kesilir: Her anlamlı darlığın çevresinden, rastgele en fazla 6 piksel kaydırılmış iki yama ve her biri için her darlıktan uzaktaki damarlardan üç yama. Küçük bir evrişimli ağ bu yamaları sekiz epoch boyunca öğrenir.

**Formüllerden koda.** Her yama kendi içinde standartlaştırılır: $(x - \bar{x})/(s + 0.02)$, `standardise`. Kayıp, logitin ikili çapraz entropisidir (`nn.BCEWithLogitsLoss()`); `train`, doğrulama AUC değeri en iyi olan epoch'un ağırlıklarını saklar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
def standardise(X):
    """Her yama kendi içinde: Ortalama gri değer 0, standart sapma 1."""
    X = np.asarray(X, dtype=np.float32)
    m, s = X.mean(axis=(1, 2), keepdims=True), X.std(axis=(1, 2), keepdims=True)
    return (X - m) / (s + 0.02)

syn = {}
for s, n in (("training", 300), ("validation", 100), ("test", 100)):
    X, y_, ids = angiogram.patches(s, n_patients=n)
    syn[s] = (standardise(X), y_, ids)
    print(s, X.shape, "darlık üzerinde:", int(y_.sum()))
X_raw, y3, _ = angiogram.patches("test", n_patients=3)
fig, ax = plt.subplots(1, 6, figsize=(10, 1.9))
for a, i in zip(ax, list(np.where(y3 == 1)[0][:3]) + list(np.where(y3 == 0)[0][:3])):
    a.imshow(X_raw[i], cmap="gray")
    a.set_title("darlık" if y3[i] else "damar", fontsize=9)
    a.axis("off")
plt.show()

**Python notu.** `angiogram.patches` bir kümenin ilk `n_patients` hastasının anjiyogramlarını çizer ve yamaları keser; yamaları, etiketlerini ve her yamanın hastasını döndürür. 1.000 anjiyogram çizmek yaklaşık iki dakika sürer. `syn`, değerleri üçlüler olan bir sözlüktür; `standardise` her yamayı kendi içinde ortalaması 0 ve standart sapması 1 olacak biçimde ölçekler; `axis=(1, 2)` ve `keepdims=True` her yama için bir ortalama verir.

In [ ]:
def conv_block(a, b):
    """3 x 3 evrişim, batch normalizasyonu, ReLU ve 2 x 2 en büyük havuzlama: Boyut yarıya iner, b kanal elde edilir."""
    return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.BatchNorm2d(b), nn.ReLU(), nn.MaxPool2d(2))


class SmallCNN(nn.Module):
    """Dört evrişim bloğu (16, 32, 64 ve 64 kanal), her kanalın ortalaması ve tek bir çıktı."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, 64), conv_block(64, 64))
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3))).squeeze(1)



class Hybrid(SmallViT):
    """Hibrit model: Üç evrişim bloğu yamayı 64 sayılık 8 x 8 konuma dönüştürür; bunlar iki Transformer bloğunun token'larıdır."""
    def __init__(self, dim=64, depth=2, heads=4):
        super().__init__(size=8, patch=1, dim=dim, depth=depth, heads=heads, channels=dim)
        self.embed = nn.Sequential(conv_block(1, 16), conv_block(16, 32), conv_block(32, dim))



@torch.no_grad()
def predict(model, X, batch=256):
    """Her yama için darlık olasılığı."""
    model.eval()
    X = torch.as_tensor(X, dtype=torch.float32)
    return torch.cat([torch.sigmoid(model(X[i:i + batch, None])) for i in range(0, len(X), batch)]).numpy()


def train(model, X, y, Xv, yv, epochs=8, seed=2026, verbose=True):
    """Adam, 0,001 öğrenme hızı, rastgele bir çeyrek tur döndürülen ve yarı yarıya aynalanan 64 yamalık gruplar; doğrulama AUC değeri en iyi olan epoch'un ağırlıkları saklanır."""
    torch.manual_seed(seed)
    X, y = torch.as_tensor(X, dtype=torch.float32)[:, None], torch.as_tensor(y, dtype=torch.float32)
    optimiser = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.BCEWithLogitsLoss()
    best, state = -1.0, None
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            b = order[i:i + 64]
            xb = torch.rot90(X[b], int(torch.randint(0, 4, (1,))), dims=(2, 3))
            if torch.rand(1) < 0.5:
                xb = torch.flip(xb, dims=(3,))
            loss = loss_fn(model(xb), y[b])
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
        a = roc_auc_score(yv, predict(model, Xv))
        if a > best:
            best, state = a, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {epoch + 1}: doğrulama AUC {a:.3f}")
    model.load_state_dict(state)
    return best

torch.manual_seed(RANDOM_STATE)
cnn_synthetic = SmallCNN()
print("eğitilebilir ağırlık:", n_parameters(cnn_synthetic))
train(cnn_synthetic, syn["training"][0], syn["training"][1], syn["validation"][0], syn["validation"][1], epochs=8)
p_syn = predict(cnn_synthetic, syn["test"][0])
print("test AUC, sentetik yamalar:", round(roc_auc_score(syn["test"][1], p_syn), 3))

**Python notu.** Bir ağ bir *sınıftır* (class): `class SmallCNN(nn.Module)`, parçalarını `__init__` içinde, bu parçalardan geçen yolu ise `forward` içinde tarif eder; `SmallCNN()` yeni rastgele ağırlıklarla bir ağ kurar; hemen önündeki `torch.manual_seed(RANDOM_STATE)` bu ağırlıkların her çalıştırmada aynı olmasını sağlar. `conv_block` bir bloğun dört katmanını tek bir `nn.Sequential` olarak döndürür. `train`, 3. haftanın döngüsüdür: Karıştırılmış 64 yamalık gruplar, `torch.rot90` ile rastgele bir çeyrek tur, yarı olasılıkla `torch.flip` ile bir aynalama, kayıp, `backward` ve Adam'ın bir adımı. Her epoch'tan sonra doğrulama AUC değeri karar verir ve `copy.deepcopy` en iyi ağırlıkları saklar. `predict` her yama için darlık olasılığını, yani logitin sigmoidini döndürür.

**Alıştırma E.** Sentetik eğitim yamalarının ne kadarı anlamlı bir darlığın üzerindedir? Oranı üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("E")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(syn["training"][1].mean()), 3)
teach.check("Alıştırma E", cevap)

## 6. ARCADE: Gerçek anjiyogramlar

*Ders: [Sentetik ve gerçek anjiyogramlar: Başka bir dünyada bir model](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#synthetic-and-real-angiograms-a-model-in-another-world)*

ARCADE'in darlık kümesi, darlık oluşturan plakların bölgelerinin uzmanlar tarafından çizildiği 512 × 512 piksellik 1.500 X ışınlı koroner anjiyogram içerir. Yamalar sentetik anjiyogramlardaki gibi kesilir; negatif yamalar, sınırı çizilmiş her darlıktan ve X ışını alanının koyu çerçevesinin bulunduğu görüntü kenarından en az 48 piksel uzaktaki, damara benzeyen piksellerin üzerindedir. 5. bölümdeki ağ bu yamalarda hiçbir değişiklik yapılmadan sınanır; ardından aynı ağ ARCADE ile eğitilir.

**Formüllerden koda.** Bir darlığın merkezi kutusunun merkezidir: `arcade.annotations(split)`. Damar pikselleri, ince ve koyu yapıları işaretleyen bir filtre olan siyah şapka dönüşümünün (black top-hat) en büyük değerlerini veren piksellerdir: `arcade.vessel_points`. AUC aralığı anjiyogramları çeker: `bootstrap_auc(y, p, ids)`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
arcade.download()
arc = {s: arcade.patches(s) for s in arcade.SPLITS}

for s in arcade.SPLITS:
    ann = arcade.annotations(s)
    print(s, len(ann), "anjiyogram,", sum(len(a["stenoses"]) for a in ann), "darlık,", len(arc[s][1]), "yama")
first = arcade.annotations("test")[0]
img = arcade.load("test", first["file"])
plt.figure(figsize=(4, 4))
plt.imshow(img, cmap="gray")
for s in first["stenoses"]:
    bx, by, bw, bh = s["box"]
    plt.gca().add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, color="#9B1B1B", lw=1.5))
plt.title("ARCADE, test anjiyogramı " + str(first["id"]))
plt.axis("off")
plt.show()

**Python notu.** `arcade.download()` ARCADE arşivini Zenodo'dan bir kez indirir, yayımlanmış sağlama toplamıyla denetler ve `data/arcade` klasöründe yalnızca darlık kümesini tutar. `arcade.patches(split)` bir kümenin bütün anjiyogramlarının yamalarını keser; yamaları, etiketlerini ve her yamanın anjiyogram numarasını döndürür. `arc`, `train`, `val` ve `test` için bu üçlüleri tutan bir sözlüktür. `arcade.annotations` her anjiyogram için dosyasını ve darlıklarını, her darlık için de bir kutuyu (x, y, genişlik, yükseklik) içeren bir sözlük döndürür. Görüntü: ARCADE, Popov ve ark. 2024, CC0.

In [ ]:
ARC = {s: (standardise(X), y_, ids) for s, (X, y_, ids) in arc.items()}
p_shift = predict(cnn_synthetic, ARC["test"][0])
lo, hi = np.percentile(bootstrap_auc(ARC["test"][1], p_shift, ARC["test"][2]), [2.5, 97.5])
print(f"5. bölümdeki ağ ARCADE'de: AUC {roc_auc_score(ARC['test'][1], p_shift):.3f} ({lo:.3f}-{hi:.3f})")
torch.manual_seed(RANDOM_STATE)
cnn_arcade = SmallCNN()
train(cnn_arcade, ARC["train"][0], ARC["train"][1], ARC["val"][0], ARC["val"][1], epochs=8)
p_cnn = predict(cnn_arcade, ARC["test"][0])
lo, hi = np.percentile(bootstrap_auc(ARC["test"][1], p_cnn, ARC["test"][2]), [2.5, 97.5])
print(f"ARCADE ile eğitilen: AUC {roc_auc_score(ARC['test'][1], p_cnn):.3f} ({lo:.3f}-{hi:.3f})")

**Gözlemle ve düşün.** Yalnızca sentetik anjiyogramlardan öğrenen ağ ARCADE'de yaklaşık 0,54 AUC değerine düşer. Sentetik bir yamayı ARCADE'den bir yamayla karşılaştırın: Ağ, sentetik görüntülerde gerçek görüntülerin sunmadığı hangi farkları kullanmış olabilir?

**İleri düzey (isteğe bağlı).** Gerçek görüntüler az olduğunda sentetik ağ yardımcı olabilir mi? Ağa ARCADE'in eğitim anjiyogramlarının onda biriyle ince ayar yapın ve rastgele ağırlıklarla başlayan aynı ağla karşılaştırın: `tenth = np.isin(ARC["train"][2], np.random.default_rng(RANDOM_STATE).choice(np.unique(ARC["train"][2]), 100, replace=False))`, ardından `train(copy.deepcopy(cnn_synthetic), ARC["train"][0][tenth], ARC["train"][1][tenth], ARC["val"][0], ARC["val"][1])`. Dersin çalıştırmasında iki ağın test AUC değerleri 0,71 ve 0,67 olarak bulunmuştur.

**Alıştırma F.** Uzmanlar ARCADE'in bir eğitim anjiyogramında ortalama kaç darlığın sınırını çizmiştir? Ortalamayı üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("F")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(np.mean([len(a["stenoses"]) for a in arcade.annotations("train")])), 3)
teach.check("Alıştırma F", cevap)

**2. uygulama durağının Python yolu tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#a-convolutional-network-a-vision-transformer-and-a-hybrid-compared) ve okumaya «Evrişimli ağ, Vision Transformer ve hibrit model karşılaştırması» bölümünden devam edin.

## 7. Küçük bir Vision Transformer ve hibrit bir model

*Ders: [Evrişimli ağ, Vision Transformer ve hibrit model karşılaştırması](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#a-convolutional-network-a-vision-transformer-and-a-hybrid-compared)*

Küçük Vision Transformer ve hibrit model ARCADE yamalarını 6. bölümdeki evrişimli ağla tamamen aynı biçimde öğrenir. Üç ağ aynı test yamalarında, test anjiyogramlarından yapılan bir bootstrap ile hesaplanan aralıklarla ve doğrulama yamalarında 0,90 duyarlılık veren eşikte karşılaştırılır.

**Formüllerden koda.** Vision Transformer'ın token'ları: `self.embed(x).flatten(2).transpose(1, 2)`, yani 64 sayılık 64 token. $\mathrm{softmax}(QK^{\top}/\sqrt{d_k})$, `Attention` sınıfındaki `torch.softmax(q @ k.transpose(-2, -1) / (d // self.heads) ** 0.5, dim=-1)` ifadesidir. 0,90 duyarlılık için eşik: Darlık içeren doğrulama yamalarının yüzde 90 kadarının ulaştığı olasılık.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

**Swin Transformer: Kaydırılmış pencerelerde dikkat.** *Ders: [Swin Transformer: Shifted window attention](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#swin-transformer-shifted-window-attention)*

Swin Transformer dikkati yalnızca M × M token'lık pencerelerin içinde hesaplar ve her ikinci katmanda pencereleri yarım pencere kaydırır; böylece bilgi pencerelerin sınırlarını aşar. Hücre, bir dikkat katmanının çarpma sayısını global olarak ve pencerelerde, Swin Transformer makalesinin formülleriyle sayar. Ardından 8 × 8 token'lık bir ızgarada 4 × 4 boyutlu pencerelerle, kaydırma olduğunda ve olmadığında, bilginin bir köşeden ne kadar uzağa yayıldığını izler.

**Formüllerden koda.** $\Omega(\mathrm{MSA}) = 4hwC^2 + 2(hw)^2C$ ve $\Omega(\mathrm{W\text{-}MSA}) = 4hwC^2 + 2M^2hwC$: `attention_cost(h, w, C)` ve `attention_cost(h, w, C, M)`. Her token'ın penceresi: `windows(grid, size, shifted)`.

In [ ]:
def windows(grid, size, shifted=False):
    """grid x grid token'lık bir ızgarada her token'ın penceresi: size x size token'lık pencereler. Kaydırılmış pencereler yarım pencere ileriden başlar; bu yüzden kenardaki pencereler daha küçüktür."""
    s = size // 2 if shifted else 0
    rows, cols = np.indices((grid, grid))
    n = grid // size + (1 if shifted else 0)
    return ((rows + s) // size) * n + (cols + s) // size


def reach(grid, size, layers, start=(0, 0), shift=True):
    """Her pencere dikkati katmanından sonra, başlangıçtaki token'a kaç token'ın bilgi aktarmış olabileceği; kaydırılmış pencereler her ikinci katmanda kullanılır ya da hiç kullanılmaz."""
    seen = np.zeros((grid, grid), bool)
    seen[start] = True
    counts = []
    for layer in range(layers):
        part = windows(grid, size, shifted=shift and layer % 2 == 1)
        seen = np.isin(part, np.unique(part[seen]))
        counts.append(int(seen.sum()))
    return counts


def attention_cost(h, w, C, M=None):
    """Swin Transformer makalesindeki gibi bir dikkat katmanının çarpma sayısı: Global dikkatte 4hwC² + 2(hw)²C, M x M token'lık pencerelerde 4hwC² + 2M²hwC."""
    return 4 * h * w * C ** 2 + (2 * (h * w) ** 2 * C if M is None else 2 * M ** 2 * h * w * C)

for name, side in (("224 x 224 görüntü, 4 x 4 yamalar", 56), ("512 x 512 anjiyogram, 4 x 4 yamalar", 128)):
    g, w = attention_cost(side, side, 96), attention_cost(side, side, 96, M=7)
    print(f"{name}: {side * side:,} token · global {g / 1e9:.2f} · 7 x 7 pencereler {w / 1e9:.3f} milyar çarpma · {g / w:.1f}x")
print("her katmandan sonra ulaşılan token sayısı, kaydırılmış pencerelerle:", reach(8, 4, 4))
print("kaydırma olmadan:", reach(8, 4, 4, shift=False))
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
for a, shifted, title in zip(ax, (False, True), ("1. katman: pencereler", "2. katman: kaydırılmış pencereler")):
    a.imshow(windows(8, 4, shifted), cmap="tab20")
    a.set_title(title, fontsize=9)
    a.set_xticks([])
    a.set_yticks([])
plt.show()

**Python notu.** `np.indices((grid, grid))` her token'ın satırını ve sütununu verir. Pencere boyutuna tam sayı bölmesi `//` her token'ın penceresini verir; kaydırma olduğunda sayım yarım pencere önce başlar, bu yüzden kenardaki pencereler, Swin Transformer'daki maskeleme sonucunda olduğu gibi daha küçüktür. `reach`, katman katman, o ana kadar ulaşılan herhangi bir token'la aynı pencerede bulunan token'ları işaretler. Kaydırma olmadığında köşedeki token kendi penceresinin ötesini hiç görmez.

In [ ]:
import time
networks = {"CNN": cnn_arcade}
seconds = {}
for name, M in (("ViT", SmallViT), ("Hybrid", Hybrid)):
    torch.manual_seed(RANDOM_STATE)
    net = M()
    t0 = time.time()
    train(net, ARC["train"][0], ARC["train"][1], ARC["val"][0], ARC["val"][1], epochs=8)
    seconds[name] = round(time.time() - t0)
    networks[name] = net
vit_arcade, hybrid_arcade = networks["ViT"], networks["Hybrid"]

**Python notu.** `SmallViT`, ders sayfasındaki parçaları koda döker. 8 × 8 filtreli ve 8 adımlı bir evrişim olan `self.embed`, 8 × 8 piksellik her kareye aynı doğrusal dönüşümü uygular; `flatten(2).transpose(1, 2)` bu çıktıyı 64 sayılık 64 token'a dönüştürür. `encode`, `torch.cat` ile sınıf token'ı `self.cls`'yi token'ların önüne koyar, `self.pos` konumlarını ekler ve token'ları bloklardan geçirir; son bloğun sınıf token'ı `self.norm` ve `self.head` katmanlarından geçer. `Hybrid` bunların hepsini `SmallViT` sınıfından devralır ve yalnızca `self.embed` katmanını üç evrişim bloğuyla değiştirir; bu blokların 8 × 8 konumluk çıktısı token'ları verir.

In [ ]:
yt, yv = ARC["test"][1], ARC["val"][1]
probs_te = {k: predict(m, ARC["test"][0]) for k, m in networks.items()}
probs_va = {k: predict(m, ARC["val"][0]) for k, m in networks.items()}
rows, thresholds = [], {}
for k in networks:
    p, pv = probs_te[k], probs_va[k]
    thresholds[k] = float(np.sort(pv[yv == 1])[int(np.floor(0.1 * yv.sum()))])
    hit = p >= thresholds[k]
    lo, hi = np.percentile(bootstrap_auc(yt, p, ARC["test"][2]), [2.5, 97.5])
    rows.append({"ağ": k, "ağırlık": n_parameters(networks[k]), "AUC": round(roc_auc_score(yt, p), 3), "95%": f"{lo:.3f}-{hi:.3f}",
                 "duyarlılık": round(hit[yt == 1].mean(), 3), "özgüllük": round(1 - hit[yt == 0].mean(), 3),
                 "PPV": round((hit & (yt == 1)).sum() / hit.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print("eğitim süresi (sn):", seconds)

**Python notu.** `np.sort(pv[yv == 1])`, darlık içeren doğrulama yamalarının olasılıklarını küçükten büyüğe sıralar; yüzde 10 konumundaki değere bu yamaların yüzde 90 kadarı ulaşır. Bu yüzden bu değer doğrulama kümesinde 0,90 duyarlılık veren eşiktir. Test kümesi bu eşiği yalnızca uygular. `to_string(index=False)` tabloyu satır numaraları olmadan yazdırır.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
names = np.unique(ARC["test"][2])
rows_of = {i: np.where(ARC["test"][2] == i)[0] for i in names}
diff = {"ViT": [], "Hybrid": []}
for _ in range(2000):
    idx = np.concatenate([rows_of[i] for i in rng.choice(names, len(names))])
    base = roc_auc_score(yt[idx], probs_te["CNN"][idx])
    for k in diff:
        diff[k].append(roc_auc_score(yt[idx], probs_te[k][idx]) - base)
for k, d in diff.items():
    print(f"{k} eksi evrişimli ağ: {np.mean(d):.3f} ({np.percentile(d, 2.5):.3f}-{np.percentile(d, 97.5):.3f})")

**Gözlemle ve düşün.** 5. bölümdeki 5.106 sentetik yamada en iyi sonucu evrişimli ağ vermişti; ARCADE'in 13.018 yamasında ise sıralama değişmektedir. İki küme arasındaki, büyüklükleri dışındaki hangi farklar bunu açıklayabilir?

**Alıştırma G.** Küçük Vision Transformer'ın eğitilebilir ağırlıklarından kaçı dört Transformer bloğunda, yani `vit_arcade.blocks` içinde bulunur?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in vit_arcade.blocks.parameters())
teach.check("Alıştırma G", cevap)

## 8. Dikkat haritaları

*Ders: [Attention haritası görselleştirme (klinik yorumlama)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#attention-maps-read-clinically-with-their-limits)*

Küçük Vision Transformer'ın her bloğu son çağrısındaki dikkati saklar. Dört katman boyunca dikkat aktarımıyla izlendiğinde sınıf token'ının dikkati her yama için 8 × 8 karelik bir harita verir; haritanın ortadaki dört kareye düşen payı, ağın bir darlığın bulunacağı yere bakıp bakmadığını gösterir.

**Formüllerden koda.** $\tilde{A}_l = \mathrm{rows}(\tfrac{1}{2}A_l + \tfrac{1}{2}I)$ ve $R = \tilde{A}_L \cdots \tilde{A}_1$: 3. bölümdeki `rollout(maps)`. Bir bloğun dikkati: `block.attn.weights`; başlar bu dizinin ikinci ekseni boyuncadır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
@torch.no_grad()
def class_token_maps(model, X):
    """Sınıf token'ının her yamanın 64 karesi üzerindeki dikkat aktarımı: n x 8 x 8 boyutlu bir dizi."""
    model.eval()
    out = []
    for i in range(0, len(X), 256):
        model(torch.as_tensor(X[i:i + 256], dtype=torch.float32)[:, None])
        A = np.stack([b.attn.weights.mean(dim=1).numpy() for b in model.blocks], axis=1)
        out += [rollout(a)[0, 1:].reshape(8, 8) for a in A]
    return np.array(out)


maps_te = class_token_maps(vit_arcade, ARC["test"][0])
share = maps_te[:, 3:5, 3:5].sum(axis=(1, 2)) / maps_te.sum(axis=(1, 2))
print("ortadaki dört karenin payı, darlık içeren:", round(share[yt == 1].mean(), 3),
      "| içermeyen:", round(share[yt == 0].mean(), 3), "| eşit dağılım:", 4 / 64)
raw_test = arc["test"][0]
pick = list(np.where(yt == 1)[0][:3]) + list(np.where(yt == 0)[0][:3])
fig, ax = plt.subplots(1, 6, figsize=(11, 2.2))
for a, i in zip(ax, pick):
    a.imshow(raw_test[i], cmap="gray")
    a.imshow(np.kron(maps_te[i], np.ones((8, 8))), cmap="inferno", alpha=0.45)
    a.set_title(f"{'darlık' if yt[i] else 'damar'} {probs_te['ViT'][i]:.2f}", fontsize=9)
    a.axis("off")
plt.show()

**Python notu.** `b.attn.weights`, son yama grubu için yama × baş × 65 × 65 boyutlu bir dizi tutar; `.mean(dim=1)` dört başın ortalamasını alır. `np.stack(..., axis=1)` her yamanın dört katmanını yan yana koyar; böylece `rollout(a)` bunları çarpabilir. `[0, 1:]` sınıf token'ının satırını, kendisine olan dikkati olmadan alır. `maps_te[:, 3:5, 3:5]` her haritanın ortadaki dört karesini, yani yamanın ortadaki 16 × 16 pikselini alır. Görüntüler: ARCADE, Popov ve ark. 2024, CC0.

In [ ]:
k = int(np.where((echo_patient == "patient0201") & (echo_view == "4CH"))[0][0])
fig, ax = plt.subplots(1, 2, figsize=(6.4, 3.2))
for a, (title, img_) in zip(ax, (("vaka, sol anjiyogram", small), ("CAMUS, patient0201, 4CH", echo[k]))):
    r = rollout(vit_attention(img_))[0, 1:].reshape(14, 14)
    a.imshow(np.asarray(Image.fromarray((img_ * 255).astype(np.uint8)).resize((224, 224))), cmap="gray")
    a.imshow(np.kron(r, np.ones((16, 16))), cmap="inferno", alpha=0.5)
    a.set_title(title, fontsize=9)
    a.axis("off")
plt.show()

**Gözlemle ve düşün.** ViT-B/16 yalnızca fotoğraflardan öğrenmiştir. Dikkat aktarımı anjiyogramda ve ekokardiyogramda nerede yoğunlaşır ve bir kardiyolog oraya bakar mıydı? Böyle bir haritayı bir hastaya ya da bir meslektaşınıza göstermeden önce neye ihtiyacınız olurdu?

**Alıştırma H.** Darlık içermeyen test yamalarında dikkat aktarımının ortadaki dört kareye düşen ortalama payı kaçtır? `maps_te` ve `yt` dizilerini kullanın. Üç ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("H")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
s = maps_te[:, 3:5, 3:5].sum(axis=(1, 2)) / maps_te.sum(axis=(1, 2))
cevap = round(float(s[yt == 0].mean()), 3)
teach.check("Alıştırma H", cevap)

## 9. Yamalardan bütün anjiyograma

*Ders: [Yamalardan bütün anjiyograma](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#from-patches-to-the-whole-angiogram)*

Hibrit model, bir test anjiyogramını baştan sona her 32 pikselde bir alınan 64 × 64 piksellik pencerelerle, yani anjiyogram başına 256 pencereyle tarar; aşağıdaki kod bunu ilk 20 test anjiyogramında yapar. Doğrulama yamalarının eşiğinde, merkezi bir darlığa 32 pikselden yakın olan bir pencere eşiğe ulaştığında darlık bulunmuş sayılır; eşiği aşan öteki her pencere bir alarmdır.

**Formüllerden koda.** Olasılık ızgarası: `heat_map(model, image)`. Bulunan darlıklar ve alarm pencereleri: `lesions_found(grid, centres, stenoses, threshold)`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
def heat_map(model, image, size=64, stride=32):
    """Her 32 pikselde bir alınan 64 x 64 piksellik pencereler için darlık olasılığı: 512 x 512 boyutlu bir anjiyogram için 16 x 16 boyutlu bir ızgara."""
    h = size // 2
    pad = np.pad(image, h, mode="reflect")
    centres = np.arange(stride // 2, image.shape[0], stride)
    windows = [pad[r:r + size, c:c + size] for r in centres for c in centres]
    return predict(model, standardise(windows)).reshape(len(centres), len(centres)), centres


def lesions_found(grid, centres, stenoses, threshold, near=32):
    """Merkezi bir darlığa 32 pikselden yakın olan bir pencere eşiğe ulaştığında darlık bulunmuş sayılır; her darlıktan 32 pikselden uzak olup eşiği aşan pencere yanlış alarmdır."""
    cy, cx = np.meshgrid(centres, centres, indexing="ij")
    hot = grid >= threshold
    found, alarm = 0, hot.copy()
    for s in stenoses:
        close = np.hypot(cx - s["x"], cy - s["y"]) <= near
        found += int((hot & close).any())
        alarm &= ~close
    return found, int(alarm.sum())

test_ann = arcade.annotations("test")[:20]
grids = []
found = total = alarms = 0
for a in test_ann:
    grid, centres = heat_map(hybrid_arcade, arcade.load("test", a["file"]))
    grids.append(grid)
    f_, n_alarm = lesions_found(grid, centres, a["stenoses"], thresholds["Hybrid"])
    found, total, alarms = found + f_, total + len(a["stenoses"]), alarms + n_alarm
flagged = np.mean([(g >= thresholds["Hybrid"]).mean() for g in grids])
print(f"bulunan darlık: {found}/{total} | anjiyogram başına alarm penceresi: {alarms / len(test_ann):.1f} | eşiği aşan pencerelerin oranı: {flagged:.2f}")
a, g = test_ann[0], grids[0]
plt.figure(figsize=(4.4, 4.4))
plt.imshow(arcade.load("test", a["file"]), cmap="gray")
plt.imshow(np.kron(g, np.ones((32, 32))), cmap="inferno", alpha=0.45, vmin=0, vmax=1)
for s in a["stenoses"]:
    bx, by, bw, bh = s["box"]
    plt.gca().add_patch(plt.Rectangle((bx, by), bw, bh, fill=False, color="white", lw=1.5))
plt.axis("off")
plt.show()

**Python notu.** `heat_map` görüntüyü kenarlarını yansıtarak genişletir, bir merkezler ızgarasının her noktası çevresinde bir pencere keser, pencereleri standartlaştırır ve ağdan olasılıklarını ister; bunları merkezlerle birlikte 16 × 16 boyutlu bir ızgara olarak döndürür. `lesions_found` ızgarayı darlıklarla karşılaştırır: `np.meshgrid` her merkezin koordinatlarını, `np.hypot` bir darlığa olan uzaklığı verir; her darlıktan uzak olup eşiği aşan pencereler alarmdır. Görüntü: ARCADE, Popov ve ark. 2024, CC0.

**Gözlemle ve düşün.** Neredeyse her darlık bulunur ve pencerelerin çoğu alarm verir. Ağ, bütün bir anjiyogramın hangi pencerelerini eğitimde hiç görmemiştir? Yeniden taramadan önce eğitim yamalarını nasıl değiştirirdiniz?

**Alıştırma I.** Yukarıdaki hücredeki 20 test anjiyogramında 6. bölümdeki evrişimli ağın pencerelerinin ne kadarı kendi eşiğine, yani `thresholds["CNN"]` değerine ulaşır? Anjiyogramlar üzerinden ortalama oranı iki ondalık basamakla saklayın.

In [ ]:
veri = alistirma_tablosu("I")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(np.mean([(heat_map(cnn_arcade, arcade.load("test", a["file"]))[0] >= thresholds["CNN"]).mean() for a in test_ann])), 2)
teach.check("Alıştırma I", cevap)

**3. uygulama durağının Python yolu tamamlandı.** [Ders sayfasına dönün](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#ai-pair-programming-part-7-reviewing-an-image-pipeline) ve okumaya «YZ asistanıyla çalışmak, 7. bölüm: Bir görüntü işlem hattını denetlemek» bölümünden devam edin. Bu Jupyter Notebook'un geri kalanı ek uygulamaları ve haftalık projeye yardımcı bölümleri içerir.

## 10. Bir görüntü işlem hattının denetimi: Rastgele bölme

*Ders: [YZ asistanıyla çalışmak, 7. bölüm: Bir görüntü işlem hattını denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/ders.html#ai-pair-programming-part-7-reviewing-an-image-pipeline)*

Bir modeli ayrılmış yamalarla sınaması istenen bir YZ asistanı, eğitim yamalarının beşte birini rastgele ayırmıştır. Aşağıdaki kod bu bölmeyi anjiyograma göre yapılan bir bölmeyle karşılaştırır; ikinci bölmede eğitim anjiyogramlarının beşte biri bütün yamalarıyla birlikte ayrılır. Model bir en yakın komşu modelidir: Eğitim yamalarını saklar ve yeni bir yamaya, ona en çok benzeyen beş eğitim yaması içinde darlık içerenlerin oranını darlık olasılığı olarak verir. Böyle bir model ezberler; bir bölmenin zayıflığı en açık biçimde bu modelde görülür.

**Formüllerden koda.** Rastgele bölme: `rng.permutation`, yamaların beşte biri. Anjiyograma göre bölme: `np.isin(ids, held)`; `held`, anjiyogramların beşte biridir. Ortak anjiyogramlar: İki kümenin kesişimi, `&`.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Jupyter Notebook'a buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB07_vision_transformer.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-07/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
Xtr, ytr, ids = ARC["train"]
rows32 = Xtr.reshape(len(Xtr), 32, 2, 32, 2).mean(axis=(2, 4)).reshape(len(Xtr), -1)
rng = np.random.default_rng(RANDOM_STATE)
held_random = np.zeros(len(ytr), dtype=bool)
held_random[rng.permutation(len(ytr))[:len(ytr) // 5]] = True
held = rng.choice(np.unique(ids), len(np.unique(ids)) // 5, replace=False)
held_image = np.isin(ids, held)
for name, out in (("rastgele", held_random), ("anjiyograma göre", held_image)):
    shared = set(ids[out]) & set(ids[~out])
    knn = KNeighborsClassifier(n_neighbors=5).fit(rows32[~out], ytr[~out])
    auc = roc_auc_score(ytr[out], knn.predict_proba(rows32[out])[:, 1])
    print(f"{name:16} iki tarafta anjiyogram: {len(shared):4} | ayrılan yamalarda AUC: {auc:.3f}")

**Python notu.** `reshape(len(Xtr), 32, 2, 32, 2).mean(axis=(2, 4))` her yamayı 2 × 2 piksellik karelerin ortalamasını alarak 32 × 32 piksele küçültür; son `reshape` yamayı 1.024 sayıdan oluşan bir satıra dönüştürür. `KNeighborsClassifier(n_neighbors=5)`, `fit` adımında eğitim satırlarını saklar. `predict_proba` her yeni satır için en yakın beş eğitim satırını bulur ve bunlar içindeki her sınıfın oranını döndürür. `np.zeros(..., dtype=bool)` `False` değerlerinden oluşan bir maske yapar; satırların rastgele bir permütasyonunun ilk beşte biri `True` olur. `rng.choice(..., replace=False)` anjiyogramların beşte birini hiçbirini tekrar etmeden çeker; `np.isin` bu anjiyogramların her yamasını işaretler. `set(...) & set(...)` iki tarafta da geçen anjiyogramları tutar.

**Gözlemle ve düşün.** Rastgele bölmede, ayrılan ve darlık içeren bir yamanın en yakın komşusu çoğu zaman onun ikizidir: Birkaç piksel kaydırılmış aynı darlık. Veri çoğaltmayla birkaç epoch eğitilen küçük bir evrişimli ağ bu modelden daha az, uzun süre eğitilen büyük bir ağ ise daha çok ezberler. Model ne olursa olsun anjiyograma, hasta biliniyorsa hastaya göre bölme neden gerekli kalır?

**Alıştırma J.** Rastgele bölmede (`held_random`) ARCADE'in kaç eğitim anjiyogramının iki tarafta da yaması vardır?

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = len(set(ids[held_random]) & set(ids[~held_random]))
teach.check("Alıştırma J", cevap)

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Girişimsel kardiyoloji | LAD'de yüzde 50-69 darlığı olan test hastalarının sentetik anjiyogramlarını çizin. ARCADE ile eğitilen hibrit model bu darlıklara anlamlı darlıkların yamalarından daha yüksek olasılıklar verir mi? Bu durum laboratuvardaki bir araç için ne anlama gelirdi? |
| Ekokardiyografi | 4. bölümdeki kalite kapısını iki boşluk ve dört boşluk görüntüleri için ayrı ayrı kurun. Kötü bir dört boşluk karesini işaretlemek, kötü bir iki boşluk karesini işaretlemekten daha mı zordur ve test kümesinde her görüntü türünden kaç kötü kare vardır? |
| Kalp yetersizliği | Kalite kapısını 4. haftanın otomatik ejeksiyon fraksiyonuyla birleştirin: Kapının işaretlediği karelerde ağ ile uzman arasındaki fark daha mı büyüktür? CAMUS'un test hastalarını kullanın. |
| Kardiyak görüntüleme | Küçük Vision Transformer'ı 8 × 8 yerine 16 × 16 piksellik karelerle (64 yerine 16 token) eğitin. Ağırlık sayısı, eğitim süresi ve test AUC değeri nasıl değişir? |
| Klinik araştırma | Hibrit modelin eğitimini beş farklı seed değeriyle tekrarlayın. Test AUC değerinin yayılımı ne kadardır ve Vision Transformer ile arasındaki fark bu yayılımdan büyük müdür? |
| Kalite ve güvenlik | Hibrit modelin 0,9'un üstünde ya da 0,1'in altında bir olasılıkla yanlış değerlendirdiği on ARCADE test yaması seçin ve dikkat haritalarıyla birlikte inceleyin. Bu hatalardan hangilerini bir kardiyolog görüntüden açıklayabilirdi? |
| Tıp bilişimi | Bir firmanın darlık aracının bir hastanenin anjiyogramlarında çalışmasından önce hastanenin uygulaması gereken kontrol listesini yazın: Veri, referans, kümeler, kararın verildiği düzeydeki ölçütler, mahremiyet ve alarmların okunması. |

## Jupyter Notebook'tan sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-07/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma yazınızı yazın ve öğrenme günlüğünü dışa aktarın. Bu haftanın haftalık projesi 7. ve 9. bölümleri genişletir: ARCADE'de üç ağ karşılaştırılır, eşikleri iki prevalansta değerlendirilir ve bütün anjiyogramlar için kurulan bir tespit modelinin hataları bir kardiyolog gözüyle okunur.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>